# Notebook 04b - Archive Robustness and Model Closure

## Purpose

This notebook performs targeted robustness experiments that close the
first-week ALMA Science Archive exploration phase. It tests whether the
preliminary relationship-aware Archive model remains useful across a
broader and deliberately adversarial sample.

The notebook focuses on:

- the live `ivoa.obscore` schema and schema drift;
- complete Member OUS retrieval and query completeness;
- cross-cycle, cross-band, mosaic, and array-related structural cases;
- `obs_id` grammar, candidate identifiers, and unexpected cardinalities;
- field ownership at Member, Source, Source-Execution, SPW, and row level;
- spatial-footprint and ASDM-association scope;
- context-scoped frequency-support-to-SPW mapping;
- defensive reconstruction behaviour for incomplete or conflicting data.

## Research questions

1. Which live TAP columns are modeled, cross-check-only, deferred, or still unexplored?
2. Do the source-by-SPW row structure and tested candidate keys survive a broader sample?
3. Which fields are stable at Member, Source, Source-Execution, Logical-SPW, or row level?
4. Can the same parsed source occur across multiple ASDM associations, and what changes across them?
5. Should spatial footprints be attached to Source Context or Source-Execution Context?
6. Does SPW-support mapping require explicit Source-Execution or raw-row scope?
7. What array-related evidence is available in `antenna_arrays`, and how stable is it?
8. Which standard ObsCore fields provide useful cross-checks without replacing ALMA-specific values?
9. How should the Archive client represent truncation, overflow, empty results, schema drift, and reconstruction conflicts?

Every relationship is reported as documentation-supported,
sample-supported, application-derived, contradicted, or unresolved.

## 1. Environment and reproducibility

In [1]:
from __future__ import annotations

from dataclasses import dataclass
from datetime import datetime, timezone
import hashlib
import platform
import re
import sys
import warnings

import astropy
from astropy import units as u
from astropy.constants import c
import numpy as np
import pandas as pd
import pyvo

from alma_duplicate.parsers.frequency_support import parse_frequency_support

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

RUN_STARTED_AT = datetime.now(timezone.utc)

print("Python:", sys.version)
print("Executable:", sys.executable)
print("Platform:", platform.platform())
print("Astropy:", astropy.__version__)
print("Pandas:", pd.__version__)
print("PyVO:", pyvo.__version__)
print("Run time UTC:", RUN_STARTED_AT.isoformat())

Python: 3.12.11 | packaged by Anaconda, Inc. | (main, Jun  5 2025, 08:06:15) [Clang 14.0.6 ]
Executable: /Users/nana/opt/anaconda3/envs/alma-duplication/bin/python
Platform: macOS-26.4.1-x86_64-i386-64bit
Astropy: 8.0.1
Pandas: 3.0.5
PyVO: 1.9.1
Run time UTC: 2026-08-25T12:59:36.981698+00:00


In [2]:
TAP_URL = "https://almascience.eso.org/tap"
EXPLORATORY_SAFETY_LIMIT = 25_000
MEMBERS_PER_STRATUM = 4

tap_service = pyvo.dal.TAPService(TAP_URL)

print("TAP endpoint:", TAP_URL)
print("Exploratory safety limit:", EXPLORATORY_SAFETY_LIMIT)
print("Members selected per stratum:", MEMBERS_PER_STRATUM)

TAP endpoint: https://almascience.eso.org/tap
Exploratory safety limit: 25000
Members selected per stratum: 4


In [3]:
@dataclass(frozen=True)
class TapQueryRun:
    label: str
    endpoint: str
    adql: str
    maxrec: int
    started_at_utc: str
    finished_at_utc: str
    retrieved_rows: int
    query_status: tuple[str, ...]
    warning_messages: tuple[str, ...]
    result: object
    table: object


def extract_query_status(result: object) -> tuple[str, ...]:
    """Best-effort extraction of VOTable INFO status values."""
    statuses = []
    votable = getattr(result, "votable", None)
    resources = getattr(votable, "resources", []) if votable is not None else []

    for resource in resources:
        for info in getattr(resource, "infos", []):
            if str(getattr(info, "name", "")).upper() == "QUERY_STATUS":
                value = getattr(info, "value", None)
                content = getattr(info, "content", None)
                statuses.append(f"{value}: {content}" if content else str(value))

    return tuple(statuses)


def run_tap_query(adql: str, *, maxrec: int, label: str) -> TapQueryRun:
    """Run one TAP query and preserve execution evidence."""
    started = datetime.now(timezone.utc)

    print(f"\n--- {label} ---")
    print("Endpoint:", TAP_URL)
    print("MAXREC:", maxrec)
    print("ADQL:")
    print(adql)

    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        result = tap_service.search(adql, maxrec=maxrec)
        table = result.to_table()

    finished = datetime.now(timezone.utc)
    warning_messages = tuple(str(item.message) for item in caught)
    query_status = extract_query_status(result)

    print("Retrieved rows:", len(table))
    print("QUERY_STATUS:", query_status or "<not exposed by client>")
    print("Warnings:", warning_messages or "<none>")

    return TapQueryRun(
        label=label,
        endpoint=TAP_URL,
        adql=adql,
        maxrec=maxrec,
        started_at_utc=started.isoformat(),
        finished_at_utc=finished.isoformat(),
        retrieved_rows=len(table),
        query_status=query_status,
        warning_messages=warning_messages,
        result=result,
        table=table,
    )


def quote_adql_string(value: str) -> str:
    return "'" + value.replace("'", "''") + "'"


query_runs: list[TapQueryRun] = []

## 2. Live schema snapshot and field-scope inventory

The Archive data dictionary currently documents the fields most relevant
to reconstruction. This experiment snapshots the complete live schema and
explicitly separates:

- fields already modeled;
- fields retained as cross-check evidence;
- fields deferred because they are not required by current reconstruction;
- newly appearing or unclassified fields.

A schema fingerprint is recorded so that later integration tests can detect
service changes without assuming that the schema is immutable.

In [4]:
schema_query = """
SELECT
    column_name,
    datatype,
    unit,
    ucd,
    description
FROM TAP_SCHEMA.columns
WHERE table_name = 'ivoa.obscore'
ORDER BY column_name
"""

schema_run = run_tap_query(
    schema_query,
    maxrec=5_000,
    label="Retrieve live ivoa.obscore schema",
)
query_runs.append(schema_run)
schema_df = schema_run.table.to_pandas()

schema_serialization = schema_df.fillna("<NULL>").astype(str).to_csv(index=False)
schema_fingerprint = hashlib.sha256(schema_serialization.encode("utf-8")).hexdigest()

print("Archive column count:", len(schema_df))
print("Schema SHA-256:", schema_fingerprint)
display(schema_df)


--- Retrieve live ivoa.obscore schema ---
Endpoint: https://almascience.eso.org/tap
MAXREC: 5000
ADQL:

SELECT
    column_name,
    datatype,
    unit,
    ucd,
    description
FROM TAP_SCHEMA.columns
WHERE table_name = 'ivoa.obscore'
ORDER BY column_name

Retrieved rows: 73
QUERY_STATUS: ('OK',)
Warnings: <none>
Archive column count: 73
Schema SHA-256: 2cb2009067ab50f1727454ccb57cb1280c81ad4bfa3a10a9c2df2f0de7044c15


,column_name,datatype,unit,ucd,description
0,access_estsize,int,kbyte,phys.size;meta.file,Estimated size of datasets in kilobytes
1,access_format,char,,meta.code.mime,Content format of the data
2,access_url,char,,meta.ref.url,URL to download the data
3,antenna_arrays,char,,meta.code.member;instr.setup,"Blank-separated list of Pad:Antenna pairs, i.e., A109:DV09 J504:DV02 J505:DV05 for antennas DV09, DV02 and DV05 sitt..."
4,asdm_uid,char,,meta.id,UID of the ASDM containing this Field.
...,...,...,...,...,...
68,t_resolution,double,s,time.resolution,typical temporal resolution
69,t_xel,int,,meta.number,Number of elements along the time axis
70,target_name,char,,meta.id;src,name of intended target
71,type,char,,,Type flags.


In [5]:
modeled_fields = {
    "proposal_id", "obs_publisher_did", "group_ous_uid", "member_ous_uid",
    "asdm_uid", "obs_id", "target_name", "s_ra", "s_dec", "s_region",
    "frequency", "bandwidth", "frequency_support", "spectral_resolution",
    "velocity_resolution", "em_resolution", "s_resolution",
    "spatial_resolution", "sensitivity_10kms",
    "cont_sensitivity_bandwidth", "antenna_arrays", "is_mosaic",
    "band_list", "pol_states", "t_min", "t_max", "science_observation",
    "scan_intent", "data_rights", "qa2_passed", "obs_release_date",
}

cross_check_fields = {
    "s_fov", "em_min", "em_max", "em_res_power", "spatial_scale_max",
    "t_exptime", "t_resolution", "dataproduct_type", "calib_level",
    "type", "pwv", "lastModified",
}

deferred_fields = {
    "access_url", "access_format", "authors", "proposal_authors",
    "proposal_abstract", "pub_abstract", "pub_title", "first_author",
    "bib_reference", "publication_year", "science_keyword",
    "scientific_category", "obs_title", "schedblock_name",
}

def classify_schema_field(name: str) -> str:
    if name in modeled_fields:
        return "MODELED"
    if name in cross_check_fields:
        return "CROSS_CHECK"
    if name in deferred_fields:
        return "DEFERRED"
    return "UNCLASSIFIED"


schema_scope_df = schema_df.copy()
schema_scope_df["current_scope"] = schema_scope_df["column_name"].map(classify_schema_field)

display(
    schema_scope_df.groupby("current_scope", dropna=False)
    .size()
    .rename("column_count")
    .reset_index()
)

display(schema_scope_df.loc[schema_scope_df["current_scope"] == "UNCLASSIFIED"])

,current_scope,column_count
0,CROSS_CHECK,12
1,DEFERRED,14
2,MODELED,31
3,UNCLASSIFIED,16


,column_name,datatype,unit,ucd,description,current_scope
0,access_estsize,int,kbyte,phys.size;meta.file,Estimated size of datasets in kilobytes,UNCLASSIFIED
10,collections,char,,,Indicates that there are external products,UNCLASSIFIED
18,em_xel,int,,meta.number,Number of elements along the spectral axis,UNCLASSIFIED
19,facility_name,char,,meta.id;instr.tel,telescope name,UNCLASSIFIED
23,gal_latitude,double,deg,pos.galactic.lat,Galactic latitude of the observation for RA/Dec. Estimated using PyEphem and RA/Dec.,UNCLASSIFIED
24,gal_longitude,double,deg,pos.galactic.lon,Galactic longitude of the observation for RA/Dec. Estimated using PyEphem and RA/Dec.,UNCLASSIFIED
26,instrument_name,char,,meta.id;instr,instrument name,UNCLASSIFIED
30,o_ucd,char,,meta.ucd,UCD describing the observable axis (pixel values),UNCLASSIFIED
31,obs_collection,char,,meta.id,short name for the data collection,UNCLASSIFIED
32,obs_creator_name,char,,meta.id,case-insensitive partial match over the full PI name. Wildcards can be used,UNCLASSIFIED


## 3. Robustness sample manifest

Discovery queries select Member OUS identifiers only. All science-target rows
belonging to the selected Members are counted and retrieved later.

The sample is purposive, deterministic where the service ordering permits,
and designed to expose structural counterexamples. It is not used to estimate
Archive-wide prevalence.

Strata cover:

- early, middle, and recent proposal identifiers;
- low- and high-frequency ALMA bands;
- mosaic and non-mosaic records;
- 7-m-, 12-m-, and total-power-like antenna-name evidence.

Antenna-prefix strata are discovery heuristics only. Their scientific meaning
must not be promoted to an authoritative array classification without ALMA
documentation or supervisor confirmation.

In [6]:
discovery_queries = {
    "early_cycles": """
        SELECT TOP 80 proposal_id, member_ous_uid, frequency,
            is_mosaic, antenna_arrays, band_list
        FROM ivoa.obscore
        WHERE science_observation = 'T'
          AND member_ous_uid IS NOT NULL
          AND (
              proposal_id LIKE '2011.%'
              OR proposal_id LIKE '2012.%'
              OR proposal_id LIKE '2013.%'
              OR proposal_id LIKE '2014.%'
          )
        ORDER BY proposal_id, member_ous_uid
    """,
    "middle_cycles": """
        SELECT TOP 80 proposal_id, member_ous_uid, frequency,
            is_mosaic, antenna_arrays, band_list
        FROM ivoa.obscore
        WHERE science_observation = 'T'
          AND member_ous_uid IS NOT NULL
          AND (
              proposal_id LIKE '2017.%'
              OR proposal_id LIKE '2018.%'
              OR proposal_id LIKE '2019.%'
          )
        ORDER BY proposal_id, member_ous_uid
    """,
    "recent_cycles": """
        SELECT TOP 80 proposal_id, member_ous_uid, frequency,
            is_mosaic, antenna_arrays, band_list
        FROM ivoa.obscore
        WHERE science_observation = 'T'
          AND member_ous_uid IS NOT NULL
          AND (
              proposal_id LIKE '2023.%'
              OR proposal_id LIKE '2024.%'
              OR proposal_id LIKE '2025.%'
          )
        ORDER BY proposal_id, member_ous_uid
    """,
    "band3": """
        SELECT TOP 80 proposal_id, member_ous_uid, frequency,
            is_mosaic, antenna_arrays, band_list
        FROM ivoa.obscore
        WHERE science_observation = 'T'
          AND member_ous_uid IS NOT NULL
          AND frequency BETWEEN 84 AND 116
        ORDER BY proposal_id DESC, member_ous_uid
    """,
    "band10": """
        SELECT TOP 80 proposal_id, member_ous_uid, frequency,
            is_mosaic, antenna_arrays, band_list
        FROM ivoa.obscore
        WHERE science_observation = 'T'
          AND member_ous_uid IS NOT NULL
          AND frequency BETWEEN 787 AND 950
        ORDER BY proposal_id DESC, member_ous_uid
    """,
    "mosaic": """
        SELECT TOP 80 proposal_id, member_ous_uid, frequency,
            is_mosaic, antenna_arrays, band_list
        FROM ivoa.obscore
        WHERE science_observation = 'T'
          AND member_ous_uid IS NOT NULL
          AND is_mosaic = 'T'
        ORDER BY proposal_id DESC, member_ous_uid
    """,
    "non_mosaic": """
        SELECT TOP 80 proposal_id, member_ous_uid, frequency,
            is_mosaic, antenna_arrays, band_list
        FROM ivoa.obscore
        WHERE science_observation = 'T'
          AND member_ous_uid IS NOT NULL
          AND is_mosaic = 'F'
        ORDER BY proposal_id DESC, member_ous_uid
    """,
    "cm_antenna_evidence": """
        SELECT TOP 80 proposal_id, member_ous_uid, frequency,
            is_mosaic, antenna_arrays, band_list
        FROM ivoa.obscore
        WHERE science_observation = 'T'
          AND member_ous_uid IS NOT NULL
          AND antenna_arrays LIKE '%:CM%'
        ORDER BY proposal_id DESC, member_ous_uid
    """,
    "da_dv_antenna_evidence": """
        SELECT TOP 80 proposal_id, member_ous_uid, frequency,
            is_mosaic, antenna_arrays, band_list
        FROM ivoa.obscore
        WHERE science_observation = 'T'
          AND member_ous_uid IS NOT NULL
          AND (antenna_arrays LIKE '%:DA%' OR antenna_arrays LIKE '%:DV%')
        ORDER BY proposal_id DESC, member_ous_uid
    """,
    "pm_antenna_evidence": """
        SELECT TOP 80 proposal_id, member_ous_uid, frequency,
            is_mosaic, antenna_arrays, band_list
        FROM ivoa.obscore
        WHERE science_observation = 'T'
          AND member_ous_uid IS NOT NULL
          AND antenna_arrays LIKE '%:PM%'
        ORDER BY proposal_id DESC, member_ous_uid
    """,
}

In [7]:
discovery_frames = []

for stratum, adql in discovery_queries.items():
    run = run_tap_query(adql, maxrec=80, label=f"Discover {stratum}")
    query_runs.append(run)
    frame = run.table.to_pandas()
    frame["sample_stratum"] = stratum
    discovery_frames.append(frame)

discovery_df = pd.concat(discovery_frames, ignore_index=True)
discovery_df["member_ous_uid"] = discovery_df["member_ous_uid"].astype("string")

manifest_parts = []
for stratum, group in discovery_df.groupby("sample_stratum", sort=False):
    selected = (
        group.dropna(subset=["member_ous_uid"])
        .drop_duplicates("member_ous_uid")
        .head(MEMBERS_PER_STRATUM)
        .copy()
    )
    manifest_parts.append(selected)

sample_manifest_long_df = pd.concat(manifest_parts, ignore_index=True)

# Preserve the known multi-ASDM structural counterexample from Notebook 2/4.
known_multi_asdm_member = "uid://A001/X3833/X1022"
sample_manifest_long_df = pd.concat(
    [
        sample_manifest_long_df,
        pd.DataFrame(
            [{
                "proposal_id": None,
                "member_ous_uid": known_multi_asdm_member,
                "frequency": None,
                "is_mosaic": None,
                "antenna_arrays": None,
                "band_list": None,
                "sample_stratum": "known_multi_asdm_counterexample",
            }]
        ),
    ],
    ignore_index=True,
)

sample_manifest_df = (
    sample_manifest_long_df.groupby("member_ous_uid", dropna=False)
    .agg(
        discovery_strata=("sample_stratum", lambda s: tuple(sorted(set(s)))),
        discovery_proposals=("proposal_id", lambda s: tuple(sorted(set(s.dropna().astype(str))))),
    )
    .reset_index()
)

print("Selected unique Member OUS datasets:", len(sample_manifest_df))
display(sample_manifest_df)


--- Discover early_cycles ---
Endpoint: https://almascience.eso.org/tap
MAXREC: 80
ADQL:

        SELECT TOP 80 proposal_id, member_ous_uid, frequency,
            is_mosaic, antenna_arrays, band_list
        FROM ivoa.obscore
        WHERE science_observation = 'T'
          AND member_ous_uid IS NOT NULL
          AND (
              proposal_id LIKE '2011.%'
              OR proposal_id LIKE '2012.%'
              OR proposal_id LIKE '2013.%'
              OR proposal_id LIKE '2014.%'
          )
        ORDER BY proposal_id, member_ous_uid
    
Retrieved rows: 80
QUERY_STATUS: ('OK',)
Warnings: <none>

--- Discover middle_cycles ---
Endpoint: https://almascience.eso.org/tap
MAXREC: 80
ADQL:

        SELECT TOP 80 proposal_id, member_ous_uid, frequency,
            is_mosaic, antenna_arrays, band_list
        FROM ivoa.obscore
        WHERE science_observation = 'T'
          AND member_ous_uid IS NOT NULL
          AND (
              proposal_id LIKE '2017.%'
              OR pro

,member_ous_uid,discovery_strata,discovery_proposals
0,uid://A001/X1288/X4c3,"(middle_cycles,)","(2017.1.00001.S,)"
1,uid://A001/X1288/X4c7,"(middle_cycles,)","(2017.1.00001.S,)"
2,uid://A001/X1288/X4d7,"(middle_cycles,)","(2017.1.00002.S,)"
3,uid://A001/X1288/X4dd,"(middle_cycles,)","(2017.1.00002.S,)"
4,uid://A001/X3621/X3ca0,"(recent_cycles,)","(2023.1.00026.S,)"
5,uid://A001/X3621/X3ca4,"(recent_cycles,)","(2023.1.00026.S,)"
6,uid://A001/X362b/Xbe5,"(recent_cycles,)","(2023.1.00014.S,)"
7,uid://A001/X3645/X15f,"(recent_cycles,)","(2023.1.00022.S,)"
8,uid://A001/X3833/X1022,"(known_multi_asdm_counterexample,)",()
9,uid://A001/X3833/X14b4,"(band10,)","(2025.1.01151.S,)"


## 4. Count and retrieve every selected Member OUS row

In [8]:
available_columns = set(schema_df["column_name"].dropna().astype(str))

desired_archive_columns = [
    "proposal_id", "obs_publisher_did", "group_ous_uid", "member_ous_uid",
    "obs_id", "asdm_uid", "target_name", "s_ra", "s_dec", "s_fov",
    "s_region", "s_resolution", "t_min", "t_max", "t_exptime",
    "t_resolution", "em_min", "em_max", "em_res_power", "frequency",
    "bandwidth", "frequency_support", "spectral_resolution",
    "velocity_resolution", "em_resolution", "spatial_resolution",
    "spatial_scale_max", "sensitivity_10kms",
    "cont_sensitivity_bandwidth", "antenna_arrays", "is_mosaic",
    "pol_states", "band_list", "scan_intent", "science_observation",
    "data_rights", "qa2_passed", "obs_release_date", "dataproduct_type",
    "calib_level", "type", "pwv", "lastModified",
]

required_columns = {
    "proposal_id", "member_ous_uid", "obs_id", "asdm_uid", "target_name",
    "s_ra", "s_dec", "s_region", "frequency", "bandwidth",
    "frequency_support", "antenna_arrays", "is_mosaic",
    "science_observation",
}

missing_required = required_columns - available_columns
if missing_required:
    raise RuntimeError(f"Required live Archive columns are missing: {sorted(missing_required)}")

selected_archive_columns = [
    name for name in desired_archive_columns if name in available_columns
]

print("Selected columns:", len(selected_archive_columns))
print("Unavailable optional columns:", sorted(set(desired_archive_columns) - available_columns))

Selected columns: 43
Unavailable optional columns: []


In [9]:
selected_member_uids = sample_manifest_df["member_ous_uid"].dropna().astype(str).tolist()
member_uid_sql = ",\n    ".join(quote_adql_string(uid) for uid in selected_member_uids)

count_query = f"""
SELECT COUNT(*) AS total_rows
FROM ivoa.obscore
WHERE science_observation = 'T'
  AND member_ous_uid IN (
    {member_uid_sql}
  )
"""

count_run = run_tap_query(count_query, maxrec=10, label="Count complete robustness sample")
query_runs.append(count_run)
expected_row_count = int(count_run.table[0]["total_rows"])

print("Expected complete rows:", expected_row_count)

if expected_row_count > EXPLORATORY_SAFETY_LIMIT:
    raise RuntimeError(
        f"Expected {expected_row_count} rows, above safety limit "
        f"{EXPLORATORY_SAFETY_LIMIT}. Reduce MEMBERS_PER_STRATUM."
    )


--- Count complete robustness sample ---
Endpoint: https://almascience.eso.org/tap
MAXREC: 10
ADQL:

SELECT COUNT(*) AS total_rows
FROM ivoa.obscore
WHERE science_observation = 'T'
  AND member_ous_uid IN (
    'uid://A001/X1288/X4c3',
    'uid://A001/X1288/X4c7',
    'uid://A001/X1288/X4d7',
    'uid://A001/X1288/X4dd',
    'uid://A001/X3621/X3ca0',
    'uid://A001/X3621/X3ca4',
    'uid://A001/X362b/Xbe5',
    'uid://A001/X3645/X15f',
    'uid://A001/X3833/X1022',
    'uid://A001/X3833/X14b4',
    'uid://A001/X3833/X1982',
    'uid://A001/X3833/X198e',
    'uid://A001/X3833/X1998',
    'uid://A001/X3845/X5cf',
    'uid://A001/X3845/X776',
    'uid://A001/X3845/X77a',
    'uid://A001/X3873/X44c',
    'uid://A001/X38cd/X1d',
    'uid://A001/X38cd/X1ea',
    'uid://A001/X38cd/X2d',
    'uid://A001/X38cd/X3c3',
    'uid://A001/X38cd/X3c7',
    'uid://A001/X38cd/Xd4',
    'uid://A001/X38cd/Xd7',
    'uid://A001/X3922/X599',
    'uid://A001/X3922/X59c',
    'uid://A001/X3922/X59f',
    'u

Retrieved rows: 1
QUERY_STATUS: ('OK',)
Warnings: <none>
Expected complete rows: 187


In [10]:
column_sql = ",\n    ".join(selected_archive_columns)

complete_query = f"""
SELECT
    {column_sql}
FROM ivoa.obscore
WHERE science_observation = 'T'
  AND member_ous_uid IN (
    {member_uid_sql}
  )
"""

complete_run = run_tap_query(
    complete_query,
    maxrec=max(expected_row_count, 1),
    label="Retrieve complete robustness sample",
)
query_runs.append(complete_run)

archive_table = complete_run.table
archive_df = archive_table.to_pandas()
archive_df.insert(0, "archive_row_index", np.arange(len(archive_df), dtype=int))

for identifier_column in [
    "proposal_id", "group_ous_uid", "member_ous_uid", "obs_id", "asdm_uid"
]:
    if identifier_column in archive_df.columns:
        archive_df[identifier_column] = archive_df[identifier_column].astype("string")

complete_retrieval = len(archive_df) == expected_row_count
print("Expected rows:", expected_row_count)
print("Retrieved rows:", len(archive_df))
print("Complete retrieval:", complete_retrieval)

if not complete_retrieval:
    raise RuntimeError("The complete robustness sample was not retrieved.")

raw_unit_inventory_df = pd.DataFrame(
    {
        "column_name": archive_table.colnames,
        "unit": [str(archive_table[name].unit or "") for name in archive_table.colnames],
        "dtype": [str(archive_table[name].dtype) for name in archive_table.colnames],
    }
)
display(raw_unit_inventory_df)


--- Retrieve complete robustness sample ---
Endpoint: https://almascience.eso.org/tap
MAXREC: 187
ADQL:

SELECT
    proposal_id,
    obs_publisher_did,
    group_ous_uid,
    member_ous_uid,
    obs_id,
    asdm_uid,
    target_name,
    s_ra,
    s_dec,
    s_fov,
    s_region,
    s_resolution,
    t_min,
    t_max,
    t_exptime,
    t_resolution,
    em_min,
    em_max,
    em_res_power,
    frequency,
    bandwidth,
    frequency_support,
    spectral_resolution,
    velocity_resolution,
    em_resolution,
    spatial_resolution,
    spatial_scale_max,
    sensitivity_10kms,
    cont_sensitivity_bandwidth,
    antenna_arrays,
    is_mosaic,
    pol_states,
    band_list,
    scan_intent,
    science_observation,
    data_rights,
    qa2_passed,
    obs_release_date,
    dataproduct_type,
    calib_level,
    type,
    pwv,
    lastModified
FROM ivoa.obscore
WHERE science_observation = 'T'
  AND member_ous_uid IN (
    'uid://A001/X1288/X4c3',
    'uid://A001/X1288/X4c7',
    'uid

,column_name,unit,dtype
0,proposal_id,,<U64
1,obs_publisher_did,,<U33
2,group_ous_uid,,<U64
3,member_ous_uid,,<U64
4,obs_id,,<U64
5,asdm_uid,,<U32
6,target_name,,<U256
7,s_ra,deg,float64
8,s_dec,deg,float64
9,s_fov,deg,float64


## 5. Missingness, sentinels, and cross-stratum coverage

In [11]:
sample_df = archive_df.merge(
    sample_manifest_df[["member_ous_uid", "discovery_strata"]],
    on="member_ous_uid",
    how="left",
    validate="many_to_one",
)

missingness_records = []
for column in selected_archive_columns:
    series = sample_df[column]
    blank_count = 0
    if pd.api.types.is_object_dtype(series) or pd.api.types.is_string_dtype(series):
        blank_count = series.fillna("").astype(str).str.strip().eq("").sum()

    missingness_records.append(
        {
            "column_name": column,
            "row_count": len(series),
            "null_count": int(series.isna().sum()),
            "blank_string_count": int(blank_count),
            "non_missing_count": int(series.notna().sum()),
        }
    )

missingness_df = pd.DataFrame(missingness_records)
display(missingness_df.sort_values(["null_count", "blank_string_count"], ascending=False))

if "obs_release_date" in sample_df:
    release_text = sample_df["obs_release_date"].astype("string")
    print("3000-01-01-like release dates:", int(release_text.str.startswith("3000-01-01", na=False).sum()))

display(
    sample_df.groupby("discovery_strata", dropna=False)
    .agg(
        rows=("archive_row_index", "size"),
        members=("member_ous_uid", "nunique"),
        proposals=("proposal_id", "nunique"),
        asdms=("asdm_uid", "nunique"),
    )
    .reset_index()
)

,column_name,row_count,null_count,blank_string_count,non_missing_count
2,group_ous_uid,187,0,8,187
0,proposal_id,187,0,0,187
1,obs_publisher_did,187,0,0,187
3,member_ous_uid,187,0,0,187
4,obs_id,187,0,0,187
5,asdm_uid,187,0,0,187
6,target_name,187,0,0,187
7,s_ra,187,0,0,187
8,s_dec,187,0,0,187
9,s_fov,187,0,0,187


3000-01-01-like release dates: 35


,discovery_strata,rows,members,proposals,asdms
0,"(band10,)",28,4,2,4
1,"(band3,)",12,3,2,3
2,"(band3, pm_antenna_evidence)",4,1,1,1
3,"(cm_antenna_evidence,)",11,2,2,2
4,"(cm_antenna_evidence, da_dv_antenna_evidence, mosaic)",4,1,1,1
5,"(cm_antenna_evidence, mosaic)",6,1,1,1
6,"(da_dv_antenna_evidence, non_mosaic)",12,3,1,3
7,"(early_cycles,)",32,4,3,4
8,"(known_multi_asdm_counterexample,)",8,1,1,2
9,"(middle_cycles,)",17,4,2,4


## 6. `obs_id` grammar, identifiers, and structural invariants

In [12]:
OBS_ID_PATTERN = re.compile(
    r"^(?P<obs_member_ous_uid>uid://.+?)"
    r"\.source\."
    r"(?P<source_name>.+)"
    r"\.spw\."
    r"(?P<spw_id>[^.]+)$"
)


def parse_obs_id_structure(raw_obs_id: object) -> dict[str, object]:
    if pd.isna(raw_obs_id):
        return {
            "obs_member_ous_uid": None,
            "source_name": None,
            "spw_id": None,
            "obs_id_parse_status": "FAILED",
            "obs_id_parse_issue": "missing_obs_id",
        }

    text = str(raw_obs_id).strip()
    match = OBS_ID_PATTERN.fullmatch(text)
    if match is None:
        return {
            "obs_member_ous_uid": None,
            "source_name": None,
            "spw_id": None,
            "obs_id_parse_status": "FAILED",
            "obs_id_parse_issue": "unexpected_obs_id_format",
        }

    return {
        **match.groupdict(),
        "obs_id_parse_status": "PARSED",
        "obs_id_parse_issue": None,
    }


parsed_obs_id_df = pd.DataFrame(
    sample_df["obs_id"].map(parse_obs_id_structure).tolist()
)
analysis_df = pd.concat(
    [sample_df.reset_index(drop=True), parsed_obs_id_df], axis=1
)
analysis_df["obs_member_matches_column"] = (
    analysis_df["obs_member_ous_uid"] == analysis_df["member_ous_uid"].astype(str)
)

display(
    analysis_df["obs_id_parse_status"]
    .value_counts(dropna=False)
    .rename_axis("parse_status")
    .reset_index(name="row_count")
)
display(analysis_df.loc[~analysis_df["obs_member_matches_column"].fillna(False)].head(50))

,parse_status,row_count
0,PARSED,187


,archive_row_index,proposal_id,obs_publisher_did,group_ous_uid,member_ous_uid,obs_id,asdm_uid,target_name,s_ra,s_dec,s_fov,s_region,s_resolution,t_min,t_max,t_exptime,t_resolution,em_min,em_max,em_res_power,frequency,bandwidth,frequency_support,spectral_resolution,velocity_resolution,em_resolution,spatial_resolution,spatial_scale_max,sensitivity_10kms,cont_sensitivity_bandwidth,antenna_arrays,is_mosaic,pol_states,band_list,scan_intent,science_observation,data_rights,qa2_passed,obs_release_date,dataproduct_type,calib_level,type,pwv,lastModified,discovery_strata,obs_member_ous_uid,source_name,spw_id,obs_id_parse_status,obs_id_parse_issue,obs_member_matches_column


In [13]:
candidate_keys = {
    "obs_id": ["obs_id"],
    "member_source_spw": ["member_ous_uid", "source_name", "spw_id"],
    "member_asdm_source_spw": ["member_ous_uid", "asdm_uid", "source_name", "spw_id"],
}

key_records = []
for key_name, columns in candidate_keys.items():
    valid = analysis_df.dropna(subset=columns)
    duplicated = valid.duplicated(subset=columns, keep=False)
    key_records.append(
        {
            "candidate_key": key_name,
            "columns": tuple(columns),
            "eligible_rows": len(valid),
            "duplicate_rows": int(duplicated.sum()),
            "unique_in_sample": not duplicated.any(),
        }
    )

candidate_key_df = pd.DataFrame(key_records)
display(candidate_key_df)

analysis_df["source_execution_context_key"] = list(
    zip(
        analysis_df["member_ous_uid"],
        analysis_df["asdm_uid"],
        analysis_df["source_name"],
    )
)

execution_structure_df = (
    analysis_df
    .groupby(["member_ous_uid", "asdm_uid"], dropna=False)
    .agg(
        archive_rows=("archive_row_index", "size"),
        source_contexts=("source_name", "nunique"),
        logical_spws=("spw_id", "nunique"),
        group_ous_values=("group_ous_uid", "nunique"),
        proposal_values=("proposal_id", "nunique"),
    )
    .reset_index()
)
execution_structure_df["expected_source_spw_rows"] = (
    execution_structure_df["source_contexts"]
    * execution_structure_df["logical_spws"]
)
execution_structure_df["complete_source_spw_grid"] = (
    execution_structure_df["archive_rows"]
    == execution_structure_df["expected_source_spw_rows"]
)

member_structure_df = (
    analysis_df
    .groupby("member_ous_uid", dropna=False)
    .agg(
        archive_rows=("archive_row_index", "size"),
        asdm_associations=("asdm_uid", "nunique"),
        source_execution_contexts=(
            "source_execution_context_key",
            "nunique",
        ),
        group_ous_values=("group_ous_uid", "nunique"),
        proposal_values=("proposal_id", "nunique"),
    )
    .reset_index()
)

display(execution_structure_df.sort_values("archive_rows", ascending=False))
display(
    execution_structure_df.loc[
        ~execution_structure_df["complete_source_spw_grid"]
    ]
)


,candidate_key,columns,eligible_rows,duplicate_rows,unique_in_sample
0,obs_id,"(obs_id,)",187,0,True
1,member_source_spw,"(member_ous_uid, source_name, spw_id)",187,0,True
2,member_asdm_source_spw,"(member_ous_uid, asdm_uid, source_name, spw_id)",187,0,True


,member_ous_uid,asdm_uid,archive_rows,source_contexts,logical_spws,group_ous_values,proposal_values,expected_source_spw_rows,complete_source_spw_grid
33,uid://A002/X845868/X3a,uid://A002/X867766/X339,12,3,4,1,1,12,True
32,uid://A002/X7028c3/X74,uid://A002/X740d04/X50,12,3,4,1,1,12,True
29,uid://A001/X3955/X44,uid://A002/X13e7f29/Xc580,11,2,7,1,1,14,False
18,uid://A001/X38cd/X1d,uid://A002/X137905e/X2bf01,10,1,10,1,1,10,True
7,uid://A001/X3645/X15f,uid://A002/X11604c2/X308b,8,1,8,1,1,8,True
11,uid://A001/X3833/X1982,uid://A002/X139bfe4/X6b3b,8,1,8,1,1,8,True
12,uid://A001/X3833/X198e,uid://A002/X13e367b/X3d5a,8,1,8,1,1,8,True
13,uid://A001/X3833/X1998,uid://A002/X139bfe4/X7ee8,8,1,8,1,1,8,True
16,uid://A001/X3845/X77a,uid://A002/X130a07a/X2acca,7,1,7,1,1,7,True
15,uid://A001/X3845/X776,uid://A002/X130a07a/X2b5ab,6,1,6,1,1,6,True


,member_ous_uid,asdm_uid,archive_rows,source_contexts,logical_spws,group_ous_values,proposal_values,expected_source_spw_rows,complete_source_spw_grid
29,uid://A001/X3955/X44,uid://A002/X13e7f29/Xc580,11,2,7,1,1,14,False


A failed parser, duplicate candidate key, or incomplete Source-Execution-by-SPW grid
is a reconstruction warning, not a row to discard. The production model
must retain the raw row and allow `UNRECONSTRUCTED` or `CONFLICTING`
status.


## 7. Empirical field ownership by reconstruction level

In [14]:
ownership_levels = {
    "MEMBER": ["member_ous_uid"],
    "SOURCE": ["member_ous_uid", "source_name"],
    "SOURCE_EXECUTION": ["member_ous_uid", "asdm_uid", "source_name"],
    "LOGICAL_SPW": ["member_ous_uid", "spw_id"],
    "SOURCE_SPW": ["member_ous_uid", "source_name", "spw_id"],
    "SOURCE_EXECUTION_SPW": ["member_ous_uid", "asdm_uid", "source_name", "spw_id"],
}

ownership_candidate_fields = [
    name for name in [
        "proposal_id", "group_ous_uid", "target_name", "s_ra", "s_dec",
        "s_fov", "s_region", "s_resolution", "spatial_resolution",
        "spatial_scale_max", "is_mosaic", "asdm_uid", "antenna_arrays",
        "t_min", "t_max", "frequency_support", "frequency", "bandwidth",
        "spectral_resolution", "velocity_resolution", "em_resolution",
        "em_min", "em_max", "em_res_power", "sensitivity_10kms",
        "cont_sensitivity_bandwidth", "pol_states", "band_list",
    ] if name in analysis_df.columns
]


def ownership_stability(dataframe, level_name, keys, field):
    available = dataframe.dropna(subset=keys)
    grouped = available.groupby(keys, dropna=False)[field]
    distinct = grouped.nunique(dropna=True)
    non_missing = grouped.count()
    groups_with_values = non_missing.gt(0)
    return {
        "level": level_name,
        "field": field,
        "total_groups": int(len(distinct)),
        "groups_with_values": int(groups_with_values.sum()),
        "conflicting_groups": int((distinct.gt(1) & groups_with_values).sum()),
        "stable_fraction_available": (
            float((distinct.le(1) & groups_with_values).sum() / groups_with_values.sum())
            if groups_with_values.any() else np.nan
        ),
    }


ownership_records = []
for level_name, keys in ownership_levels.items():
    for field in ownership_candidate_fields:
        ownership_records.append(ownership_stability(analysis_df, level_name, keys, field))

ownership_stability_df = pd.DataFrame(ownership_records)
ownership_pivot_df = ownership_stability_df.pivot(
    index="field", columns="level", values="stable_fraction_available"
)

display(ownership_pivot_df)
display(
    ownership_stability_df.loc[ownership_stability_df["conflicting_groups"] > 0]
    .sort_values(["field", "level"])
)

level,LOGICAL_SPW,MEMBER,SOURCE,SOURCE_EXECUTION,SOURCE_EXECUTION_SPW,SOURCE_SPW
field,,,,,,
antenna_arrays,0.95092,0.939394,1.000000,1.000000,1.0,1.0
asdm_uid,0.97546,0.969697,1.000000,1.000000,1.0,1.0
band_list,1.00000,1.000000,1.000000,1.000000,1.0,1.0
bandwidth,1.00000,0.727273,0.769231,0.769231,1.0,1.0
cont_sensitivity_bandwidth,0.90184,0.878788,1.000000,1.000000,1.0,1.0
em_max,0.90184,0.000000,0.000000,0.000000,1.0,1.0
em_min,0.90184,0.000000,0.000000,0.000000,1.0,1.0
em_res_power,1.00000,0.000000,0.000000,0.000000,1.0,1.0
em_resolution,0.90184,0.000000,0.000000,0.000000,1.0,1.0


,level,field,total_groups,groups_with_values,conflicting_groups,stable_fraction_available
96,LOGICAL_SPW,antenna_arrays,163,163,8,0.950920
12,MEMBER,antenna_arrays,33,33,2,0.939394
95,LOGICAL_SPW,asdm_uid,163,163,4,0.975460
11,MEMBER,asdm_uid,33,33,1,0.969697
17,MEMBER,bandwidth,33,33,9,0.727273
45,SOURCE,bandwidth,39,39,9,0.769231
73,SOURCE_EXECUTION,bandwidth,39,39,9,0.769231
109,LOGICAL_SPW,cont_sensitivity_bandwidth,163,163,16,0.901840
25,MEMBER,cont_sensitivity_bandwidth,33,33,4,0.878788
106,LOGICAL_SPW,em_max,163,163,16,0.901840


## 8. Repeated sources across ASDM associations and footprint scope

In [15]:
source_asdm_summary_df = (
    analysis_df.dropna(subset=["source_name"])
    .groupby(["member_ous_uid", "source_name"], dropna=False)
    .agg(
        asdm_count=("asdm_uid", "nunique"),
        row_count=("archive_row_index", "size"),
        region_count=("s_region", "nunique"),
        coordinate_count=("s_ra", "nunique"),
        mosaic_state_count=("is_mosaic", "nunique"),
        support_signature_count=("frequency_support", "nunique"),
    )
    .reset_index()
)

repeated_source_across_asdm_df = source_asdm_summary_df.loc[
    source_asdm_summary_df["asdm_count"] > 1
].copy()

print("Source Contexts repeated across multiple ASDM associations:", len(repeated_source_across_asdm_df))
display(repeated_source_across_asdm_df)

Source Contexts repeated across multiple ASDM associations: 0


,member_ous_uid,source_name,asdm_count,row_count,region_count,coordinate_count,mosaic_state_count,support_signature_count


In [16]:
def geometry_type(raw_region: object) -> str:
    if pd.isna(raw_region) or not str(raw_region).strip():
        return "MISSING"
    return str(raw_region).strip().split()[0].upper()


spatial_df = analysis_df.copy()
spatial_df["geometry_type"] = spatial_df["s_region"].map(geometry_type)
spatial_df["coordinate_signature"] = list(zip(spatial_df["s_ra"], spatial_df["s_dec"]))

spatial_context_df = (
    spatial_df.groupby(
        ["member_ous_uid", "asdm_uid", "source_name"], dropna=False
    )
    .agg(
        rows=("archive_row_index", "size"),
        coordinates=("coordinate_signature", "nunique"),
        regions=("s_region", "nunique"),
        geometry_types=("geometry_type", lambda s: tuple(sorted(set(s)))),
        mosaic_states=("is_mosaic", lambda s: tuple(sorted(set(s.dropna().astype(str))))),
        spws=("spw_id", "nunique"),
    )
    .reset_index()
)

display(
    spatial_df.groupby(["is_mosaic", "geometry_type"], dropna=False)
    .size()
    .rename("row_count")
    .reset_index()
)
display(spatial_context_df.loc[(spatial_context_df["coordinates"] > 1) | (spatial_context_df["regions"] > 1)])

,is_mosaic,geometry_type,row_count
0,F,CIRCLE,95
1,F,POLYGON,44
2,T,POLYGON,16
3,T,UNION,32


,member_ous_uid,asdm_uid,source_name,rows,coordinates,regions,geometry_types,mosaic_states,spws


In [17]:
footprint_scope_evidence = pd.DataFrame(
    [
        {
            "question": "Footprint stable within Source-Execution Context",
            "tested_contexts": len(spatial_context_df),
            "counterexamples": int(
                ((spatial_context_df["coordinates"] > 1) | (spatial_context_df["regions"] > 1)).sum()
            ),
            "evidence_status": "SAMPLE_SUPPORTED",
        },
        {
            "question": "Footprint belongs only to Source Context",
            "tested_contexts": len(repeated_source_across_asdm_df),
            "counterexamples": int(
                (
                    (repeated_source_across_asdm_df["region_count"] > 1)
                    | (repeated_source_across_asdm_df["coordinate_count"] > 1)
                ).sum()
            ) if len(repeated_source_across_asdm_df) else 0,
            "evidence_status": (
                "TESTABLE_IN_SAMPLE" if len(repeated_source_across_asdm_df) else "UNRESOLVED_NO_REPEATED_SOURCE_ACROSS_ASDM"
            ),
        },
    ]
)
display(footprint_scope_evidence)

,question,tested_contexts,counterexamples,evidence_status
0,Footprint stable within Source-Execution Context,39,0,SAMPLE_SUPPORTED
1,Footprint belongs only to Source Context,0,0,UNRESOLVED_NO_REPEATED_SOURCE_ACROSS_ASDM


## 9. Frequency-support parsing and context-scoped SPW mapping

In [18]:
def issue_codes(issues) -> tuple[str, ...]:
    return tuple(issue.code for issue in issues)


signature_columns = [
    "member_ous_uid", "asdm_uid", "source_name", "frequency_support"
]
signature_inventory_df = analysis_df[signature_columns].drop_duplicates().copy()

component_records = []
for row in signature_inventory_df.itertuples(index=False):
    result = parse_frequency_support(row.frequency_support)

    if not result.components:
        component_records.append(
            {
                "member_ous_uid": row.member_ous_uid,
                "asdm_uid": row.asdm_uid,
                "source_name": row.source_name,
                "frequency_support": row.frequency_support,
                "parser_version": result.parser_version,
                "result_parse_status": result.parse_status.value,
                "result_parse_issues": issue_codes(result.parse_issues),
                "component_index": None,
            }
        )
        continue

    for component in result.components:
        interval = component.frequency_interval
        component_records.append(
            {
                "member_ous_uid": row.member_ous_uid,
                "asdm_uid": row.asdm_uid,
                "source_name": row.source_name,
                "frequency_support": row.frequency_support,
                "parser_version": result.parser_version,
                "result_parse_status": result.parse_status.value,
                "result_parse_issues": issue_codes(result.parse_issues),
                "component_index": component.component_index,
                "component_parse_status": component.parse_status.value,
                "component_parse_issues": issue_codes(component.parse_issues),
                "component_validation_issues": issue_codes(component.validation_issues),
                "frequency_low_ghz": None if interval is None else (
                    interval.low * u.Unit(interval.unit)
                ).to_value(u.GHz),
                "frequency_high_ghz": None if interval is None else (
                    interval.high * u.Unit(interval.unit)
                ).to_value(u.GHz),
            }
        )

frequency_component_df = pd.DataFrame(component_records)
frequency_component_df["component_centre_ghz"] = (
    frequency_component_df["frequency_low_ghz"]
    + frequency_component_df["frequency_high_ghz"]
) / 2

display(
    frequency_component_df["result_parse_status"]
    .value_counts(dropna=False)
    .rename_axis("parse_status")
    .reset_index(name="signature_count")
)

,parse_status,signature_count
0,PARSED,176
1,FAILED,2


In [19]:
support_context_columns = [
    "member_ous_uid", "asdm_uid", "source_name", "frequency_support"
]

row_context_counts_df = (
    analysis_df.groupby(support_context_columns, dropna=False)
    .size()
    .reset_index(name="archive_row_count")
)
component_context_counts_df = (
    frequency_component_df.groupby(support_context_columns, dropna=False)
    .agg(parsed_component_count=("component_index", "count"))
    .reset_index()
)
context_count_comparison_df = row_context_counts_df.merge(
    component_context_counts_df,
    on=support_context_columns,
    how="outer",
    validate="one_to_one",
)
context_count_comparison_df["count_agreement"] = (
    context_count_comparison_df["archive_row_count"]
    == context_count_comparison_df["parsed_component_count"]
)

display(
    context_count_comparison_df["count_agreement"]
    .value_counts(dropna=False)
    .rename_axis("count_agreement")
    .reset_index(name="context_count")
)
display(context_count_comparison_df.loc[~context_count_comparison_df["count_agreement"]])

,count_agreement,context_count
0,True,37
1,False,2


,member_ous_uid,asdm_uid,source_name,frequency_support,archive_row_count,parsed_component_count,count_agreement
29,uid://A001/X3955/X44,uid://A002/X13e7f29/Xc580,Moon_full,"{229.98GHz,2000000.00kHz,20.8mJy/beam@10km/s,1.3mJy/beam@native, XX YY} U {231.98GHz,2000000.00kHz,21.1mJy/beam@10km...",7,0,False
30,uid://A001/X3955/X44,uid://A002/X13e7f29/Xc580,Moon_reg,"{229.98GHz,2000000.00kHz,9.3mJy/beam@10km/s,577.4uJy/beam@native, XX YY} U {231.98GHz,2000000.00kHz,9.4mJy/beam@10km...",4,0,False


In [20]:
component_groups = {
    key: group.copy()
    for key, group in frequency_component_df.groupby(
        support_context_columns, dropna=False, sort=False
    )
}

assignment_records = []
for context_key, row_group in analysis_df.groupby(
    support_context_columns, dropna=False, sort=False
):
    component_group = component_groups.get(context_key)

    if component_group is None or len(row_group) != len(component_group):
        for row in row_group.itertuples(index=False):
            assignment_records.append(
                {
                    "archive_row_index": row.archive_row_index,
                    "assignment_status": "COUNT_MISMATCH",
                }
            )
        continue

    sorted_rows = row_group.sort_values(["frequency", "spw_id"])
    sorted_components = component_group.sort_values(
        ["component_centre_ghz", "component_index"]
    )

    for row, component in zip(
        sorted_rows.itertuples(index=False),
        sorted_components.itertuples(index=False),
        strict=True,
    ):
        inside = component.frequency_low_ghz <= row.frequency <= component.frequency_high_ghz
        assignment_records.append(
            {
                "archive_row_index": row.archive_row_index,
                "obs_id": row.obs_id,
                "member_ous_uid": row.member_ous_uid,
                "asdm_uid": row.asdm_uid,
                "source_name": row.source_name,
                "spw_id": row.spw_id,
                "frequency_support": row.frequency_support,
                "component_index": component.component_index,
                "frequency_inside_interval": inside,
                "centre_difference_mhz": abs(
                    row.frequency - component.component_centre_ghz
                ) * 1e3,
                "assignment_status": "ASSIGNED" if inside else "OUTSIDE_INTERVAL",
            }
        )

assignment_df = pd.DataFrame(assignment_records)
display(
    assignment_df["assignment_status"]
    .value_counts(dropna=False)
    .rename_axis("assignment_status")
    .reset_index(name="row_count")
)

assigned_df = assignment_df.loc[assignment_df["assignment_status"] == "ASSIGNED"].copy()
print(
    "Unique when scoped to Source-Execution-SPW:",
    not assigned_df.duplicated(
        ["member_ous_uid", "asdm_uid", "source_name", "spw_id"]
    ).any(),
)
print(
    "Unique when scoped only to Member-SPW:",
    not assigned_df.duplicated(["member_ous_uid", "spw_id"]).any(),
)

,assignment_status,row_count
0,ASSIGNED,176
1,COUNT_MISMATCH,11


Unique when scoped to Source-Execution-SPW: True
Unique when scoped only to Member-SPW: False


## 10. Antenna-array evidence and standard ObsCore cross-checks

In [21]:
ANTENNA_CODE_PATTERN = re.compile(r"(?:^|\s)[^:]*:(?P<code>[A-Za-z]+)\d+")


def antenna_prefixes(raw_value: object) -> tuple[str, ...]:
    if pd.isna(raw_value):
        return tuple()
    return tuple(sorted(set(ANTENNA_CODE_PATTERN.findall(str(raw_value)))))


def classify_array_evidence(prefixes: tuple[str, ...]) -> str:
    prefix_set = set(prefixes)
    classes = []
    if prefix_set & {"DA", "DV"}:
        classes.append("12M_LIKE")
    if "CM" in prefix_set:
        classes.append("7M_LIKE")
    if "PM" in prefix_set:
        classes.append("TP_LIKE")
    return "+".join(classes) if classes else "UNKNOWN"


analysis_df["antenna_prefixes"] = analysis_df["antenna_arrays"].map(antenna_prefixes)
analysis_df["array_evidence_class"] = analysis_df["antenna_prefixes"].map(classify_array_evidence)

display(
    analysis_df.groupby(["array_evidence_class", "antenna_prefixes"], dropna=False)
    .size()
    .rename("row_count")
    .reset_index()
    .sort_values("row_count", ascending=False)
)

,array_evidence_class,antenna_prefixes,row_count
0,12M_LIKE,"(DA, DV)",87
3,12M_LIKE+TP_LIKE,"(DA, DV, PM)",36
5,7M_LIKE,"(CM,)",25
2,12M_LIKE+7M_LIKE+TP_LIKE,"(CM, DA, DV, PM)",12
6,TP_LIKE,"(PM,)",11
1,12M_LIKE+7M_LIKE,"(CM, DA, DV)",8
4,12M_LIKE+TP_LIKE,"(DV, PM)",8


In [22]:
cross_check_records = []

if {"em_min", "em_max", "frequency"}.issubset(analysis_df.columns):
    valid = analysis_df[["em_min", "em_max", "frequency"]].dropna().copy()
    speed_of_light = c.to_value(u.m / u.s)
    valid["obscore_frequency_low_ghz"] = speed_of_light / valid["em_max"] / 1e9
    valid["obscore_frequency_high_ghz"] = speed_of_light / valid["em_min"] / 1e9
    valid["row_frequency_inside_obscore_interval"] = (
        (valid["frequency"] >= valid["obscore_frequency_low_ghz"])
        & (valid["frequency"] <= valid["obscore_frequency_high_ghz"])
    )
    cross_check_records.append(
        {
            "cross_check": "row frequency inside em_min/em_max-derived interval",
            "eligible_rows": len(valid),
            "passing_rows": int(valid["row_frequency_inside_obscore_interval"].sum()),
        }
    )

if {"s_resolution", "spatial_resolution"}.issubset(analysis_df.columns):
    valid = analysis_df[["s_resolution", "spatial_resolution"]].dropna().copy()
    valid["absolute_difference_arcsec"] = (
        valid["s_resolution"] - valid["spatial_resolution"]
    ).abs()
    cross_check_records.append(
        {
            "cross_check": "s_resolution vs spatial_resolution exact equality",
            "eligible_rows": len(valid),
            "passing_rows": int(valid["absolute_difference_arcsec"].eq(0).sum()),
            "maximum_absolute_difference": valid["absolute_difference_arcsec"].max(),
        }
    )

cross_check_df = pd.DataFrame(cross_check_records)
display(cross_check_df)

for field in ["s_fov", "spatial_scale_max", "dataproduct_type", "calib_level", "type", "pwv"]:
    if field in analysis_df.columns:
        print(f"\n{field} value summary")
        display(
            analysis_df[field]
            .value_counts(dropna=False)
            .head(30)
            .rename_axis(field)
            .reset_index(name="row_count")
        )

,cross_check,eligible_rows,passing_rows,maximum_absolute_difference
0,row frequency inside em_min/em_max-derived interval,187,182,NaN
1,s_resolution vs spatial_resolution exact equality,187,176,2.37189



s_fov value summary


,s_fov,row_count
0,1.284477,10
1,0.001878,8
2,0.001878,8
3,0.001878,8
4,0.001987,8
5,0.007926,7
6,0.006768,7
7,4.904239,6
8,0.015018,5
9,0.004610,4



spatial_scale_max value summary


,spatial_scale_max,row_count
0,12.635589,10
1,1.946542,8
2,2.144062,8
3,0.685347,8
4,3.947074,8
5,19.628257,7
6,389.852545,7
7,34.691365,6
8,12.729274,5
9,10.917023,4


...

,calib_level,row_count
0,2,187



type value summary


,type,row_count
0,S,117
1,T,34
2,E,28
3,SV,4
4,V,4



pwv value summary


,pwv,row_count
0,1.800000,21
1,0.910000,15
2,0.968139,12
3,1.485904,10
4,0.514900,8
5,0.460215,8
6,1.146463,8
7,0.696352,8
8,0.363570,8
9,7.072905,5


Array classifications above are labeled evidence, not authoritative domain
values. Standard ObsCore fields are used as independent cross-checks and
remain separate from ALMA-specific metadata even when they agree numerically.

## 11. TAP completeness, overflow, and empty-result behaviour

In [23]:
largest_member = (
    analysis_df.groupby("member_ous_uid")
    .size()
    .sort_values(ascending=False)
    .index[0]
)
largest_member_expected = int(
    (analysis_df["member_ous_uid"] == largest_member).sum()
)

limited_query = f"""
SELECT
    member_ous_uid,
    obs_id,
    frequency
FROM ivoa.obscore
WHERE science_observation = 'T'
  AND member_ous_uid = {quote_adql_string(largest_member)}
"""

limited_maxrec = max(1, min(5, largest_member_expected - 1))
limited_run = run_tap_query(
    limited_query,
    maxrec=limited_maxrec,
    label="Intentional MAXREC truncation experiment",
)
query_runs.append(limited_run)

truncation_experiment_df = pd.DataFrame(
    [{
        "member_ous_uid": largest_member,
        "expected_rows_from_complete_sample": largest_member_expected,
        "requested_maxrec": limited_maxrec,
        "retrieved_rows": limited_run.retrieved_rows,
        "count_proves_incomplete": limited_run.retrieved_rows < largest_member_expected,
        "query_status": limited_run.query_status,
        "warnings": limited_run.warning_messages,
    }]
)
display(truncation_experiment_df)


--- Intentional MAXREC truncation experiment ---
Endpoint: https://almascience.eso.org/tap
MAXREC: 5
ADQL:

SELECT
    member_ous_uid,
    obs_id,
    frequency
FROM ivoa.obscore
WHERE science_observation = 'T'
  AND member_ous_uid = 'uid://A002/X845868/X3a'

Retrieved rows: 5
QUERY_STATUS: ('OK', 'OVERFLOW')
Warnings: <none>


,member_ous_uid,expected_rows_from_complete_sample,requested_maxrec,retrieved_rows,count_proves_incomplete,query_status,warnings
0,uid://A002/X845868/X3a,12,5,5,True,"(OK, OVERFLOW)",()


In [24]:
impossible_member = "uid://A000/THIS_MEMBER_SHOULD_NOT_EXIST/X0"
empty_query = f"""
SELECT member_ous_uid, obs_id
FROM ivoa.obscore
WHERE member_ous_uid = {quote_adql_string(impossible_member)}
"""

empty_run = run_tap_query(empty_query, maxrec=10, label="Empty-result control")
query_runs.append(empty_run)

print("Empty result returned zero rows:", empty_run.retrieved_rows == 0)
print("A zero-row result is valid only when query execution is complete and successful.")


--- Empty-result control ---
Endpoint: https://almascience.eso.org/tap
MAXREC: 10
ADQL:

SELECT member_ous_uid, obs_id
FROM ivoa.obscore
WHERE member_ous_uid = 'uid://A000/THIS_MEMBER_SHOULD_NOT_EXIST/X0'

Retrieved rows: 0
QUERY_STATUS: ('OK',)
Warnings: <none>
Empty result returned zero rows: True
A zero-row result is valid only when query execution is complete and successful.


In [25]:
query_provenance_df = pd.DataFrame(
    [
        {
            "label": run.label,
            "endpoint": run.endpoint,
            "started_at_utc": run.started_at_utc,
            "finished_at_utc": run.finished_at_utc,
            "maxrec": run.maxrec,
            "retrieved_rows": run.retrieved_rows,
            "query_status": run.query_status,
            "warnings": run.warning_messages,
            "adql_sha256": hashlib.sha256(run.adql.encode("utf-8")).hexdigest(),
        }
        for run in query_runs
    ]
)
display(query_provenance_df)

,label,endpoint,started_at_utc,finished_at_utc,maxrec,retrieved_rows,query_status,warnings,adql_sha256
0,Retrieve live ivoa.obscore schema,https://almascience.eso.org/tap,2026-08-25T12:59:37.036486+00:00,2026-08-25T12:59:37.903958+00:00,5000,73,"(OK,)",(),7668db0fd5524c6b01c928fe85f9755927f9ee131e62a72eb278f0bc520e57ac
1,Discover early_cycles,https://almascience.eso.org/tap,2026-08-25T12:59:38.008739+00:00,2026-08-25T12:59:38.437282+00:00,80,80,"(OK,)",(),45325b884d04c3c1a5261955904ba63e83cf8e8e2c050039efd1f87e78226f19
2,Discover middle_cycles,https://almascience.eso.org/tap,2026-08-25T12:59:38.440770+00:00,2026-08-25T12:59:39.409598+00:00,80,80,"(OK,)",(),3b4ea8ca78a6f29d0c938828b11419ab4946f211ad0126fbd13b4670d4072dbf
3,Discover recent_cycles,https://almascience.eso.org/tap,2026-08-25T12:59:39.415520+00:00,2026-08-25T12:59:40.586473+00:00,80,80,"(OK,)",(),f053d7f2dbe86b51e3f22745c5f99e7d42aa2ccd183f3bf830e3bd8766288dba
4,Discover band3,https://almascience.eso.org/tap,2026-08-25T12:59:40.593914+00:00,2026-08-25T12:59:46.691600+00:00,80,80,"(OK,)",(),3d67414035d859a75708cc9246dc648f195546c18ad74b6144535e3a29bfe1d9
5,Discover band10,https://almascience.eso.org/tap,2026-08-25T12:59:46.697933+00:00,2026-08-25T12:59:50.425066+00:00,80,80,"(OK,)",(),ead9ab16ff01167d088d44991ece2683a890217b9b3e6272ac46333187692916
6,Discover mosaic,https://almascience.eso.org/tap,2026-08-25T12:59:50.430809+00:00,2026-08-25T12:59:51.198893+00:00,80,80,"(OK,)",(),ea78ff8d73aa4c8d1218ffb057a15419b23df7f9318b9a18b49126dbc7d0f93d
7,Discover non_mosaic,https://almascience.eso.org/tap,2026-08-25T12:59:51.202625+00:00,2026-08-25T12:59:56.252576+00:00,80,80,"(OK,)",(),d76f6481dfd7f503b1608eaf093652eef9e197dbab8038998be22c7ec6038efe
8,Discover cm_antenna_evidence,https://almascience.eso.org/tap,2026-08-25T12:59:56.267067+00:00,2026-08-25T12:59:57.405619+00:00,80,80,"(OK,)",(),b510346e3c6cd8243d03647af25b6e430c889a43fc162d754a0107b3e55e064e
9,Discover da_dv_antenna_evidence,https://almascience.eso.org/tap,2026-08-25T12:59:57.408816+00:00,2026-08-25T13:00:02.585345+00:00,80,80,"(OK,)",(),ab803406eb28a252d0f35ce695753679a0ba2215d07c33c4178f80bf8a632a5c


## 12. Defensive reconstruction with controlled anomalies

In [26]:
def missing_or_blank_mask(series: pd.Series) -> pd.Series:

    normalized_text = series.astype("string").str.strip()

    return (
        series.isna()
        | normalized_text.eq("").fillna(False)
    )


def validate_reconstruction_input(dataframe: pd.DataFrame) -> pd.DataFrame:
    records = []

    parsed = pd.DataFrame(
        dataframe["obs_id"].map(parse_obs_id_structure).tolist()
    )

    duplicate_affected_mask = (
        dataframe["obs_id"].notna()
        & dataframe.duplicated("obs_id", keep=False)
    )

    records.append(
        {
            "check": "obs_id parse failures",
            "affected_rows": int(
                parsed["obs_id_parse_status"].ne("PARSED").sum()
            ),
            "severity": "WARNING",
        }
    )

    records.append(
        {
            "check": "duplicate raw obs_id",
            # Both rows in a duplicate pair are affected.
            "affected_rows": int(duplicate_affected_mask.sum()),
            "severity": "WARNING",
        }
    )

    records.append(
        {
            "check": "missing or blank frequency_support",
            "affected_rows": int(
                missing_or_blank_mask(
                    dataframe["frequency_support"]
                ).sum()
            ),
            "severity": "WARNING",
        }
    )

    records.append(
        {
            "check": "blank Group OUS identifiers",
            "affected_rows": int(
                missing_or_blank_mask(
                    dataframe["group_ous_uid"]
                ).sum()
            ),
            "severity": "NORMALIZE_TO_MISSING",
        }
    )

    return pd.DataFrame(records)


def make_unique_control_obs_id(raw_obs_id: object, label: str) -> str:

    text = str(raw_obs_id)
    prefix, spw_id = text.rsplit(".spw.", maxsplit=1)

    return f"{prefix}.spw.{spw_id}__CONTROL_{label}"


valid_fixture_mask = (
    analysis_df["obs_id_parse_status"].eq("PARSED")
    & ~missing_or_blank_mask(analysis_df["frequency_support"])
    & ~missing_or_blank_mask(analysis_df["group_ous_uid"])
)

anomaly_fixture_df = (
    analysis_df.loc[valid_fixture_mask, selected_archive_columns]
    .head(4)
    .copy()
)

if len(anomaly_fixture_df) != 4:
    raise RuntimeError(
        "Could not find four clean rows for the controlled anomaly fixture."
    )


malformed = anomaly_fixture_df.iloc[[0]].copy()
malformed["obs_id"] = "unexpected-format"


duplicate = anomaly_fixture_df.iloc[[1]].copy()


missing_support = anomaly_fixture_df.iloc[[2]].copy()
missing_support["obs_id"] = missing_support["obs_id"].map(
    lambda value: make_unique_control_obs_id(
        value,
        "MISSING_SUPPORT",
    )
)
missing_support["frequency_support"] = None


blank_group = anomaly_fixture_df.iloc[[3]].copy()
blank_group["obs_id"] = blank_group["obs_id"].map(
    lambda value: make_unique_control_obs_id(
        value,
        "BLANK_GROUP",
    )
)
blank_group["group_ous_uid"] = ""


controlled_anomaly_df = pd.concat(
    [
        anomaly_fixture_df,
        malformed,
        duplicate,
        missing_support,
        blank_group,
    ],
    ignore_index=True,
)

controlled_anomaly_result_df = validate_reconstruction_input(
    controlled_anomaly_df
)

display(controlled_anomaly_result_df)

,check,affected_rows,severity
0,obs_id parse failures,1,WARNING
1,duplicate raw obs_id,2,WARNING
2,missing or blank frequency_support,1,WARNING
3,blank Group OUS identifiers,1,NORMALIZE_TO_MISSING


In [27]:
expected_anomaly_counts = {
    "obs_id parse failures": 1,
    "duplicate raw obs_id": 2,
    "missing or blank frequency_support": 1,
    "blank Group OUS identifiers": 1,
}

actual_anomaly_counts = (
    controlled_anomaly_result_df
    .set_index("check")["affected_rows"]
    .to_dict()
)

assert actual_anomaly_counts == expected_anomaly_counts, (
    f"Unexpected controlled-anomaly result: {actual_anomaly_counts}"
)

duplicate_surplus_rows = int(
    controlled_anomaly_df.duplicated("obs_id", keep="first").sum()
)

print("Controlled anomaly experiment passed.")
print("Duplicate affected rows:", actual_anomaly_counts["duplicate raw obs_id"])
print("Duplicate surplus rows:", duplicate_surplus_rows)

Controlled anomaly experiment passed.
Duplicate affected rows: 2
Duplicate surplus rows: 1


## Targeted follow-up A: non-Cartesian Member and brace-style support

This section inspects the observed counterexample
`uid://A001/X3955/X44`.

The goal is to determine:

1. which SPWs belong to each source;
2. the grammar of brace-style frequency-support components;
3. how component tokens relate to row frequency, bandwidth, and spectral
   resolution;
4. the observed antenna-array evidence;
5. whether `s_resolution` and `spatial_resolution` agree.

In [28]:
TARGET_MEMBER_OUS_UID = "uid://A001/X3955/X44"

target_member_df = (
    analysis_df.loc[
        analysis_df["member_ous_uid"].eq(TARGET_MEMBER_OUS_UID)
    ]
    .copy()
    .sort_values(
        ["source_name", "frequency", "spw_id"],
        kind="stable",
    )
)

print("Observed rows:", len(target_member_df))
print("Observed sources:", target_member_df["source_name"].nunique())
print("Observed Member-level SPWs:", target_member_df["spw_id"].nunique())

if len(target_member_df) != 11:
    warnings.warn(
        "The previous 04b run observed 11 rows, but this run returned "
        f"{len(target_member_df)}. Check whether the live Archive changed."
    )

target_display_columns = [
    "archive_row_index",
    "member_ous_uid",
    "asdm_uid",
    "source_name",
    "spw_id",
    "frequency",
    "bandwidth",
    "spectral_resolution",
    "velocity_resolution",
    "frequency_support",
    "antenna_arrays",
    "antenna_prefixes",
    "array_evidence_class",
    "is_mosaic",
    "s_resolution",
    "spatial_resolution",
    "s_region",
    "type",
    "dataproduct_type",
]

target_display_columns = [
    column
    for column in target_display_columns
    if column in target_member_df.columns
]

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.width", 240,
):
    display(target_member_df[target_display_columns])

Observed rows: 11
Observed sources: 2
Observed Member-level SPWs: 7


,archive_row_index,member_ous_uid,asdm_uid,source_name,spw_id,frequency,bandwidth,spectral_resolution,velocity_resolution,frequency_support,antenna_arrays,antenna_prefixes,array_evidence_class,is_mosaic,s_resolution,spatial_resolution,s_region,type,dataproduct_type
166,166,uid://A001/X3955/X44,uid://A002/X13e7f29/Xc580,Moon_full,4,229.981328,2.000000e+09,2000000.0,2.417681e+06,"{229.98GHz,2000000.00kHz,20.8mJy/beam@10km/s,1.3mJy/beam@native, XX YY} U {231.98GHz,2000000.00kHz,21.1mJy/beam@10km/s,1.3mJy/beam@native, XX YY} U {245.98GHz,2000000.00kHz,21.2mJy/beam@10km/s,1.4mJy/beam@native, XX YY} U {247.98GHz,2000000.00kHz,22.5mJy/beam@10km/s,1.4mJy/beam@native, XX YY}",T701:PM04 T702:PM03 T703:PM01 T704:PM02,"(PM,)",TP_LIKE,F,24.365779,21.993889,Circle ICRS 28.938550 17.240772 0.003384,S,image
168,168,uid://A001/X3955/X44,uid://A002/X13e7f29/Xc580,Moon_full,0,229.981374,2.000000e+09,2000000.0,2.417681e+06,"{229.98GHz,2000000.00kHz,20.8mJy/beam@10km/s,1.3mJy/beam@native, XX YY} U {231.98GHz,2000000.00kHz,21.1mJy/beam@10km/s,1.3mJy/beam@native, XX YY} U {245.98GHz,2000000.00kHz,21.2mJy/beam@10km/s,1.4mJy/beam@native, XX YY} U {247.98GHz,2000000.00kHz,22.5mJy/beam@10km/s,1.4mJy/beam@native, XX YY}",T701:PM04 T702:PM03 T703:PM01 T704:PM02,"(PM,)",TP_LIKE,F,24.365779,21.993889,Circle ICRS 28.938550 17.240772 0.003384,S,image
164,164,uid://A001/X3955/X44,uid://A002/X13e7f29/Xc580,Moon_full,1,231.981212,2.000000e+09,2000000.0,2.417681e+06,"{229.98GHz,2000000.00kHz,20.8mJy/beam@10km/s,1.3mJy/beam@native, XX YY} U {231.98GHz,2000000.00kHz,21.1mJy/beam@10km/s,1.3mJy/beam@native, XX YY} U {245.98GHz,2000000.00kHz,21.2mJy/beam@10km/s,1.4mJy/beam@native, XX YY} U {247.98GHz,2000000.00kHz,22.5mJy/beam@10km/s,1.4mJy/beam@native, XX YY}",T701:PM04 T702:PM03 T703:PM01 T704:PM02,"(PM,)",TP_LIKE,F,24.365779,21.993889,Circle ICRS 28.938550 17.240772 0.003384,S,image
167,167,uid://A001/X3955/X44,uid://A002/X13e7f29/Xc580,Moon_full,5,231.981212,2.000000e+09,2000000.0,2.417681e+06,"{229.98GHz,2000000.00kHz,20.8mJy/beam@10km/s,1.3mJy/beam@native, XX YY} U {231.98GHz,2000000.00kHz,21.1mJy/beam@10km/s,1.3mJy/beam@native, XX YY} U {245.98GHz,2000000.00kHz,21.2mJy/beam@10km/s,1.4mJy/beam@native, XX YY} U {247.98GHz,2000000.00kHz,22.5mJy/beam@10km/s,1.4mJy/beam@native, XX YY}",T701:PM04 T702:PM03 T703:PM01 T704:PM02,"(PM,)",TP_LIKE,F,24.365779,21.993889,Circle ICRS 28.938550 17.240772 0.003384,S,image
169,169,uid://A001/X3955/X44,uid://A002/X13e7f29/Xc580,Moon_full,2,245.980078,2.000000e+09,2000000.0,2.417681e+06,"{229.98GHz,2000000.00kHz,20.8mJy/beam@10km/s,1.3mJy/beam@native, XX YY} U {231.98GHz,2000000.00kHz,21.1mJy/beam@10km/s,1.3mJy/beam@native, XX YY} U {245.98GHz,2000000.00kHz,21.2mJy/beam@10km/s,1.4mJy/beam@native, XX YY} U {247.98GHz,2000000.00kHz,22.5mJy/beam@10km/s,1.4mJy/beam@native, XX YY}",T701:PM04 T702:PM03 T703:PM01 T704:PM02,"(PM,)",TP_LIKE,F,24.365779,21.993889,Circle ICRS 28.938550 17.240772 0.003384,S,image
165,165,uid://A001/X3955/X44,uid://A002/X13e7f29/Xc580,Moon_full,3,247.979917,2.000000e+09,2000000.0,2.417681e+06,"{229.98GHz,2000000.00kHz,20.8mJy/beam@10km/s,1.3mJy/beam@native, XX YY} U {231.98GHz,2000000.00kHz,21.1mJy/beam@10km/s,1.3mJy/beam@native, XX YY} U {245.98GHz,2000000.00kHz,21.2mJy/beam@10km/s,1.4mJy/beam@native, XX YY} U {247.98GHz,2000000.00kHz,22.5mJy/beam@10km/s,1.4mJy/beam@native, XX YY}",T701:PM04 T702:PM03 T703:PM01 T704:PM02,"(PM,)",TP_LIKE,F,24.365779,21.993889,Circle ICRS 28.938550 17.240772 0.003384,S,image
170,170,uid://A001/X3955/X44,uid://A002/X13e7f29/Xc580,Moon_full,6,247.979917,2.000000e+09,2000000.0,2.417681e+06,"{229.98GHz,2000000.00kHz,20.8mJy/beam@10km/s,1.3mJy/beam@native, XX YY} U {231.98GHz,2000000.00kHz,21.1mJy/beam@10km/s,1.3mJy/beam@native, XX YY} U {245.98GHz,2000000.00kHz,21.2mJy/beam@10km/s,1.4mJy/beam@native, XX YY} U {247.98GHz,2000000.00kHz,22.5mJy/beam@10km/s,1.4mJy/beam@native, XX YY}",T701:PM04 T702:PM03 T703:PM01 T704:PM02,"(PM,)",TP_LIKE,F,24.365779,21.993889,Circle ICRS 28.938550 17

In [29]:
source_spw_inventory_df = (
    target_member_df
    .groupby(
        ["member_ous_uid", "asdm_uid", "source_name"],
        dropna=False,
    )
    .agg(
        archive_rows=("archive_row_index", "size"),
        spw_count=("spw_id", "nunique"),
        spw_ids=(
            "spw_id",
            lambda values: tuple(
                sorted(
                    values.dropna().astype(str).unique()
                )
            ),
        ),
        row_frequencies_ghz=(
            "frequency",
            lambda values: tuple(
                sorted(
                    round(float(value), 9)
                    for value in values.dropna().unique()
                )
            ),
        ),
        support_signatures=("frequency_support", "nunique"),
        antenna_prefix_sets=(
            "antenna_prefixes",
            lambda values: tuple(
                sorted(set(values))
            ),
        ),
        array_evidence_classes=(
            "array_evidence_class",
            lambda values: tuple(
                sorted(set(values.dropna().astype(str)))
            ),
        ),
    )
    .reset_index()
)

display(source_spw_inventory_df)

,member_ous_uid,asdm_uid,source_name,archive_rows,spw_count,spw_ids,row_frequencies_ghz,support_signatures,antenna_prefix_sets,array_evidence_classes
0,uid://A001/X3955/X44,uid://A002/X13e7f29/Xc580,Moon_full,7,7,"(0, 1, 2, 3, 4, 5, 6)","(229.981328176, 229.981374189, 231.981212225, 245.98007848, 247.979916516)",1,"((PM,),)","(TP_LIKE,)"
1,uid://A001/X3955/X44,uid://A002/X13e7f29/Xc580,Moon_reg,4,4,"(0, 1, 2, 3)","(229.981382307, 231.981220414, 245.980087163, 247.97992527)",1,"((PM,),)","(TP_LIKE,)"


In [30]:
member_level_spws = set(
    target_member_df["spw_id"].dropna().astype(str)
)

source_spw_set_df = (
    target_member_df
    .groupby("source_name", dropna=False)["spw_id"]
    .agg(
        lambda values: set(values.dropna().astype(str))
    )
    .rename("observed_spws")
    .reset_index()
)

source_spw_set_df["member_level_spws"] = [
    member_level_spws
    for _ in range(len(source_spw_set_df))
]

source_spw_set_df["missing_from_source"] = (
    source_spw_set_df["observed_spws"]
    .map(lambda observed: member_level_spws - observed)
)

source_spw_set_df["uses_all_member_spws"] = (
    source_spw_set_df["missing_from_source"]
    .map(lambda missing: len(missing) == 0)
)

display(source_spw_set_df)

,source_name,observed_spws,member_level_spws,missing_from_source,uses_all_member_spws
0,Moon_full,"{2, 1, 0, 4, 6, 5, 3}","{2, 1, 0, 4, 6, 5, 3}",{},True
1,Moon_reg,"{2, 1, 3, 0}","{2, 1, 0, 4, 6, 5, 3}","{5, 6, 4}",False


In [31]:
BRACE_COMPONENT_PATTERN = re.compile(r"\{(?P<body>[^{}]+)\}")

GHZ_TOKEN_PATTERN = re.compile(
    r"^\s*(?P<value>[+-]?\d+(?:\.\d+)?)\s*GHz\s*$",
    flags=re.IGNORECASE,
)

KHZ_TOKEN_PATTERN = re.compile(
    r"^\s*(?P<value>[+-]?\d+(?:\.\d+)?)\s*kHz\s*$",
    flags=re.IGNORECASE,
)


def parse_numeric_token(
    token: str,
    pattern: re.Pattern,
) -> float:
    match = pattern.fullmatch(token)

    if match is None:
        return np.nan

    return float(match.group("value"))


def decimal_places_from_token(
    token: str,
    pattern: re.Pattern,
) -> int | None:
    """Return displayed decimal places without inventing extra precision."""
    match = pattern.fullmatch(token)
    if match is None:
        return None

    numeric_text = match.group("value")
    if "." not in numeric_text:
        return 0

    return len(numeric_text.rsplit(".", maxsplit=1)[1])


brace_signature_columns = [
    "member_ous_uid",
    "asdm_uid",
    "source_name",
    "frequency_support",
]

brace_signature_df = (
    target_member_df[brace_signature_columns]
    .drop_duplicates()
    .copy()
)

brace_component_records = []

for row in brace_signature_df.itertuples(index=False):
    raw_support = str(row.frequency_support)
    bodies = BRACE_COMPONENT_PATTERN.findall(raw_support)

    for component_index, body in enumerate(bodies):
        tokens = [
            token.strip()
            for token in body.split(",")
        ]

        record = {
            "member_ous_uid": row.member_ous_uid,
            "asdm_uid": row.asdm_uid,
            "source_name": row.source_name,
            "frequency_support": row.frequency_support,
            "component_index": component_index,
            "component_grammar": "CENTRE_RESOLUTION",
            "token_count": len(tokens),
            "token_1_raw": tokens[0] if len(tokens) > 0 else None,
            "token_2_raw": tokens[1] if len(tokens) > 1 else None,
            "token_3_raw": tokens[2] if len(tokens) > 2 else None,
            "token_4_raw": tokens[3] if len(tokens) > 3 else None,
            "token_5_raw": tokens[4] if len(tokens) > 4 else None,
        }

        record["token_1_ghz"] = parse_numeric_token(
            record["token_1_raw"] or "",
            GHZ_TOKEN_PATTERN,
        )
        record["token_2_khz"] = parse_numeric_token(
            record["token_2_raw"] or "",
            KHZ_TOKEN_PATTERN,
        )

        token_1_decimal_places = decimal_places_from_token(
            record["token_1_raw"] or "",
            GHZ_TOKEN_PATTERN,
        )
        record["token_1_decimal_places"] = token_1_decimal_places

        if token_1_decimal_places is None:
            record["token_1_representation_step_mhz"] = np.nan
            record["token_1_representation_tolerance_mhz"] = np.nan
        else:
            # Example: 229.98 GHz has a displayed step of 0.01 GHz = 10 MHz.
            representation_step_mhz = 10 ** (-token_1_decimal_places) * 1e3
            record["token_1_representation_step_mhz"] = representation_step_mhz
            record["token_1_representation_tolerance_mhz"] = (
                representation_step_mhz / 2
            )

        brace_component_records.append(record)

brace_component_df = pd.DataFrame(brace_component_records)

display(
    brace_component_df[
        [
            "source_name",
            "component_index",
            "component_grammar",
            "token_count",
            "token_1_raw",
            "token_1_ghz",
            "token_1_decimal_places",
            "token_1_representation_tolerance_mhz",
            "token_2_raw",
            "token_2_khz",
            "token_3_raw",
            "token_4_raw",
            "token_5_raw",
        ]
    ]
)

display(
    brace_component_df
    .groupby("source_name", dropna=False)
    .agg(
        parsed_brace_components=("component_index", "size"),
        token_counts=("token_count", lambda values: tuple(sorted(set(values)))),
        token_1_frequencies_ghz=(
            "token_1_ghz",
            lambda values: tuple(sorted(values.dropna())),
        ),
        representation_tolerances_mhz=(
            "token_1_representation_tolerance_mhz",
            lambda values: tuple(sorted(set(values.dropna()))),
        ),
    )
    .reset_index()
)


,source_name,component_index,component_grammar,token_count,token_1_raw,token_1_ghz,token_1_decimal_places,token_1_representation_tolerance_mhz,token_2_raw,token_2_khz,token_3_raw,token_4_raw,token_5_raw
0,Moon_full,0,CENTRE_RESOLUTION,5,229.98GHz,229.98,2,5.0,2000000.00kHz,2000000.0,20.8mJy/beam@10km/s,1.3mJy/beam@native,XX YY
1,Moon_full,1,CENTRE_RESOLUTION,5,231.98GHz,231.98,2,5.0,2000000.00kHz,2000000.0,21.1mJy/beam@10km/s,1.3mJy/beam@native,XX YY
2,Moon_full,2,CENTRE_RESOLUTION,5,245.98GHz,245.98,2,5.0,2000000.00kHz,2000000.0,21.2mJy/beam@10km/s,1.4mJy/beam@native,XX YY
3,Moon_full,3,CENTRE_RESOLUTION,5,247.98GHz,247.98,2,5.0,2000000.00kHz,2000000.0,22.5mJy/beam@10km/s,1.4mJy/beam@native,XX YY
4,Moon_reg,0,CENTRE_RESOLUTION,5,229.98GHz,229.98,2,5.0,2000000.00kHz,2000000.0,9.3mJy/beam@10km/s,577.4uJy/beam@native,XX YY
5,Moon_reg,1,CENTRE_RESOLUTION,5,231.98GHz,231.98,2,5.0,2000000.00kHz,2000000.0,9.4mJy/beam@10km/s,586.5uJy/beam@native,XX YY
6,Moon_reg,2,CENTRE_RESOLUTION,5,245.98GHz,245.98,2,5.0,2000000.00kHz,2000000.0,9.5mJy/beam@10km/s,608.2uJy/beam@native,XX YY
7,Moon_reg,3,CENTRE_RESOLUTION,5,247.98GHz,247.98,2,5.0,2000000.00kHz,2000000.0,10.1mJy/beam@10km/s,648.7uJy/beam@native,XX YY


,source_name,parsed_brace_components,token_counts,token_1_frequencies_ghz,representation_tolerances_mhz
0,Moon_full,4,"(5,)","(229.98, 231.98, 245.98, 247.98)","(5.0,)"
1,Moon_reg,4,"(5,)","(229.98, 231.98, 245.98, 247.98)","(5.0,)"


In [32]:
support_context_keys = [
    "member_ous_uid",
    "asdm_uid",
    "source_name",
    "frequency_support",
]


def rows_matching_context(
    dataframe: pd.DataFrame,
    keys: list[str],
    context_values: tuple[object, ...],
) -> pd.DataFrame:
    mask = pd.Series(True, index=dataframe.index)

    for key, value in zip(keys, context_values, strict=True):
        if pd.isna(value):
            mask &= dataframe[key].isna()
        else:
            mask &= dataframe[key].eq(value)

    return dataframe.loc[mask].copy()


brace_mapping_records = []

for context_values, row_group in target_member_df.groupby(
    support_context_keys,
    dropna=False,
    sort=False,
):
    if not isinstance(context_values, tuple):
        context_values = (context_values,)

    component_group = rows_matching_context(
        brace_component_df,
        support_context_keys,
        context_values,
    )

    for _, archive_row in row_group.iterrows():
        record = archive_row.to_dict()
        record.update(
            {
                "mapping_method": "NEAREST_CENTRE_MANY_TO_ONE",
                "mapping_status": "NO_COMPONENT",
                "nearest_candidate_count": 0,
                "component_index": np.nan,
                "token_1_raw": None,
                "token_1_ghz": np.nan,
                "token_1_representation_tolerance_mhz": np.nan,
                "token_2_raw": None,
                "token_2_khz": np.nan,
                "token_3_raw": None,
                "token_4_raw": None,
                "token_5_raw": None,
            }
        )

        usable_components = component_group.dropna(
            subset=["token_1_ghz"]
        ).copy()

        if usable_components.empty or pd.isna(archive_row["frequency"]):
            brace_mapping_records.append(record)
            continue

        usable_components["absolute_centre_difference_mhz"] = (
            usable_components["token_1_ghz"]
            .sub(float(archive_row["frequency"]))
            .abs()
            .mul(1e3)
        )

        minimum_difference_mhz = float(
            usable_components["absolute_centre_difference_mhz"].min()
        )
        nearest_mask = np.isclose(
            usable_components["absolute_centre_difference_mhz"],
            minimum_difference_mhz,
            rtol=0.0,
            atol=1e-9,
        )
        nearest_candidates = (
            usable_components.loc[nearest_mask]
            .sort_values("component_index", kind="stable")
        )
        chosen_component = nearest_candidates.iloc[0]

        representation_tolerance_mhz = float(
            chosen_component["token_1_representation_tolerance_mhz"]
        )
        within_representation_tolerance = (
            minimum_difference_mhz <= representation_tolerance_mhz
        )

        if len(nearest_candidates) > 1:
            mapping_status = "AMBIGUOUS_EQUAL_DISTANCE"
        elif within_representation_tolerance:
            mapping_status = "ASSIGNED_NEAREST_CENTRE"
        else:
            mapping_status = "OUTSIDE_REPRESENTATION_TOLERANCE"

        record.update(
            {
                "mapping_status": mapping_status,
                "nearest_candidate_count": len(nearest_candidates),
                "component_index": chosen_component["component_index"],
                "token_1_raw": chosen_component["token_1_raw"],
                "token_1_ghz": chosen_component["token_1_ghz"],
                "token_1_representation_tolerance_mhz": (
                    representation_tolerance_mhz
                ),
                "token_2_raw": chosen_component["token_2_raw"],
                "token_2_khz": chosen_component["token_2_khz"],
                "token_3_raw": chosen_component["token_3_raw"],
                "token_4_raw": chosen_component["token_4_raw"],
                "token_5_raw": chosen_component["token_5_raw"],
                "absolute_centre_difference_mhz": minimum_difference_mhz,
                "within_token_1_representation_tolerance": (
                    within_representation_tolerance
                ),
            }
        )
        brace_mapping_records.append(record)

brace_row_relationship_df = pd.DataFrame(brace_mapping_records)

brace_row_relationship_df["token1_minus_row_frequency_mhz"] = (
    brace_row_relationship_df["token_1_ghz"]
    .sub(brace_row_relationship_df["frequency"])
    .mul(1e3)
)

# Token 2 occupies the resolution position in the observed support grammar.
brace_row_relationship_df["token2_minus_spectral_resolution_khz"] = (
    brace_row_relationship_df["token_2_khz"]
    - brace_row_relationship_df["spectral_resolution"]
)

# The bandwidth comparison is retained as separate empirical evidence. Equality
# here does not redefine token 2 as bandwidth.
brace_row_relationship_df["token2_as_hz"] = (
    brace_row_relationship_df["token_2_khz"] * 1e3
)
brace_row_relationship_df["token2_minus_row_bandwidth_hz"] = (
    brace_row_relationship_df["token2_as_hz"]
    - brace_row_relationship_df["bandwidth"]
)

brace_row_relationship_df["estimated_resolution_elements"] = (
    brace_row_relationship_df["bandwidth"]
    / (
        brace_row_relationship_df["spectral_resolution"]
        * 1e3
    )
)

component_usage_keys = support_context_keys + ["component_index", "token_1_ghz"]
brace_component_usage_df = (
    brace_row_relationship_df.loc[
        brace_row_relationship_df["component_index"].notna()
    ]
    .groupby(component_usage_keys, dropna=False)
    .agg(
        mapped_archive_rows=("archive_row_index", "size"),
        mapped_spw_ids=(
            "spw_id",
            lambda values: tuple(sorted(values.dropna().astype(str))),
        ),
        maximum_centre_difference_mhz=(
            "absolute_centre_difference_mhz",
            "max",
        ),
        mapping_statuses=(
            "mapping_status",
            lambda values: tuple(sorted(set(values))),
        ),
    )
    .reset_index()
)

relationship_columns = [
    "source_name",
    "spw_id",
    "component_index",
    "mapping_method",
    "mapping_status",
    "nearest_candidate_count",
    "frequency",
    "token_1_ghz",
    "absolute_centre_difference_mhz",
    "token_1_representation_tolerance_mhz",
    "within_token_1_representation_tolerance",
    "bandwidth",
    "spectral_resolution",
    "token_2_khz",
    "token2_minus_spectral_resolution_khz",
    "token2_minus_row_bandwidth_hz",
    "estimated_resolution_elements",
    "token_3_raw",
    "token_4_raw",
    "token_5_raw",
    "antenna_prefixes",
    "array_evidence_class",
    "s_resolution",
    "spatial_resolution",
]

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.width", 260,
):
    display(brace_row_relationship_df[relationship_columns])
    display(brace_component_usage_df)


,source_name,spw_id,component_index,mapping_method,mapping_status,nearest_candidate_count,frequency,token_1_ghz,absolute_centre_difference_mhz,token_1_representation_tolerance_mhz,within_token_1_representation_tolerance,bandwidth,spectral_resolution,token_2_khz,token2_minus_spectral_resolution_khz,token2_minus_row_bandwidth_hz,estimated_resolution_elements,token_3_raw,token_4_raw,token_5_raw,antenna_prefixes,array_evidence_class,s_resolution,spatial_resolution
0,Moon_full,4,0,NEAREST_CENTRE_MANY_TO_ONE,ASSIGNED_NEAREST_CENTRE,1,229.981328,229.98,1.328176,5.0,True,2.000000e+09,2000000.0,2000000.0,0.0,0.0,1.0,20.8mJy/beam@10km/s,1.3mJy/beam@native,XX YY,"(PM,)",TP_LIKE,24.365779,21.993889
1,Moon_full,0,0,NEAREST_CENTRE_MANY_TO_ONE,ASSIGNED_NEAREST_CENTRE,1,229.981374,229.98,1.374189,5.0,True,2.000000e+09,2000000.0,2000000.0,0.0,0.0,1.0,20.8mJy/beam@10km/s,1.3mJy/beam@native,XX YY,"(PM,)",TP_LIKE,24.365779,21.993889
2,Moon_full,1,1,NEAREST_CENTRE_MANY_TO_ONE,ASSIGNED_NEAREST_CENTRE,1,231.981212,231.98,1.212225,5.0,True,2.000000e+09,2000000.0,2000000.0,0.0,0.0,1.0,21.1mJy/beam@10km/s,1.3mJy/beam@native,XX YY,"(PM,)",TP_LIKE,24.365779,21.993889
3,Moon_full,5,1,NEAREST_CENTRE_MANY_TO_ONE,ASSIGNED_NEAREST_CENTRE,1,231.981212,231.98,1.212225,5.0,True,2.000000e+09,2000000.0,2000000.0,0.0,0.0,1.0,21.1mJy/beam@10km/s,1.3mJy/beam@native,XX YY,"(PM,)",TP_LIKE,24.365779,21.993889
4,Moon_full,2,2,NEAREST_CENTRE_MANY_TO_ONE,ASSIGNED_NEAREST_CENTRE,1,245.980078,245.98,0.078480,5.0,True,2.000000e+09,2000000.0,2000000.0,0.0,0.0,1.0,21.2mJy/beam@10km/s,1.4mJy/beam@native,XX YY,"(PM,)",TP_LIKE,24.365779,21.993889
5,Moon_full,3,3,NEAREST_CENTRE_MANY_TO_ONE,ASSIGNED_NEAREST_CENTRE,1,247.979917,247.98,0.083484,5.0,True,2.000000e+09,2000000.0,2000000.0,0.0,0.0,1.0,22.5mJy/beam@10km/s,1.4mJy/beam@native,XX YY,"(PM,)",TP_LIKE,24.365779,21.993889
6,Moon_full,6,3,NEAREST_CENTRE_MANY_TO_ONE,ASSIGNED_NEAREST_CENTRE,1,247.979917,247.98,0.083484,5.0,True,2.000000e+09,2000000.0,2000000.0,0.0,0.0,1.0,22.5mJy/beam@10km/s,1.4mJy/beam@native,XX YY,"(PM,)",TP_LIKE,24.365779,21.993889
7,Moon_reg,0,0,NEAREST_CENTRE_MANY_TO_ONE,ASSIGNED_NEAREST_CENTRE,1,229.981382,229.98,1.382307,5.0,True,2.000000e+09,2000000.0,2000000.0,0.0,0.0,1.0,9.3mJy/beam@10km/s,577.4uJy/beam@native,XX YY,"(PM,)",TP_LIKE,24.365778,21.993888
8,Moon_reg,1,1,NEAREST_CENTRE_MANY_TO_ONE,ASSIGNED_NEAREST_CENTRE,1,231.981220,231.98,1.220414,5.0,True,2.000000e+09,2000000.0,2000000.0,0.0,0.0,1.0,9.4mJy/beam@10km/s,586.5uJy/beam@native,XX YY,"(PM,)",TP_LIKE,24.365778,21.993888
9,Moon_reg,2,2,NEAREST_CENTRE_MANY_TO_ONE,ASSIGNED_NEAREST_CENTRE,1,245.980087,245.98,0.087163,5.0,True,2.000000e+09,2000000.0,2000000.0,0.0,0.0,1.0,9.5mJy/beam@10km/s,608.2uJy/beam@native,XX YY,"(PM,)",TP_LIKE,24.365778,21.993888


,member_ous_uid,asdm_uid,source_name,frequency_support,component_index,token_1_ghz,mapped_archive_rows,mapped_spw_ids,maximum_centre_difference_mhz,mapping_statuses
0,uid://A001/X3955/X44,uid://A002/X13e7f29/Xc580,Moon_full,"{229.98GHz,2000000.00kHz,20.8mJy/beam@10km/s,1.3mJy/beam@native, XX YY} U {231.98GHz,2000000.00kHz,21.1mJy/beam@10km/s,1.3mJy/beam@native, XX YY} U {245.98GHz,2000000.00kHz,21.2mJy/beam@10km/s,1.4mJy/beam@native, XX YY} U {247.98GHz,2000000.00kHz,22.5mJy/beam@10km/s,1.4mJy/beam@native, XX YY}",0,229.98,2,"(0, 4)",1.374189,"(ASSIGNED_NEAREST_CENTRE,)"
1,uid://A001/X3955/X44,uid://A002/X13e7f29/Xc580,Moon_full,"{229.98GHz,2000000.00kHz,20.8mJy/beam@10km/s,1.3mJy/beam@native, XX YY} U {231.98GHz,2000000.00kHz,21.1mJy/beam@10km/s,1.3mJy/beam@native, XX YY} U {245.98GHz,2000000.00kHz,21.2mJy/beam@10km/s,1.4mJy/beam@native, XX YY} U {247.98GHz,2000000.00kHz,22.5mJy/beam@10km/s,1.4mJy/beam@native, XX YY}",1,231.98,2,"(1, 5)",1.212225,"(ASSIGNED_NEAREST_CENTRE,)"
2,uid://A001/X3955/X44,uid://A002/X13e7f29/Xc580,Moon_full,"{229.98GHz,2000000.00kHz,20.8mJy/beam@10km/s,1.3mJy/beam@native, XX YY} U {231.98GHz,2000000.00kHz,21.1mJy/beam@10km/s,1.3mJy/beam@native, XX YY} U {245.98GHz,2000000.00kHz,21.2mJy/beam@10km/s,1.4mJy/beam@native, XX YY} U {247.98GHz,2000000.00kHz,22.5mJy/beam@10km/s,1.4mJy/beam@native, XX YY}",2,245.98,1,"(2,)",0.078480,"(ASSIGNED_NEAREST_CENTRE,)"
3,uid://A001/X3955/X44,uid://A002/X13e7f29/Xc580,Moon_full,"{229.98GHz,2000000.00kHz,20.8mJy/beam@10km/s,1.3mJy/beam@native, XX YY} U {231.98GHz,2000000.00kHz,21.1mJy/beam@10km/s,1.3mJy/beam@native, XX YY} U {245.98GHz,2000000.00kHz,21.2mJy/beam@10km/s,1.4mJy/beam@native, XX YY} U {247.98GHz,2000000.00kHz,22.5mJy/beam@10km/s,1.4mJy/beam@native, XX YY}",3,247.98,2,"(3, 6)",0.083484,"(ASSIGNED_NEAREST_CENTRE,)"
4,uid://A001/X3955/X44,uid://A002/X13e7f29/Xc580,Moon_reg,"{229.98GHz,2000000.00kHz,9.3mJy/beam@10km/s,577.4uJy/beam@native, XX YY} U {231.98GHz,2000000.00kHz,9.4mJy/beam@10km/s,586.5uJy/beam@native, XX YY} U {245.98GHz,2000000.00kHz,9.5mJy/beam@10km/s,608.2uJy/beam@native, XX YY} U {247.98GHz,2000000.00kHz,10.1mJy/beam@10km/s,648.7uJy/beam@native, XX YY}",0,229.98,1,"(0,)",1.382307,"(ASSIGNED_NEAREST_CENTRE,)"
5,uid://A001/X3955/X44,uid://A002/X13e7f29/Xc580,Moon_reg,"{229.98GHz,2000000.00kHz,9.3mJy/beam@10km/s,577.4uJy/beam@native, XX YY} U {231.98GHz,2000000.00kHz,9.4mJy/beam@10km/s,586.5uJy/beam@native, XX YY} U {245.98GHz,2000000.00kHz,9.5mJy/beam@10km/s,608.2uJy/beam@native, XX YY} U {247.98GHz,2000000.00kHz,10.1mJy/beam@10km/s,648.7uJy/beam@native, XX YY}",1,231.98,1,"(1,)",1.220414,"(ASSIGNED_NEAREST_CENTRE,)"
6,uid://A001/X3955/X44,uid://A002/X13e7f29/Xc580,Moon_reg,"{229.98GHz,2000000.00kHz,9.3mJy/beam@10km/s,577.4uJy/beam@native, XX YY} U {231.98GHz,2000000.00kHz,9.4mJy/beam@10km/s,586.5uJy/beam@native, XX YY} U {245.98GHz,2000000.00kHz,9.5mJy/beam@10km/s,608.2uJy/beam@native, XX YY} U {247.98GHz,2000000.00kHz,10.1mJy/beam@10km/s,648.7uJy/beam@native, XX YY}",2,245.98,1,"(2,)",0.087163,"(ASSIGNED_NEAREST_CENTRE,)"
7,uid://A001/X3955/X44,uid://A002/X13e7f29/Xc580,Moon_reg,"{229.98GHz,2000000.00kHz,9.3mJy/beam@10km/s,577.4uJy/beam@native, XX YY} U {231.98GHz,2000000.00kHz,9.4mJy/beam@10km/s,586.5uJy/beam@native, XX YY} U {245.98GHz,2000000.00kHz,9.5mJy/beam@10km/s,608.2uJy/beam@native, XX YY} U {247.98GHz,2000000.00kHz,10.1mJy/beam@10km/s,648.7uJy/beam@native, XX YY}",3,247.98,1,"(3,)",0.074730,"(ASSIGNED_NEAREST_CENTRE,)"


In [33]:
token_1_eligible_mask = (
    brace_row_relationship_df["absolute_centre_difference_mhz"].notna()
    & brace_row_relationship_df[
        "token_1_representation_tolerance_mhz"
    ].notna()
)

token_2_resolution_eligible_mask = (
    brace_row_relationship_df["token_2_khz"].notna()
    & brace_row_relationship_df["spectral_resolution"].notna()
)

token_2_bandwidth_eligible_mask = (
    brace_row_relationship_df["token_2_khz"].notna()
    & brace_row_relationship_df["bandwidth"].notna()
)

brace_relationship_summary_df = pd.DataFrame(
    [
        {
            "relationship": (
                "token 1 nearest centre within displayed-precision tolerance"
            ),
            "eligible_rows": int(token_1_eligible_mask.sum()),
            "matching_rows": int(
                brace_row_relationship_df.loc[
                    token_1_eligible_mask,
                    "within_token_1_representation_tolerance",
                ].fillna(False).sum()
            ),
        },
        {
            "relationship": "token 2 equals Archive spectral_resolution",
            "eligible_rows": int(token_2_resolution_eligible_mask.sum()),
            "matching_rows": int(
                np.isclose(
                    brace_row_relationship_df.loc[
                        token_2_resolution_eligible_mask,
                        "token2_minus_spectral_resolution_khz",
                    ],
                    0.0,
                    rtol=0.0,
                    atol=1e-6,
                    equal_nan=False,
                ).sum()
            ),
        },
        {
            "relationship": "token 2 converted to Hz equals Archive bandwidth",
            "eligible_rows": int(token_2_bandwidth_eligible_mask.sum()),
            "matching_rows": int(
                np.isclose(
                    brace_row_relationship_df.loc[
                        token_2_bandwidth_eligible_mask,
                        "token2_minus_row_bandwidth_hz",
                    ],
                    0.0,
                    rtol=0.0,
                    atol=1.0,
                    equal_nan=False,
                ).sum()
            ),
        },
    ]
)

display(brace_relationship_summary_df)

many_to_one_component_count = int(
    brace_component_usage_df["mapped_archive_rows"].gt(1).sum()
)

print(
    "Brace mappings by status:",
    brace_row_relationship_df["mapping_status"]
    .value_counts(dropna=False)
    .to_dict(),
)
print(
    "Support components receiving multiple Archive rows:",
    many_to_one_component_count,
)


,relationship,eligible_rows,matching_rows
0,token 1 nearest centre within displayed-precision tolerance,11,11
1,token 2 equals Archive spectral_resolution,11,11
2,token 2 converted to Hz equals Archive bandwidth,11,11


Brace mappings by status: {'ASSIGNED_NEAREST_CENTRE': 11}
Support components receiving multiple Archive rows: 3


In [34]:
antenna_resolution_diagnostic_df = (
    target_member_df[
        [
            "source_name",
            "spw_id",
            "antenna_arrays",
            "antenna_prefixes",
            "array_evidence_class",
            "s_resolution",
            "spatial_resolution",
        ]
    ]
    .copy()
)

antenna_resolution_diagnostic_df[
    "resolution_difference_arcsec"
] = (
    antenna_resolution_diagnostic_df["s_resolution"]
    - antenna_resolution_diagnostic_df["spatial_resolution"]
)

antenna_resolution_diagnostic_df[
    "resolution_values_equal"
] = np.isclose(
    antenna_resolution_diagnostic_df["s_resolution"],
    antenna_resolution_diagnostic_df["spatial_resolution"],
    rtol=0.0,
    atol=1e-12,
    equal_nan=False,
)

with pd.option_context(
    "display.max_rows", None,
    "display.max_colwidth", None,
):
    display(antenna_resolution_diagnostic_df)

,source_name,spw_id,antenna_arrays,antenna_prefixes,array_evidence_class,s_resolution,spatial_resolution,resolution_difference_arcsec,resolution_values_equal
166,Moon_full,4,T701:PM04 T702:PM03 T703:PM01 T704:PM02,"(PM,)",TP_LIKE,24.365779,21.993889,2.37189,False
168,Moon_full,0,T701:PM04 T702:PM03 T703:PM01 T704:PM02,"(PM,)",TP_LIKE,24.365779,21.993889,2.37189,False
164,Moon_full,1,T701:PM04 T702:PM03 T703:PM01 T704:PM02,"(PM,)",TP_LIKE,24.365779,21.993889,2.37189,False
167,Moon_full,5,T701:PM04 T702:PM03 T703:PM01 T704:PM02,"(PM,)",TP_LIKE,24.365779,21.993889,2.37189,False
169,Moon_full,2,T701:PM04 T702:PM03 T703:PM01 T704:PM02,"(PM,)",TP_LIKE,24.365779,21.993889,2.37189,False
165,Moon_full,3,T701:PM04 T702:PM03 T703:PM01 T704:PM02,"(PM,)",TP_LIKE,24.365779,21.993889,2.37189,False
170,Moon_full,6,T701:PM04 T702:PM03 T703:PM01 T704:PM02,"(PM,)",TP_LIKE,24.365779,21.993889,2.37189,False
183,Moon_reg,0,T701:PM04 T702:PM03 T703:PM01 T704:PM02,"(PM,)",TP_LIKE,24.365778,21.993888,2.37189,False
184,Moon_reg,1,T701:PM04 T702:PM03 T703:PM01 T704:PM02,"(PM,)",TP_LIKE,24.365778,21.993888,2.37189,False
185,Moon_reg,2,T701:PM04 T702:PM03 T703:PM01 T704:PM02,"(PM,)",TP_LIKE,24.365778,21.993888,2.37189,False


## Targeted follow-up B: frequency outside the ObsCore wavelength interval

`em_min` and `em_max` are wavelength bounds in metres. They are converted
to frequency using `frequency = c / wavelength`.

This diagnostic preserves both representations and reports disagreements.
It does not overwrite one value with the other.

In [35]:
frequency_interval_diagnostic_df = analysis_df.copy()

required_interval_columns = {
    "em_min",
    "em_max",
    "frequency",
}

missing_interval_columns = (
    required_interval_columns
    - set(frequency_interval_diagnostic_df.columns)
)

if missing_interval_columns:
    raise RuntimeError(
        "Missing interval diagnostic columns: "
        f"{sorted(missing_interval_columns)}"
    )

eligible_interval_mask = (
    frequency_interval_diagnostic_df["em_min"].notna()
    & frequency_interval_diagnostic_df["em_max"].notna()
    & frequency_interval_diagnostic_df["frequency"].notna()
    & frequency_interval_diagnostic_df["em_min"].gt(0)
    & frequency_interval_diagnostic_df["em_max"].gt(0)
)

frequency_interval_diagnostic_df = (
    frequency_interval_diagnostic_df
    .loc[eligible_interval_mask]
    .copy()
)

speed_of_light_m_s = c.to_value(u.m / u.s)

# Larger wavelength corresponds to lower frequency.
frequency_interval_diagnostic_df[
    "obscore_frequency_low_ghz"
] = (
    speed_of_light_m_s
    / frequency_interval_diagnostic_df["em_max"]
    / 1e9
)

frequency_interval_diagnostic_df[
    "obscore_frequency_high_ghz"
] = (
    speed_of_light_m_s
    / frequency_interval_diagnostic_df["em_min"]
    / 1e9
)

frequency_interval_diagnostic_df[
    "row_frequency_inside_obscore_interval_exact"
] = (
    frequency_interval_diagnostic_df["frequency"].ge(
        frequency_interval_diagnostic_df[
            "obscore_frequency_low_ghz"
        ]
    )
    & frequency_interval_diagnostic_df["frequency"].le(
        frequency_interval_diagnostic_df[
            "obscore_frequency_high_ghz"
        ]
    )
)

# A 1-Hz tolerance is only for floating-point/unit-conversion robustness.
# It is not a scientific frequency-equivalence or duplication tolerance.
NUMERICAL_ATOL_GHZ = 1e-9

frequency_interval_diagnostic_df[
    "row_frequency_inside_obscore_interval_tolerant"
] = (
    frequency_interval_diagnostic_df["frequency"].ge(
        frequency_interval_diagnostic_df[
            "obscore_frequency_low_ghz"
        ] - NUMERICAL_ATOL_GHZ
    )
    & frequency_interval_diagnostic_df["frequency"].le(
        frequency_interval_diagnostic_df[
            "obscore_frequency_high_ghz"
        ] + NUMERICAL_ATOL_GHZ
    )
)

frequency_interval_diagnostic_df["exact_interval_position"] = np.select(
    [
        frequency_interval_diagnostic_df["frequency"].lt(
            frequency_interval_diagnostic_df[
                "obscore_frequency_low_ghz"
            ]
        ),
        frequency_interval_diagnostic_df["frequency"].gt(
            frequency_interval_diagnostic_df[
                "obscore_frequency_high_ghz"
            ]
        ),
    ],
    [
        "BELOW_LOW_BOUND",
        "ABOVE_HIGH_BOUND",
    ],
    default="INSIDE",
)

frequency_interval_diagnostic_df[
    "nearest_boundary_distance_mhz"
] = np.select(
    [
        frequency_interval_diagnostic_df[
            "exact_interval_position"
        ].eq("BELOW_LOW_BOUND"),
        frequency_interval_diagnostic_df[
            "exact_interval_position"
        ].eq("ABOVE_HIGH_BOUND"),
    ],
    [
        (
            frequency_interval_diagnostic_df[
                "obscore_frequency_low_ghz"
            ]
            - frequency_interval_diagnostic_df["frequency"]
        ) * 1e3,
        (
            frequency_interval_diagnostic_df["frequency"]
            - frequency_interval_diagnostic_df[
                "obscore_frequency_high_ghz"
            ]
        ) * 1e3,
    ],
    default=0.0,
)

frequency_interval_diagnostic_df[
    "boundary_distance_in_spectral_resolution_units"
] = (
    frequency_interval_diagnostic_df[
        "nearest_boundary_distance_mhz"
    ]
    * 1e3
    / frequency_interval_diagnostic_df["spectral_resolution"]
)

frequency_interval_diagnostic_df["interval_consistency_status"] = np.select(
    [
        frequency_interval_diagnostic_df[
            "row_frequency_inside_obscore_interval_exact"
        ],
        frequency_interval_diagnostic_df[
            "row_frequency_inside_obscore_interval_tolerant"
        ],
    ],
    [
        "CONSISTENT_EXACT",
        "CONSISTENT_WITH_NUMERICAL_TOLERANCE",
    ],
    default="OUTSIDE_NUMERICAL_TOLERANCE",
)

exact_outside_obscore_interval_df = (
    frequency_interval_diagnostic_df.loc[
        ~frequency_interval_diagnostic_df[
            "row_frequency_inside_obscore_interval_exact"
        ]
    ]
    .copy()
    .sort_values(
        ["member_ous_uid", "source_name", "frequency"],
        kind="stable",
    )
)

outside_obscore_interval_df = (
    frequency_interval_diagnostic_df.loc[
        ~frequency_interval_diagnostic_df[
            "row_frequency_inside_obscore_interval_tolerant"
        ]
    ]
    .copy()
    .sort_values(
        ["member_ous_uid", "source_name", "frequency"],
        kind="stable",
    )
)

print(
    "Rows outside by exact floating-point comparison:",
    len(exact_outside_obscore_interval_df),
)
print(
    "Rows outside after 1-Hz numerical tolerance:",
    len(outside_obscore_interval_df),
)

if len(exact_outside_obscore_interval_df) != 5:
    warnings.warn(
        "The previous 04b run observed 5 exact-boundary mismatches, "
        f"but this run found {len(exact_outside_obscore_interval_df)}. "
        "Check whether the live Archive changed."
    )

interval_display_columns = [
    "archive_row_index",
    "proposal_id",
    "member_ous_uid",
    "asdm_uid",
    "source_name",
    "spw_id",
    "obs_id",
    "frequency",
    "em_min",
    "em_max",
    "obscore_frequency_low_ghz",
    "obscore_frequency_high_ghz",
    "exact_interval_position",
    "nearest_boundary_distance_mhz",
    "spectral_resolution",
    "boundary_distance_in_spectral_resolution_units",
    "interval_consistency_status",
    "bandwidth",
    "frequency_support",
    "band_list",
    "type",
    "dataproduct_type",
]

interval_display_columns = [
    column
    for column in interval_display_columns
    if column in exact_outside_obscore_interval_df.columns
]

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.width", 280,
):
    display(
        exact_outside_obscore_interval_df[
            interval_display_columns
        ]
    )


Rows outside by exact floating-point comparison: 5
Rows outside after 1-Hz numerical tolerance: 0


,archive_row_index,proposal_id,member_ous_uid,asdm_uid,source_name,spw_id,obs_id,frequency,em_min,em_max,obscore_frequency_low_ghz,obscore_frequency_high_ghz,exact_interval_position,nearest_boundary_distance_mhz,spectral_resolution,boundary_distance_in_spectral_resolution_units,interval_consistency_status,bandwidth,frequency_support,band_list,type,dataproduct_type
166,166,2025.A.00035.S,uid://A001/X3955/X44,uid://A002/X13e7f29/Xc580,Moon_full,4,uid://A001/X3955/X44.source.Moon_full.spw.4,229.981328,0.001304,0.001304,229.981328,229.981328,BELOW_LOW_BOUND,2.842171e-11,2000000.0,1.421085e-14,CONSISTENT_WITH_NUMERICAL_TOLERANCE,2.000000e+09,"{229.98GHz,2000000.00kHz,20.8mJy/beam@10km/s,1.3mJy/beam@native, XX YY} U {231.98GHz,2000000.00kHz,21.1mJy/beam@10km/s,1.3mJy/beam@native, XX YY} U {245.98GHz,2000000.00kHz,21.2mJy/beam@10km/s,1.4mJy/beam@native, XX YY} U {247.98GHz,2000000.00kHz,22.5mJy/beam@10km/s,1.4mJy/beam@native, XX YY}",6,S,image
168,168,2025.A.00035.S,uid://A001/X3955/X44,uid://A002/X13e7f29/Xc580,Moon_full,0,uid://A001/X3955/X44.source.Moon_full.spw.0,229.981374,0.001304,0.001304,229.981374,229.981374,BELOW_LOW_BOUND,2.842171e-11,2000000.0,1.421085e-14,CONSISTENT_WITH_NUMERICAL_TOLERANCE,2.000000e+09,"{229.98GHz,2000000.00kHz,20.8mJy/beam@10km/s,1.3mJy/beam@native, XX YY} U {231.98GHz,2000000.00kHz,21.1mJy/beam@10km/s,1.3mJy/beam@native, XX YY} U {245.98GHz,2000000.00kHz,21.2mJy/beam@10km/s,1.4mJy/beam@native, XX YY} U {247.98GHz,2000000.00kHz,22.5mJy/beam@10km/s,1.4mJy/beam@native, XX YY}",6,S,image
165,165,2025.A.00035.S,uid://A001/X3955/X44,uid://A002/X13e7f29/Xc580,Moon_full,3,uid://A001/X3955/X44.source.Moon_full.spw.3,247.979917,0.001209,0.001209,247.979917,247.979917,BELOW_LOW_BOUND,2.842171e-11,2000000.0,1.421085e-14,CONSISTENT_WITH_NUMERICAL_TOLERANCE,2.000000e+09,"{229.98GHz,2000000.00kHz,20.8mJy/beam@10km/s,1.3mJy/beam@native, XX YY} U {231.98GHz,2000000.00kHz,21.1mJy/beam@10km/s,1.3mJy/beam@native, XX YY} U {245.98GHz,2000000.00kHz,21.2mJy/beam@10km/s,1.4mJy/beam@native, XX YY} U {247.98GHz,2000000.00kHz,22.5mJy/beam@10km/s,1.4mJy/beam@native, XX YY}",6,S,image
170,170,2025.A.00035.S,uid://A001/X3955/X44,uid://A002/X13e7f29/Xc580,Moon_full,6,uid://A001/X3955/X44.source.Moon_full.spw.6,247.979917,0.001209,0.001209,247.979917,247.979917,BELOW_LOW_BOUND,2.842171e-11,2000000.0,1.421085e-14,CONSISTENT_WITH_NUMERICAL_TOLERANCE,2.000000e+09,"{229.98GHz,2000000.00kHz,20.8mJy/beam@10km/s,1.3mJy/beam@native, XX YY} U {231.98GHz,2000000.00kHz,21.1mJy/beam@10km/s,1.3mJy/beam@native, XX YY} U {245.98GHz,2000000.00kHz,21.2mJy/beam@10km/s,1.4mJy/beam@native, XX YY} U {247.98GHz,2000000.00kHz,22.5mJy/beam@10km/s,1.4mJy/beam@native, XX YY}",6,S,image
183,183,2025.A.00035.S,uid://A001/X3955/X44,uid://A002/X13e7f29/Xc580,Moon_reg,0,uid://A001/X3955/X44.source.Moon_reg.spw.0,229.981382,0.001304,0.001304,229.981382,229.981382,BELOW_LOW_BOUND,2.842171e-11,2000000.0,1.421085e-14,CONSISTENT_WITH_NUMERICAL_TOLERANCE,2.000000e+09,"{229.98GHz,2000000.00kHz,9.3mJy/beam@10km/s,577.4uJy/beam@native, XX YY} U {231.98GHz,2000000.00kHz,9.4mJy/beam@10km/s,586.5uJy/beam@native, XX YY} U {245.98GHz,2000000.00kHz,9.5mJy/beam@10km/s,608.2uJy/beam@native, XX YY} U {247.98GHz,2000000.00kHz,10.1mJy/beam@10km/s,648.7uJy/beam@native, XX YY}",6,S,image


In [36]:
interval_consistency_summary_df = (
    frequency_interval_diagnostic_df[
        "interval_consistency_status"
    ]
    .value_counts(dropna=False)
    .rename_axis("interval_consistency_status")
    .reset_index(name="row_count")
)

display(interval_consistency_summary_df)

if len(exact_outside_obscore_interval_df):
    exact_outside_interval_summary_df = (
        exact_outside_obscore_interval_df
        .groupby(
            ["member_ous_uid", "asdm_uid", "source_name"],
            dropna=False,
        )
        .agg(
            exact_boundary_rows=("archive_row_index", "size"),
            affected_spws=(
                "spw_id",
                lambda values: tuple(
                    sorted(values.dropna().astype(str).unique())
                ),
            ),
            exact_positions=(
                "exact_interval_position",
                lambda values: tuple(sorted(set(values))),
            ),
            maximum_boundary_distance_mhz=(
                "nearest_boundary_distance_mhz",
                "max",
            ),
            maximum_distance_in_resolution_units=(
                "boundary_distance_in_spectral_resolution_units",
                "max",
            ),
            consistency_statuses=(
                "interval_consistency_status",
                lambda values: tuple(sorted(set(values))),
            ),
        )
        .reset_index()
    )
else:
    exact_outside_interval_summary_df = pd.DataFrame()

display(exact_outside_interval_summary_df)

print(
    "Scientifically meaningful interval conflicts after numerical tolerance:",
    len(outside_obscore_interval_df),
)


,interval_consistency_status,row_count
0,CONSISTENT_EXACT,182
1,CONSISTENT_WITH_NUMERICAL_TOLERANCE,5


,member_ous_uid,asdm_uid,source_name,exact_boundary_rows,affected_spws,exact_positions,maximum_boundary_distance_mhz,maximum_distance_in_resolution_units,consistency_statuses
0,uid://A001/X3955/X44,uid://A002/X13e7f29/Xc580,Moon_full,4,"(0, 3, 4, 6)","(BELOW_LOW_BOUND,)",2.842171e-11,1.421085e-14,"(CONSISTENT_WITH_NUMERICAL_TOLERANCE,)"
1,uid://A001/X3955/X44,uid://A002/X13e7f29/Xc580,Moon_reg,1,"(0,)","(BELOW_LOW_BOUND,)",2.842171e-11,1.421085e-14,"(CONSISTENT_WITH_NUMERICAL_TOLERANCE,)"


Scientifically meaningful interval conflicts after numerical tolerance: 0


## Step 5 — Archive-wide `frequency_support` grammar census

This experiment uses complete `COUNT(*)` queries for grammar prevalence and
small, explicitly limited discovery queries for examples. It distinguishes
Archive-wide counts from intentionally limited examples. Unknown formats are
preserved rather than treated as missing or discarded.

In [37]:
SUPPORT_GRAMMAR_CONDITIONS = {
    "BRACKET_INTERVAL": "frequency_support LIKE '[%'",
    "BRACE_CENTRE_RESOLUTION": "frequency_support LIKE '{%'",
    "MISSING": "frequency_support IS NULL",
    "BLANK_EXACT": "frequency_support = ''",
    "UNKNOWN_NONBLANK": """
        frequency_support IS NOT NULL
        AND frequency_support <> ''
        AND frequency_support NOT LIKE '[%'
        AND frequency_support NOT LIKE '{%'
    """,
}

GRAMMAR_EXAMPLE_LIMIT = 40


def classify_support_grammar(raw_value: object) -> str:
    if pd.isna(raw_value):
        return "MISSING"

    text = str(raw_value).strip()
    if not text:
        return "BLANK"
    if text.startswith("["):
        return "BRACKET_INTERVAL"
    if text.startswith("{"):
        return "BRACE_CENTRE_RESOLUTION"
    return "UNKNOWN_NONBLANK"


grammar_census_records = []

grammar_total_query = """
SELECT COUNT(*) AS total_rows
FROM ivoa.obscore
WHERE science_observation = 'T'
"""
grammar_total_run = run_tap_query(
    grammar_total_query,
    maxrec=10,
    label="Count all science-target rows for grammar partition",
)
query_runs.append(grammar_total_run)
grammar_total_science_rows = int(grammar_total_run.table[0]["total_rows"])

for grammar_family, condition in SUPPORT_GRAMMAR_CONDITIONS.items():
    grammar_count_query = f"""
    SELECT COUNT(*) AS total_rows
    FROM ivoa.obscore
    WHERE science_observation = 'T'
      AND ({condition})
    """

    run = run_tap_query(
        grammar_count_query,
        maxrec=10,
        label=f"Count support grammar {grammar_family}",
    )
    query_runs.append(run)

    grammar_census_records.append(
        {
            "grammar_family": grammar_family,
            "archive_row_count": int(run.table[0]["total_rows"]),
            "count_query_status": run.query_status,
            "count_is_complete": "OVERFLOW" not in " ".join(run.query_status),
        }
    )

support_grammar_census_df = pd.DataFrame(grammar_census_records)
grammar_partition_rows = int(
    support_grammar_census_df["archive_row_count"].sum()
)
grammar_partition_complete = (
    grammar_partition_rows == grammar_total_science_rows
)

grammar_coverage_accounting_df = pd.DataFrame(
    [
        {
            "total_science_target_rows": grammar_total_science_rows,
            "rows_accounted_for_by_disjoint_grammar_categories": (
                grammar_partition_rows
            ),
            "unaccounted_rows": (
                grammar_total_science_rows - grammar_partition_rows
            ),
            "grammar_partition_complete": grammar_partition_complete,
        }
    ]
)

display(support_grammar_census_df)
display(grammar_coverage_accounting_df)

if not grammar_partition_complete:
    warnings.warn(
        "Grammar categories do not partition all science-target rows. "
        "Inspect COUNT-query conditions before interpreting prevalence."
    )

grammar_example_columns = [
    name
    for name in [
        "proposal_id", "member_ous_uid", "obs_id", "asdm_uid",
        "frequency_support", "frequency", "bandwidth",
        "spectral_resolution", "antenna_arrays", "dataproduct_type",
        "calib_level", "band_list", "type", "is_mosaic",
    ]
    if name in available_columns
]
grammar_example_column_sql = ",\n    ".join(grammar_example_columns)

grammar_example_frames = []

for census_row in support_grammar_census_df.itertuples(index=False):
    if census_row.archive_row_count == 0:
        continue

    condition = SUPPORT_GRAMMAR_CONDITIONS[census_row.grammar_family]
    grammar_example_query = f"""
    SELECT TOP {GRAMMAR_EXAMPLE_LIMIT}
        {grammar_example_column_sql}
    FROM ivoa.obscore
    WHERE science_observation = 'T'
      AND ({condition})
    ORDER BY proposal_id DESC, member_ous_uid, obs_id
    """

    run = run_tap_query(
        grammar_example_query,
        maxrec=GRAMMAR_EXAMPLE_LIMIT,
        label=f"Discover examples for {census_row.grammar_family}",
    )
    query_runs.append(run)

    frame = run.table.to_pandas()
    frame["discovery_grammar_condition"] = census_row.grammar_family
    frame["discovery_completeness"] = "INTENTIONALLY_LIMITED_DISCOVERY"
    grammar_example_frames.append(frame)

if grammar_example_frames:
    support_grammar_examples_df = pd.concat(
        grammar_example_frames,
        ignore_index=True,
    )
    support_grammar_examples_df["observed_grammar_family"] = (
        support_grammar_examples_df["frequency_support"]
        .map(classify_support_grammar)
    )
else:
    support_grammar_examples_df = pd.DataFrame(
        columns=grammar_example_columns
        + [
            "discovery_grammar_condition",
            "discovery_completeness",
            "observed_grammar_family",
        ]
    )

display(
    support_grammar_examples_df
    .groupby(
        [
            "discovery_grammar_condition",
            "observed_grammar_family",
            "dataproduct_type",
        ],
        dropna=False,
    )
    .agg(
        example_rows=("member_ous_uid", "size"),
        example_members=("member_ous_uid", "nunique"),
        example_bands=(
            "band_list",
            lambda values: tuple(sorted(set(values.dropna().astype(str)))),
        ),
    )
    .reset_index()
)

unknown_support_examples_df = support_grammar_examples_df.loc[
    support_grammar_examples_df["observed_grammar_family"].eq(
        "UNKNOWN_NONBLANK"
    )
].copy()

with pd.option_context("display.max_colwidth", None):
    display(unknown_support_examples_df.head(GRAMMAR_EXAMPLE_LIMIT))


--- Count all science-target rows for grammar partition ---
Endpoint: https://almascience.eso.org/tap
MAXREC: 10
ADQL:

SELECT COUNT(*) AS total_rows
FROM ivoa.obscore
WHERE science_observation = 'T'

Retrieved rows: 1
QUERY_STATUS: ('OK',)
Warnings: <none>

--- Count support grammar BRACKET_INTERVAL ---
Endpoint: https://almascience.eso.org/tap
MAXREC: 10
ADQL:

    SELECT COUNT(*) AS total_rows
    FROM ivoa.obscore
    WHERE science_observation = 'T'
      AND (frequency_support LIKE '[%')
    
Retrieved rows: 1
QUERY_STATUS: ('OK',)
Warnings: <none>

--- Count support grammar BRACE_CENTRE_RESOLUTION ---
Endpoint: https://almascience.eso.org/tap
MAXREC: 10
ADQL:

    SELECT COUNT(*) AS total_rows
    FROM ivoa.obscore
    WHERE science_observation = 'T'
      AND (frequency_support LIKE '{%')
    
Retrieved rows: 1
QUERY_STATUS: ('OK',)
Warnings: <none>

--- Count support grammar MISSING ---
Endpoint: https://almascience.eso.org/tap
MAXREC: 10
ADQL:

    SELECT COUNT(*) AS total_ro

,grammar_family,archive_row_count,count_query_status,count_is_complete
0,BRACKET_INTERVAL,442452,"(OK,)",True
1,BRACE_CENTRE_RESOLUTION,55,"(OK,)",True
2,MISSING,0,"(OK,)",True
3,BLANK_EXACT,0,"(OK,)",True
4,UNKNOWN_NONBLANK,0,"(OK,)",True


,total_science_target_rows,rows_accounted_for_by_disjoint_grammar_categories,unaccounted_rows,grammar_partition_complete
0,442507,442507,0,True



--- Discover examples for BRACKET_INTERVAL ---
Endpoint: https://almascience.eso.org/tap
MAXREC: 40
ADQL:

    SELECT TOP 40
        proposal_id,
    member_ous_uid,
    obs_id,
    asdm_uid,
    frequency_support,
    frequency,
    bandwidth,
    spectral_resolution,
    antenna_arrays,
    dataproduct_type,
    calib_level,
    band_list,
    type,
    is_mosaic
    FROM ivoa.obscore
    WHERE science_observation = 'T'
      AND (frequency_support LIKE '[%')
    ORDER BY proposal_id DESC, member_ous_uid, obs_id
    
Retrieved rows: 40
QUERY_STATUS: ('OK',)
Warnings: <none>

--- Discover examples for BRACE_CENTRE_RESOLUTION ---
Endpoint: https://almascience.eso.org/tap
MAXREC: 40
ADQL:

    SELECT TOP 40
        proposal_id,
    member_ous_uid,
    obs_id,
    asdm_uid,
    frequency_support,
    frequency,
    bandwidth,
    spectral_resolution,
    antenna_arrays,
    dataproduct_type,
    calib_level,
    band_list,
    type,
    is_mosaic
    FROM ivoa.obscore
    WHERE science_

,discovery_grammar_condition,observed_grammar_family,dataproduct_type,example_rows,example_members,example_bands
0,BRACE_CENTRE_RESOLUTION,BRACE_CENTRE_RESOLUTION,image,40,5,"(3, 6)"
1,BRACKET_INTERVAL,BRACKET_INTERVAL,cube,16,4,"(1,)"
2,BRACKET_INTERVAL,BRACKET_INTERVAL,image,24,6,"(1, 3, 5, 6)"


,proposal_id,member_ous_uid,obs_id,asdm_uid,frequency_support,frequency,bandwidth,spectral_resolution,antenna_arrays,dataproduct_type,calib_level,band_list,type,is_mosaic,discovery_grammar_condition,discovery_completeness,observed_grammar_family


## Step 6 — Source–SPW and support-component cardinality census

The census combines the original robustness Members with Members discovered
from each support-grammar family. Every selected Member is then counted and
retrieved completely. The experiment measures sparse Source–SPW associations
and component-count relationships without assuming a Cartesian grid or a
universal one-to-one mapping.

In [38]:
CARDINALITY_MEMBER_LIMIT = 80
GRAMMAR_MEMBERS_PER_FAMILY = 12

grammar_member_manifest_df = (
    support_grammar_examples_df
    .dropna(subset=["member_ous_uid"])
    .sort_values(
        ["observed_grammar_family", "member_ous_uid"],
        kind="stable",
    )
    .drop_duplicates(
        ["observed_grammar_family", "member_ous_uid"]
    )
    .groupby("observed_grammar_family", dropna=False, sort=False)
    .head(GRAMMAR_MEMBERS_PER_FAMILY)
    [["member_ous_uid", "observed_grammar_family"]]
    .drop_duplicates("member_ous_uid")
)

base_cardinality_manifest_df = sample_manifest_df[
    ["member_ous_uid"]
].copy()
base_cardinality_manifest_df["observed_grammar_family"] = (
    "ORIGINAL_ROBUSTNESS_SAMPLE"
)

cardinality_member_manifest_df = (
    pd.concat(
        [base_cardinality_manifest_df, grammar_member_manifest_df],
        ignore_index=True,
    )
    .dropna(subset=["member_ous_uid"])
    .drop_duplicates("member_ous_uid", keep="first")
    .head(CARDINALITY_MEMBER_LIMIT)
    .copy()
)

cardinality_member_uids = (
    cardinality_member_manifest_df["member_ous_uid"]
    .astype(str)
    .tolist()
)
cardinality_member_sql = ",\n    ".join(
    quote_adql_string(uid)
    for uid in cardinality_member_uids
)

cardinality_count_query = f"""
SELECT COUNT(*) AS total_rows
FROM ivoa.obscore
WHERE science_observation = 'T'
  AND member_ous_uid IN (
    {cardinality_member_sql}
  )
"""

cardinality_count_run = run_tap_query(
    cardinality_count_query,
    maxrec=10,
    label="Count complete cardinality-census sample",
)
query_runs.append(cardinality_count_run)
cardinality_expected_rows = int(
    cardinality_count_run.table[0]["total_rows"]
)

if cardinality_expected_rows > EXPLORATORY_SAFETY_LIMIT:
    raise RuntimeError(
        f"Cardinality census expects {cardinality_expected_rows} rows, "
        f"above safety limit {EXPLORATORY_SAFETY_LIMIT}. "
        "Reduce CARDINALITY_MEMBER_LIMIT."
    )

cardinality_column_sql = ",\n    ".join(selected_archive_columns)
cardinality_query = f"""
SELECT
    {cardinality_column_sql}
FROM ivoa.obscore
WHERE science_observation = 'T'
  AND member_ous_uid IN (
    {cardinality_member_sql}
  )
"""

cardinality_run = run_tap_query(
    cardinality_query,
    maxrec=max(cardinality_expected_rows, 1),
    label="Retrieve complete cardinality-census sample",
)
query_runs.append(cardinality_run)

cardinality_df = cardinality_run.table.to_pandas()
cardinality_df.insert(
    0,
    "cardinality_row_index",
    np.arange(len(cardinality_df), dtype=int),
)

if len(cardinality_df) != cardinality_expected_rows:
    raise RuntimeError(
        "Cardinality census retrieval is incomplete: "
        f"{len(cardinality_df)} / {cardinality_expected_rows} rows."
    )

cardinality_parsed_obs_df = pd.DataFrame(
    cardinality_df["obs_id"].map(parse_obs_id_structure).tolist()
)
cardinality_analysis_df = pd.concat(
    [
        cardinality_df.reset_index(drop=True),
        cardinality_parsed_obs_df,
    ],
    axis=1,
)
cardinality_analysis_df["support_grammar_family"] = (
    cardinality_analysis_df["frequency_support"]
    .map(classify_support_grammar)
)


def support_component_count(raw_support: object) -> dict[str, object]:
    grammar_family = classify_support_grammar(raw_support)

    if grammar_family == "BRACKET_INTERVAL":
        result = parse_frequency_support(raw_support)
        return {
            "support_component_count": len(result.components),
            "support_count_status": result.parse_status.value,
        }

    if grammar_family == "BRACE_CENTRE_RESOLUTION":
        component_count = len(
            BRACE_COMPONENT_PATTERN.findall(str(raw_support))
        )
        return {
            "support_component_count": component_count,
            "support_count_status": (
                "COUNTED_BRACE_COMPONENTS"
                if component_count
                else "BRACE_WITHOUT_COMPONENTS"
            ),
        }

    return {
        "support_component_count": 0,
        "support_count_status": grammar_family,
    }


cardinality_support_metadata_df = pd.DataFrame(
    cardinality_analysis_df["frequency_support"]
    .map(support_component_count)
    .tolist()
)
for metadata_column in cardinality_support_metadata_df.columns:
    cardinality_analysis_df[metadata_column] = (
        cardinality_support_metadata_df[metadata_column].to_numpy()
    )

assert cardinality_analysis_df.columns.is_unique, (
    "Cardinality reconstruction produced duplicate columns: "
    f"{cardinality_analysis_df.columns[cardinality_analysis_df.columns.duplicated()].tolist()}"
)

cardinality_context_keys = [
    "member_ous_uid",
    "asdm_uid",
    "source_name",
    "frequency_support",
]

context_cardinality_df = (
    cardinality_analysis_df
    .groupby(cardinality_context_keys, dropna=False)
    .agg(
        archive_row_count=("cardinality_row_index", "size"),
        parsed_spw_count=("spw_id", "nunique"),
        unique_row_frequency_count=("frequency", "nunique"),
        support_component_count=("support_component_count", "first"),
        support_grammar_family=("support_grammar_family", "first"),
        support_count_status=("support_count_status", "first"),
    )
    .reset_index()
)

context_cardinality_df["cardinality_class"] = np.select(
    [
        context_cardinality_df["support_component_count"].eq(0),
        context_cardinality_df["archive_row_count"].eq(
            context_cardinality_df["support_component_count"]
        ),
        context_cardinality_df["archive_row_count"].gt(
            context_cardinality_df["support_component_count"]
        ),
        context_cardinality_df["archive_row_count"].lt(
            context_cardinality_df["support_component_count"]
        ),
    ],
    [
        "UNPARSED_OR_NO_COMPONENT",
        "COUNT_EQUAL_ONE_TO_ONE_CANDIDATE",
        "MORE_ROWS_THAN_COMPONENTS_MANY_TO_ONE_CANDIDATE",
        "MORE_COMPONENTS_THAN_ROWS",
    ],
    default="UNKNOWN",
)

cardinality_analysis_df["source_execution_context_key"] = list(
    zip(
        cardinality_analysis_df["member_ous_uid"],
        cardinality_analysis_df["asdm_uid"],
        cardinality_analysis_df["source_name"],
    )
)

execution_cardinality_df = (
    cardinality_analysis_df
    .groupby(["member_ous_uid", "asdm_uid"], dropna=False)
    .agg(
        archive_rows=("cardinality_row_index", "size"),
        source_contexts=("source_name", "nunique"),
        execution_spw_count=("spw_id", "nunique"),
        support_grammars=(
            "support_grammar_family",
            lambda values: tuple(sorted(set(values))),
        ),
    )
    .reset_index()
)
execution_cardinality_df["cartesian_expected_rows"] = (
    execution_cardinality_df["source_contexts"]
    * execution_cardinality_df["execution_spw_count"]
)
execution_cardinality_df["source_spw_structure"] = np.select(
    [
        execution_cardinality_df["archive_rows"].eq(
            execution_cardinality_df["cartesian_expected_rows"]
        ),
        execution_cardinality_df["archive_rows"].lt(
            execution_cardinality_df["cartesian_expected_rows"]
        ),
    ],
    ["COMPLETE_CARTESIAN_GRID", "SPARSE_SOURCE_SPW_ASSOCIATION"],
    default="ROWS_EXCEED_SIMPLE_CARTESIAN_EXPECTATION",
)

member_cardinality_df = (
    cardinality_analysis_df
    .groupby("member_ous_uid", dropna=False)
    .agg(
        archive_rows=("cardinality_row_index", "size"),
        asdm_associations=("asdm_uid", "nunique"),
        source_execution_contexts=(
            "source_execution_context_key",
            "nunique",
        ),
        support_grammars=(
            "support_grammar_family",
            lambda values: tuple(sorted(set(values))),
        ),
    )
    .reset_index()
)

execution_spw_sets = {
    (str(member_uid), str(asdm_uid)): set(
        group["spw_id"].dropna().astype(str)
    )
    for (member_uid, asdm_uid), group in cardinality_analysis_df.groupby(
        ["member_ous_uid", "asdm_uid"],
        dropna=False,
    )
}

source_spw_association_df = (
    cardinality_analysis_df
    .groupby(
        ["member_ous_uid", "asdm_uid", "source_name"],
        dropna=False,
    )
    .agg(
        archive_rows=("cardinality_row_index", "size"),
        observed_spw_ids=(
            "spw_id",
            lambda values: tuple(sorted(set(values.dropna().astype(str)))),
        ),
    )
    .reset_index()
)
source_spw_association_df["execution_spw_ids"] = (
    source_spw_association_df.apply(
        lambda row: tuple(
            sorted(
                execution_spw_sets.get(
                    (str(row["member_ous_uid"]), str(row["asdm_uid"])),
                    set(),
                )
            )
        ),
        axis=1,
    )
)
source_spw_association_df["missing_execution_spw_ids"] = (
    source_spw_association_df.apply(
        lambda row: tuple(
            sorted(
                set(row["execution_spw_ids"])
                - set(row["observed_spw_ids"])
            )
        ),
        axis=1,
    )
)
source_spw_association_df["is_sparse_source_association"] = (
    source_spw_association_df["missing_execution_spw_ids"].map(bool)
)

display(
    execution_cardinality_df["source_spw_structure"]
    .value_counts(dropna=False)
    .rename_axis("source_spw_structure")
    .reset_index(name="execution_count")
)
display(
    context_cardinality_df
    .groupby(
        ["support_grammar_family", "cardinality_class"],
        dropna=False,
    )
    .size()
    .rename("context_count")
    .reset_index()
)
display(
    source_spw_association_df.loc[
        source_spw_association_df["is_sparse_source_association"]
    ]
)
display(
    context_cardinality_df.loc[
        ~context_cardinality_df["cardinality_class"].eq(
            "COUNT_EQUAL_ONE_TO_ONE_CANDIDATE"
        )
    ]
)



--- Count complete cardinality-census sample ---
Endpoint: https://almascience.eso.org/tap
MAXREC: 10
ADQL:

SELECT COUNT(*) AS total_rows
FROM ivoa.obscore
WHERE science_observation = 'T'
  AND member_ous_uid IN (
    'uid://A001/X1288/X4c3',
    'uid://A001/X1288/X4c7',
    'uid://A001/X1288/X4d7',
    'uid://A001/X1288/X4dd',
    'uid://A001/X3621/X3ca0',
    'uid://A001/X3621/X3ca4',
    'uid://A001/X362b/Xbe5',
    'uid://A001/X3645/X15f',
    'uid://A001/X3833/X1022',
    'uid://A001/X3833/X14b4',
    'uid://A001/X3833/X1982',
    'uid://A001/X3833/X198e',
    'uid://A001/X3833/X1998',
    'uid://A001/X3845/X5cf',
    'uid://A001/X3845/X776',
    'uid://A001/X3845/X77a',
    'uid://A001/X3873/X44c',
    'uid://A001/X38cd/X1d',
    'uid://A001/X38cd/X1ea',
    'uid://A001/X38cd/X2d',
    'uid://A001/X38cd/X3c3',
    'uid://A001/X38cd/X3c7',
    'uid://A001/X38cd/Xd4',
    'uid://A001/X38cd/Xd7',
    'uid://A001/X3922/X599',
    'uid://A001/X3922/X59c',
    'uid://A001/X3922/X59f'

,source_spw_structure,execution_count
0,COMPLETE_CARTESIAN_GRID,39
1,SPARSE_SOURCE_SPW_ASSOCIATION,1


,support_grammar_family,cardinality_class,context_count
0,BRACE_CENTRE_RESOLUTION,COUNT_EQUAL_ONE_TO_ONE_CANDIDATE,9
1,BRACE_CENTRE_RESOLUTION,MORE_ROWS_THAN_COMPONENTS_MANY_TO_ONE_CANDIDATE,1
2,BRACKET_INTERVAL,COUNT_EQUAL_ONE_TO_ONE_CANDIDATE,39


,member_ous_uid,asdm_uid,source_name,archive_rows,observed_spw_ids,execution_spw_ids,missing_execution_spw_ids,is_sparse_source_association
40,uid://A001/X3955/X44,uid://A002/X13e7f29/Xc580,Moon_reg,4,"(0, 1, 2, 3)","(0, 1, 2, 3, 4, 5, 6)","(4, 5, 6)",True


,member_ous_uid,asdm_uid,source_name,frequency_support,archive_row_count,parsed_spw_count,unique_row_frequency_count,support_component_count,support_grammar_family,support_count_status,cardinality_class
39,uid://A001/X3955/X44,uid://A002/X13e7f29/Xc580,Moon_full,"{229.98GHz,2000000.00kHz,20.8mJy/beam@10km/s,1.3mJy/beam@native, XX YY} U {231.98GHz,2000000.00kHz,21.1mJy/beam@10km...",7,7,5,4,BRACE_CENTRE_RESOLUTION,COUNTED_BRACE_COMPONENTS,MORE_ROWS_THAN_COMPONENTS_MANY_TO_ONE_CANDIDATE


## Step 7 — Search for the same parsed source across multiple ASDM associations

The first queries are intentionally limited discovery scans. Candidate Members
are subsequently counted and retrieved completely before ownership evidence is
evaluated. Absence in a limited discovery scan is never treated as an
Archive-wide proof of absence.

In [39]:
MULTI_ASDM_DISCOVERY_LIMIT = 600
REPEATED_SOURCE_MEMBER_LIMIT = 25

multi_asdm_discovery_queries = {
    "early": f"""
        SELECT TOP {MULTI_ASDM_DISCOVERY_LIMIT}
            proposal_id, member_ous_uid, asdm_uid, obs_id
        FROM ivoa.obscore
        WHERE science_observation = 'T'
          AND (
              proposal_id LIKE '2011.%'
              OR proposal_id LIKE '2012.%'
              OR proposal_id LIKE '2013.%'
              OR proposal_id LIKE '2014.%'
              OR proposal_id LIKE '2015.%'
          )
        ORDER BY member_ous_uid, asdm_uid, obs_id
    """,
    "middle": f"""
        SELECT TOP {MULTI_ASDM_DISCOVERY_LIMIT}
            proposal_id, member_ous_uid, asdm_uid, obs_id
        FROM ivoa.obscore
        WHERE science_observation = 'T'
          AND (
              proposal_id LIKE '2016.%'
              OR proposal_id LIKE '2017.%'
              OR proposal_id LIKE '2018.%'
              OR proposal_id LIKE '2019.%'
              OR proposal_id LIKE '2020.%'
              OR proposal_id LIKE '2021.%'
          )
        ORDER BY member_ous_uid, asdm_uid, obs_id
    """,
    "recent": f"""
        SELECT TOP {MULTI_ASDM_DISCOVERY_LIMIT}
            proposal_id, member_ous_uid, asdm_uid, obs_id
        FROM ivoa.obscore
        WHERE science_observation = 'T'
          AND (
              proposal_id LIKE '2022.%'
              OR proposal_id LIKE '2023.%'
              OR proposal_id LIKE '2024.%'
              OR proposal_id LIKE '2025.%'
              OR proposal_id LIKE '2026.%'
          )
        ORDER BY member_ous_uid, asdm_uid, obs_id
    """,
}

multi_asdm_discovery_frames = []

for discovery_stratum, adql in multi_asdm_discovery_queries.items():
    run = run_tap_query(
        adql,
        maxrec=MULTI_ASDM_DISCOVERY_LIMIT,
        label=f"Discover multi-ASDM candidates: {discovery_stratum}",
    )
    query_runs.append(run)

    frame = run.table.to_pandas()
    frame["discovery_stratum"] = discovery_stratum
    frame["discovery_completeness"] = "INTENTIONALLY_LIMITED_DISCOVERY"
    multi_asdm_discovery_frames.append(frame)

multi_asdm_discovery_df = pd.concat(
    multi_asdm_discovery_frames,
    ignore_index=True,
)
multi_asdm_discovery_parsed_df = pd.DataFrame(
    multi_asdm_discovery_df["obs_id"]
    .map(parse_obs_id_structure)
    .tolist()
)
multi_asdm_discovery_analysis_df = pd.concat(
    [
        multi_asdm_discovery_df.reset_index(drop=True),
        multi_asdm_discovery_parsed_df,
    ],
    axis=1,
)

discovered_repeated_source_groups_df = (
    multi_asdm_discovery_analysis_df
    .groupby(["member_ous_uid", "source_name"], dropna=False)
    .agg(
        discovered_asdm_count=("asdm_uid", "nunique"),
        discovered_rows=("obs_id", "size"),
    )
    .reset_index()
)
discovered_repeated_source_groups_df = (
    discovered_repeated_source_groups_df.loc[
        discovered_repeated_source_groups_df["discovered_asdm_count"] > 1
    ]
    .copy()
)

discovered_multi_asdm_members_df = (
    multi_asdm_discovery_analysis_df
    .groupby("member_ous_uid", dropna=False)
    .agg(discovered_asdm_count=("asdm_uid", "nunique"))
    .reset_index()
)
discovered_multi_asdm_members_df = (
    discovered_multi_asdm_members_df.loc[
        discovered_multi_asdm_members_df["discovered_asdm_count"] > 1
    ]
)

# The complete cardinality-census sample is stronger discovery evidence than
# the TOP-limited scans, so include its multi-ASDM and repeated-source Members.
cardinality_repeated_source_groups_df = (
    cardinality_analysis_df
    .groupby(["member_ous_uid", "source_name"], dropna=False)
    .agg(discovered_asdm_count=("asdm_uid", "nunique"))
    .reset_index()
)
cardinality_repeated_source_groups_df = (
    cardinality_repeated_source_groups_df.loc[
        cardinality_repeated_source_groups_df["discovered_asdm_count"] > 1
    ]
)

cardinality_multi_asdm_members_df = (
    cardinality_analysis_df
    .groupby("member_ous_uid", dropna=False)
    .agg(discovered_asdm_count=("asdm_uid", "nunique"))
    .reset_index()
)
cardinality_multi_asdm_members_df = (
    cardinality_multi_asdm_members_df.loc[
        cardinality_multi_asdm_members_df["discovered_asdm_count"] > 1
    ]
)

repeated_source_candidate_uids = list(
    dict.fromkeys(
        cardinality_repeated_source_groups_df["member_ous_uid"]
        .dropna()
        .astype(str)
        .tolist()
        + cardinality_multi_asdm_members_df["member_ous_uid"]
        .dropna()
        .astype(str)
        .tolist()
        + discovered_repeated_source_groups_df["member_ous_uid"]
        .dropna()
        .astype(str)
        .tolist()
        + discovered_multi_asdm_members_df["member_ous_uid"]
        .dropna()
        .astype(str)
        .tolist()
        + [known_multi_asdm_member]
    )
)[:REPEATED_SOURCE_MEMBER_LIMIT]

repeated_source_candidate_sql = ",\n    ".join(
    quote_adql_string(uid)
    for uid in repeated_source_candidate_uids
)

ownership_columns = [
    name
    for name in [
        "proposal_id", "member_ous_uid", "group_ous_uid", "asdm_uid",
        "obs_id", "target_name", "s_ra", "s_dec", "s_fov",
        "s_region", "is_mosaic", "s_resolution", "spatial_resolution",
        "spatial_scale_max", "frequency_support", "antenna_arrays",
        "t_min", "t_max", "cont_sensitivity_bandwidth",
        "dataproduct_type", "calib_level", "frequency", "bandwidth",
    ]
    if name in available_columns
]
ownership_column_sql = ",\n    ".join(ownership_columns)

repeated_source_count_query = f"""
SELECT COUNT(*) AS total_rows
FROM ivoa.obscore
WHERE science_observation = 'T'
  AND member_ous_uid IN (
    {repeated_source_candidate_sql}
  )
"""

repeated_source_count_run = run_tap_query(
    repeated_source_count_query,
    maxrec=10,
    label="Count complete repeated-source candidate sample",
)
query_runs.append(repeated_source_count_run)
repeated_source_expected_rows = int(
    repeated_source_count_run.table[0]["total_rows"]
)

if repeated_source_expected_rows > EXPLORATORY_SAFETY_LIMIT:
    raise RuntimeError(
        "Repeated-source candidate sample exceeds the safety limit. "
        "Reduce REPEATED_SOURCE_MEMBER_LIMIT."
    )

repeated_source_query = f"""
SELECT
    {ownership_column_sql}
FROM ivoa.obscore
WHERE science_observation = 'T'
  AND member_ous_uid IN (
    {repeated_source_candidate_sql}
  )
"""

repeated_source_run = run_tap_query(
    repeated_source_query,
    maxrec=max(repeated_source_expected_rows, 1),
    label="Retrieve complete repeated-source candidate sample",
)
query_runs.append(repeated_source_run)
repeated_source_df = repeated_source_run.table.to_pandas()

if len(repeated_source_df) != repeated_source_expected_rows:
    raise RuntimeError(
        "Repeated-source candidate retrieval is incomplete: "
        f"{len(repeated_source_df)} / {repeated_source_expected_rows}."
    )

repeated_source_parsed_df = pd.DataFrame(
    repeated_source_df["obs_id"].map(parse_obs_id_structure).tolist()
)
repeated_source_analysis_df = pd.concat(
    [repeated_source_df.reset_index(drop=True), repeated_source_parsed_df],
    axis=1,
)

verified_repeated_source_df = (
    repeated_source_analysis_df
    .groupby(["member_ous_uid", "source_name"], dropna=False)
    .agg(
        asdm_count=("asdm_uid", "nunique"),
        archive_rows=("obs_id", "size"),
        asdm_uids=(
            "asdm_uid",
            lambda values: tuple(sorted(set(values.dropna().astype(str)))),
        ),
    )
    .reset_index()
)
verified_repeated_source_df = verified_repeated_source_df.loc[
    verified_repeated_source_df["asdm_count"] > 1
].copy()

ownership_test_fields = [
    name
    for name in [
        "target_name", "s_ra", "s_dec", "s_fov", "s_region",
        "is_mosaic", "s_resolution", "spatial_resolution",
        "spatial_scale_max", "frequency_support", "antenna_arrays",
        "t_min", "t_max", "cont_sensitivity_bandwidth",
    ]
    if name in repeated_source_analysis_df.columns
]


def exact_value_signature(series: pd.Series) -> tuple[str, ...]:
    return tuple(
        sorted(
            set(
                "<MISSING>" if pd.isna(value) else repr(value)
                for value in series
            )
        )
    )


cross_asdm_ownership_records = []

for repeated_group in verified_repeated_source_df.itertuples(index=False):
    group = repeated_source_analysis_df.loc[
        repeated_source_analysis_df["member_ous_uid"].eq(
            repeated_group.member_ous_uid
        )
        & repeated_source_analysis_df["source_name"].eq(
            repeated_group.source_name
        )
    ]

    for field in ownership_test_fields:
        signatures_by_asdm = {
            str(asdm_uid): exact_value_signature(asdm_group[field])
            for asdm_uid, asdm_group in group.groupby(
                "asdm_uid",
                dropna=False,
            )
        }
        signature_count = len(set(signatures_by_asdm.values()))

        cross_asdm_ownership_records.append(
            {
                "member_ous_uid": repeated_group.member_ous_uid,
                "source_name": repeated_group.source_name,
                "field": field,
                "asdm_count": repeated_group.asdm_count,
                "distinct_asdm_value_signatures": signature_count,
                "cross_asdm_status": (
                    "STABLE_ACROSS_ASDM"
                    if signature_count <= 1
                    else "VARIES_ACROSS_ASDM"
                ),
                "values_by_asdm": signatures_by_asdm,
            }
        )

cross_asdm_ownership_df = pd.DataFrame(cross_asdm_ownership_records)

repeated_source_ownership_status = (
    "TESTABLE_REPEATED_SOURCE_FOUND"
    if len(verified_repeated_source_df)
    else "UNRESOLVED_NO_REPEATED_SOURCE_ACROSS_ASDM"
)

print("Discovery scans are intentionally limited.")
print("Candidate Members retrieved completely:", len(repeated_source_candidate_uids))
print("Repeated Source Contexts verified across ASDM:", len(verified_repeated_source_df))
print("Ownership evidence status:", repeated_source_ownership_status)
display(discovered_repeated_source_groups_df)
display(verified_repeated_source_df)
display(cross_asdm_ownership_df)


--- Discover multi-ASDM candidates: early ---
Endpoint: https://almascience.eso.org/tap
MAXREC: 600
ADQL:

        SELECT TOP 600
            proposal_id, member_ous_uid, asdm_uid, obs_id
        FROM ivoa.obscore
        WHERE science_observation = 'T'
          AND (
              proposal_id LIKE '2011.%'
              OR proposal_id LIKE '2012.%'
              OR proposal_id LIKE '2013.%'
              OR proposal_id LIKE '2014.%'
              OR proposal_id LIKE '2015.%'
          )
        ORDER BY member_ous_uid, asdm_uid, obs_id
    


Retrieved rows: 600
QUERY_STATUS: ('OK',)
Warnings: <none>

--- Discover multi-ASDM candidates: middle ---
Endpoint: https://almascience.eso.org/tap
MAXREC: 600
ADQL:

        SELECT TOP 600
            proposal_id, member_ous_uid, asdm_uid, obs_id
        FROM ivoa.obscore
        WHERE science_observation = 'T'
          AND (
              proposal_id LIKE '2016.%'
              OR proposal_id LIKE '2017.%'
              OR proposal_id LIKE '2018.%'
              OR proposal_id LIKE '2019.%'
              OR proposal_id LIKE '2020.%'
              OR proposal_id LIKE '2021.%'
          )
        ORDER BY member_ous_uid, asdm_uid, obs_id
    
Retrieved rows: 600
QUERY_STATUS: ('OK',)
Warnings: <none>

--- Discover multi-ASDM candidates: recent ---
Endpoint: https://almascience.eso.org/tap
MAXREC: 600
ADQL:

        SELECT TOP 600
            proposal_id, member_ous_uid, asdm_uid, obs_id
        FROM ivoa.obscore
        WHERE science_observation = 'T'
          AND (
              pr

,member_ous_uid,source_name,discovered_asdm_count,discovered_rows


,member_ous_uid,source_name,asdm_count,archive_rows,asdm_uids


""


## Step 8 — ObsCore product granularity and mask-aware field inspection

This experiment adds data-product and axis-size fields without downloading any
science products. It tests whether an Archive row should be modeled as ObsCore
product metadata rather than as an independent physical observation. Astropy
masks, Pandas nulls, blank strings, and sentinel dates are counted separately.

In [40]:
PRODUCT_MEMBER_LIMIT = 50

product_member_uids = cardinality_member_uids[:PRODUCT_MEMBER_LIMIT]
product_member_sql = ",\n    ".join(
    quote_adql_string(uid)
    for uid in product_member_uids
)

product_columns = [
    name
    for name in [
        "proposal_id", "member_ous_uid", "group_ous_uid", "asdm_uid",
        "obs_id", "target_name", "frequency_support", "frequency",
        "bandwidth", "spectral_resolution", "dataproduct_type",
        "calib_level", "em_xel", "pol_xel", "s_xel1", "s_xel2",
        "t_xel", "access_format", "access_estsize", "facility_name",
        "instrument_name", "obs_collection", "antenna_arrays",
        "is_mosaic", "obs_release_date",
    ]
    if name in available_columns
]
product_column_sql = ",\n    ".join(product_columns)

product_count_query = f"""
SELECT COUNT(*) AS total_rows
FROM ivoa.obscore
WHERE science_observation = 'T'
  AND member_ous_uid IN (
    {product_member_sql}
  )
"""

product_count_run = run_tap_query(
    product_count_query,
    maxrec=10,
    label="Count complete product-granularity sample",
)
query_runs.append(product_count_run)
product_expected_rows = int(product_count_run.table[0]["total_rows"])

if product_expected_rows > EXPLORATORY_SAFETY_LIMIT:
    raise RuntimeError(
        "Product-granularity sample exceeds the safety limit. "
        "Reduce PRODUCT_MEMBER_LIMIT."
    )

product_query = f"""
SELECT
    {product_column_sql}
FROM ivoa.obscore
WHERE science_observation = 'T'
  AND member_ous_uid IN (
    {product_member_sql}
  )
"""

product_run = run_tap_query(
    product_query,
    maxrec=max(product_expected_rows, 1),
    label="Retrieve complete product-granularity sample",
)
query_runs.append(product_run)
product_table = product_run.table
product_df = product_table.to_pandas()

if len(product_df) != product_expected_rows:
    raise RuntimeError(
        "Product-granularity retrieval is incomplete: "
        f"{len(product_df)} / {product_expected_rows}."
    )

product_mask_counts = {}
for column_name in product_table.colnames:
    mask = getattr(product_table[column_name], "mask", None)
    product_mask_counts[column_name] = (
        int(np.asarray(mask).sum())
        if mask is not None
        else 0
    )

product_missingness_records = []
for column_name in product_columns:
    series = product_df[column_name]
    blank_count = 0
    if (
        pd.api.types.is_object_dtype(series)
        or pd.api.types.is_string_dtype(series)
    ):
        blank_count = int(
            series.astype("string")
            .str.strip()
            .eq("")
            .fillna(False)
            .sum()
        )

    sentinel_count = 0
    if column_name == "obs_release_date":
        sentinel_count = int(
            series.astype("string")
            .str.startswith("3000-01-01", na=False)
            .sum()
        )

    product_missingness_records.append(
        {
            "column_name": column_name,
            "astropy_mask_count": product_mask_counts.get(column_name, 0),
            "pandas_null_count": int(series.isna().sum()),
            "blank_string_count": blank_count,
            "sentinel_count": sentinel_count,
        }
    )

product_missingness_df = pd.DataFrame(product_missingness_records)

product_parsed_obs_df = pd.DataFrame(
    product_df["obs_id"].map(parse_obs_id_structure).tolist()
)
product_analysis_df = pd.concat(
    [product_df.reset_index(drop=True), product_parsed_obs_df],
    axis=1,
)
product_analysis_df["support_grammar_family"] = (
    product_analysis_df["frequency_support"]
    .map(classify_support_grammar)
)
product_analysis_df["antenna_prefixes"] = (
    product_analysis_df["antenna_arrays"].map(antenna_prefixes)
)
product_analysis_df["array_evidence_class"] = (
    product_analysis_df["antenna_prefixes"]
    .map(classify_array_evidence)
)

axis_fields = [
    name
    for name in ["em_xel", "pol_xel", "s_xel1", "s_xel2", "t_xel"]
    if name in product_analysis_df.columns
]

product_shape_records = []
product_group_keys = [
    "dataproduct_type",
    "calib_level",
    "support_grammar_family",
    "array_evidence_class",
]

for group_key, group in product_analysis_df.groupby(
    product_group_keys,
    dropna=False,
):
    record = dict(zip(product_group_keys, group_key, strict=True))
    record.update(
        {
            "archive_rows": len(group),
            "member_count": group["member_ous_uid"].nunique(),
        }
    )

    for axis_field in axis_fields:
        numeric_values = pd.to_numeric(
            group[axis_field],
            errors="coerce",
        ).dropna()
        record[f"{axis_field}_available_rows"] = len(numeric_values)
        record[f"{axis_field}_min"] = (
            numeric_values.min() if len(numeric_values) else np.nan
        )
        record[f"{axis_field}_median"] = (
            numeric_values.median() if len(numeric_values) else np.nan
        )
        record[f"{axis_field}_max"] = (
            numeric_values.max() if len(numeric_values) else np.nan
        )

    product_shape_records.append(record)

product_shape_inventory_df = pd.DataFrame(product_shape_records)

source_spw_product_df = (
    product_analysis_df
    .groupby(
        ["member_ous_uid", "asdm_uid", "source_name", "spw_id"],
        dropna=False,
    )
    .agg(
        archive_rows=("obs_id", "size"),
        obs_id_count=("obs_id", "nunique"),
        dataproduct_types=(
            "dataproduct_type",
            lambda values: tuple(sorted(set(values.dropna().astype(str)))),
        ),
        calibration_levels=(
            "calib_level",
            lambda values: tuple(sorted(set(values.dropna()))),
        ),
        access_formats=(
            "access_format",
            lambda values: tuple(sorted(set(values.dropna().astype(str)))),
        ) if "access_format" in product_analysis_df.columns else (
            "obs_id", "size"
        ),
    )
    .reset_index()
)
source_spw_product_df["multiple_product_rows"] = (
    source_spw_product_df["archive_rows"] > 1
)

display(
    product_missingness_df.sort_values(
        [
            "astropy_mask_count",
            "pandas_null_count",
            "blank_string_count",
            "sentinel_count",
        ],
        ascending=False,
    )
)
display(product_shape_inventory_df)
display(
    source_spw_product_df.loc[
        source_spw_product_df["multiple_product_rows"]
    ]
)



--- Count complete product-granularity sample ---
Endpoint: https://almascience.eso.org/tap
MAXREC: 10
ADQL:

SELECT COUNT(*) AS total_rows
FROM ivoa.obscore
WHERE science_observation = 'T'
  AND member_ous_uid IN (
    'uid://A001/X1288/X4c3',
    'uid://A001/X1288/X4c7',
    'uid://A001/X1288/X4d7',
    'uid://A001/X1288/X4dd',
    'uid://A001/X3621/X3ca0',
    'uid://A001/X3621/X3ca4',
    'uid://A001/X362b/Xbe5',
    'uid://A001/X3645/X15f',
    'uid://A001/X3833/X1022',
    'uid://A001/X3833/X14b4',
    'uid://A001/X3833/X1982',
    'uid://A001/X3833/X198e',
    'uid://A001/X3833/X1998',
    'uid://A001/X3845/X5cf',
    'uid://A001/X3845/X776',
    'uid://A001/X3845/X77a',
    'uid://A001/X3873/X44c',
    'uid://A001/X38cd/X1d',
    'uid://A001/X38cd/X1ea',
    'uid://A001/X38cd/X2d',
    'uid://A001/X38cd/X3c3',
    'uid://A001/X38cd/X3c7',
    'uid://A001/X38cd/Xd4',
    'uid://A001/X38cd/Xd7',
    'uid://A001/X3922/X599',
    'uid://A001/X3922/X59c',
    'uid://A001/X3922/X59f

,column_name,astropy_mask_count,pandas_null_count,blank_string_count,sentinel_count
14,s_xel1,227,227,0,0
15,s_xel2,227,227,0,0
18,access_estsize,227,227,0,0
2,group_ous_uid,0,0,8,0
24,obs_release_date,0,0,0,39
0,proposal_id,0,0,0,0
1,member_ous_uid,0,0,0,0
3,asdm_uid,0,0,0,0
4,obs_id,0,0,0,0
5,target_name,0,0,0,0


,dataproduct_type,calib_level,support_grammar_family,array_evidence_class,archive_rows,member_count,em_xel_available_rows,em_xel_min,em_xel_median,em_xel_max,pol_xel_available_rows,pol_xel_min,pol_xel_median,pol_xel_max,s_xel1_available_rows,s_xel1_min,s_xel1_median,s_xel1_max,s_xel2_available_rows,s_xel2_min,s_xel2_median,s_xel2_max,t_xel_available_rows,t_xel_min,t_xel_median,t_xel_max
0,cube,2,BRACKET_INTERVAL,12M_LIKE,79,14,79,240,1920.0,3840,79,2,2.0,2,0,NaN,NaN,NaN,0,NaN,NaN,NaN,79,1,1.0,1
1,cube,2,BRACKET_INTERVAL,12M_LIKE+7M_LIKE,4,1,4,960,960.0,960,4,4,4.0,4,0,NaN,NaN,NaN,0,NaN,NaN,NaN,4,1,1.0,1
2,cube,2,BRACKET_INTERVAL,12M_LIKE+TP_LIKE,12,3,12,480,960.0,3840,12,2,2.0,2,0,NaN,NaN,NaN,0,NaN,NaN,NaN,12,1,1.0,1
3,cube,2,BRACKET_INTERVAL,7M_LIKE,25,4,25,240,960.0,3840,25,2,2.0,2,0,NaN,NaN,NaN,0,NaN,NaN,NaN,25,1,1.0,1
4,image,2,BRACE_CENTRE_RESOLUTION,TP_LIKE,43,5,43,1,1.0,1,43,2,2.0,2,0,NaN,NaN,NaN,0,NaN,NaN,NaN,43,1,1.0,1
5,image,2,BRACKET_INTERVAL,12M_LIKE,16,4,16,128,128.0,128,16,2,2.0,2,0,NaN,NaN,NaN,0,NaN,NaN,NaN,16,1,1.0,1
6,image,2,BRACKET_INTERVAL,12M_LIKE+7M_LIKE,4,1,4,128,128.0,128,4,2,2.0,2,0,NaN,NaN,NaN,0,NaN,NaN,NaN,4,1,1.0,1
7,image,2,BRACKET_INTERVAL,12M_LIKE+7M_LIKE+TP_LIKE,12,1,12,128,128.0,128,12,2,2.0,2,0,NaN,NaN,NaN,0,NaN,NaN,NaN,12,1,1.0,1
8,image,2,BRACKET_INTERVAL,12M_LIKE+TP_LIKE,32,6,32,64,128.0,128,32,2,2.0,4,0,NaN,NaN,NaN,0,NaN,NaN,NaN,32,1,1.0,1


,member_ous_uid,asdm_uid,source_name,spw_id,archive_rows,obs_id_count,dataproduct_types,calibration_levels,access_formats,multiple_product_rows


## Step 9 — STC-S spatial-geometry coverage

This is a structural grammar and ownership diagnostic, not a duplication
overlap calculation. Complete raw `s_region` values are preserved. The parser
records top-level geometry, coordinate frame, child geometry count, numeric
token count, balanced parentheses, and explicit issue codes.

In [41]:
STCS_NUMBER_PATTERN = re.compile(
    r"[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[Ee][-+]?\d+)?"
)
KNOWN_STCS_GEOMETRIES = {
    "CIRCLE", "POLYGON", "UNION", "BOX", "POSITION",
    "INTERSECTION", "NOT",
}


def parse_stcs_structure(raw_region: object) -> dict[str, object]:
    if pd.isna(raw_region) or not str(raw_region).strip():
        return {
            "geometry_type": "MISSING",
            "coordinate_frame": None,
            "geometry_component_count": 0,
            "numeric_token_count": 0,
            "parentheses_balanced": True,
            "geometry_parse_status": "MISSING",
            "geometry_issue_codes": ("missing_region",),
            "normalized_geometry_hash": None,
            "numeric_range_wrap_candidate_heuristic": False,
        }

    raw_text = str(raw_region).strip()
    normalized_text = " ".join(raw_text.split())
    header_match = re.match(
        r"^(?P<geometry>[A-Za-z]+)(?:\s+(?P<frame>[A-Za-z0-9_]+))?",
        normalized_text,
    )

    issues = []
    if header_match is None:
        geometry_type = "UNKNOWN"
        coordinate_frame = None
        issues.append("unrecognized_header")
    else:
        geometry_type = header_match.group("geometry").upper()
        coordinate_frame = (
            header_match.group("frame").upper()
            if header_match.group("frame")
            else None
        )

    if geometry_type not in KNOWN_STCS_GEOMETRIES:
        issues.append("unknown_geometry_type")

    parentheses_balanced = (
        normalized_text.count("(") == normalized_text.count(")")
    )
    if not parentheses_balanced:
        issues.append("unbalanced_parentheses")

    numeric_values = [
        float(value)
        for value in STCS_NUMBER_PATTERN.findall(normalized_text)
    ]

    child_geometry_types = tuple(
        re.findall(
            r"\b(CIRCLE|POLYGON|BOX|POSITION)\b",
            normalized_text.upper(),
        )
    )

    if geometry_type == "UNION":
        geometry_component_count = len(child_geometry_types)
        if geometry_component_count == 0:
            issues.append("union_without_detected_children")
    else:
        geometry_component_count = 1

    if geometry_type == "CIRCLE" and len(numeric_values) < 3:
        issues.append("circle_requires_three_numeric_values")
    if geometry_type == "POLYGON":
        if len(numeric_values) < 6:
            issues.append("polygon_requires_at_least_three_vertices")
        if len(numeric_values) % 2:
            issues.append("polygon_has_odd_coordinate_count")

    # This intentionally scans all numeric tokens. It is only a screening
    # heuristic; it does not prove that the extreme values occupy RA slots.
    numeric_range_wrap_candidate_heuristic = (
        any(0 <= value < 5 for value in numeric_values)
        and any(355 < value <= 360 for value in numeric_values)
    )

    parse_status = (
        "PARSED_STRUCTURALLY"
        if not issues
        else "PARTIAL_OR_INVALID"
    )

    return {
        "geometry_type": geometry_type,
        "coordinate_frame": coordinate_frame,
        "geometry_component_count": geometry_component_count,
        "child_geometry_types": child_geometry_types,
        "numeric_token_count": len(numeric_values),
        "parentheses_balanced": parentheses_balanced,
        "geometry_parse_status": parse_status,
        "geometry_issue_codes": tuple(issues),
        "normalized_geometry_hash": hashlib.sha256(
            normalized_text.encode("utf-8")
        ).hexdigest(),
        "numeric_range_wrap_candidate_heuristic": (
            numeric_range_wrap_candidate_heuristic
        ),
    }


geometry_source_df = cardinality_analysis_df.copy()
geometry_parsed_df = pd.DataFrame(
    geometry_source_df["s_region"]
    .map(parse_stcs_structure)
    .tolist()
)
geometry_analysis_df = pd.concat(
    [geometry_source_df.reset_index(drop=True), geometry_parsed_df],
    axis=1,
)

geometry_analysis_df["source_execution_context_key"] = list(
    zip(
        geometry_analysis_df["member_ous_uid"],
        geometry_analysis_df["asdm_uid"],
        geometry_analysis_df["source_name"],
    )
)

geometry_grammar_inventory_df = (
    geometry_analysis_df
    .groupby(
        [
            "geometry_type",
            "coordinate_frame",
            "geometry_parse_status",
            "is_mosaic",
        ],
        dropna=False,
    )
    .agg(
        archive_rows=("cardinality_row_index", "size"),
        member_count=("member_ous_uid", "nunique"),
        source_execution_context_count=(
            "source_execution_context_key",
            "nunique",
        ),
        maximum_component_count=("geometry_component_count", "max"),
        maximum_numeric_token_count=("numeric_token_count", "max"),
        numeric_range_wrap_candidates_heuristic=(
            "numeric_range_wrap_candidate_heuristic",
            "sum",
        ),
    )
    .reset_index()
)

geometry_parse_failure_df = geometry_analysis_df.loc[
    ~geometry_analysis_df["geometry_parse_status"].eq(
        "PARSED_STRUCTURALLY"
    )
].copy()

spatial_context_geometry_df = (
    geometry_analysis_df
    .groupby(
        ["member_ous_uid", "asdm_uid", "source_name"],
        dropna=False,
    )
    .agg(
        archive_rows=("cardinality_row_index", "size"),
        distinct_footprints=("normalized_geometry_hash", "nunique"),
        geometry_types=(
            "geometry_type",
            lambda values: tuple(sorted(set(values))),
        ),
        mosaic_states=(
            "is_mosaic",
            lambda values: tuple(sorted(set(values.dropna().astype(str)))),
        ),
        parse_statuses=(
            "geometry_parse_status",
            lambda values: tuple(sorted(set(values))),
        ),
    )
    .reset_index()
)

geometry_examples_df = (
    geometry_analysis_df
    .sort_values(
        ["geometry_type", "member_ous_uid", "source_name"],
        kind="stable",
    )
    .groupby("geometry_type", dropna=False, sort=False)
    .head(3)
    [
        [
            "member_ous_uid", "asdm_uid", "source_name", "is_mosaic",
            "geometry_type", "coordinate_frame", "geometry_component_count",
            "child_geometry_types", "numeric_token_count",
            "geometry_parse_status", "geometry_issue_codes", "s_region",
        ]
    ]
)

display(geometry_grammar_inventory_df)
display(geometry_examples_df)
display(
    spatial_context_geometry_df.loc[
        spatial_context_geometry_df["distinct_footprints"] > 1
    ]
)
display(geometry_parse_failure_df)


,geometry_type,coordinate_frame,geometry_parse_status,is_mosaic,archive_rows,member_count,source_execution_context_count,maximum_component_count,maximum_numeric_token_count,numeric_range_wrap_candidates_heuristic
0,CIRCLE,ICRS,PARSED_STRUCTURALLY,F,131,21,28,1,3,0
1,POLYGON,ICRS,PARSED_STRUCTURALLY,F,48,11,11,1,40,0
2,POLYGON,ICRS,PARSED_STRUCTURALLY,T,16,3,4,1,102,0
3,UNION,ICRS,PARSED_STRUCTURALLY,T,32,6,6,11,352,4


,member_ous_uid,asdm_uid,source_name,is_mosaic,geometry_type,coordinate_frame,geometry_component_count,child_geometry_types,numeric_token_count,geometry_parse_status,geometry_issue_codes,s_region
0,uid://A001/X1288/X4d7,uid://A002/Xca795f/Xa6dc,ngc4945,F,CIRCLE,ICRS,1,"(CIRCLE,)",3,PARSED_STRUCTURALLY,(),Circle ICRS 196.363662 -49.467900 0.007509
1,uid://A001/X1288/X4d7,uid://A002/Xca795f/Xa6dc,ngc4945,F,CIRCLE,ICRS,1,"(CIRCLE,)",3,PARSED_STRUCTURALLY,(),Circle ICRS 196.363662 -49.467900 0.007509
2,uid://A001/X1288/X4d7,uid://A002/Xca795f/Xa6dc,ngc4945,F,CIRCLE,ICRS,1,"(CIRCLE,)",3,PARSED_STRUCTURALLY,(),Circle ICRS 196.363662 -49.467900 0.007509
86,uid://A001/X1288/X4c3,uid://A002/Xc72427/X171b,UDF2,F,POLYGON,ICRS,1,"(POLYGON,)",40,PARSED_STRUCTURALLY,(),Polygon ICRS 53.183849 -27.776703 53.184036 -27.777569 53.183849 -27.778436 53.183622 -27.778831 53.183197 -27.77928...
87,uid://A001/X1288/X4c3,uid://A002/Xc72427/X171b,UDF2,F,POLYGON,ICRS,1,"(POLYGON,)",40,PARSED_STRUCTURALLY,(),Polygon ICRS 53.183849 -27.776703 53.184036 -27.777569 53.183849 -27.778436 53.183622 -27.778831 53.183197 -27.77928...
88,uid://A001/X1288/X4c3,uid://A002/Xc72427/X171b,UDF2,F,POLYGON,ICRS,1,"(POLYGON,)",40,PARSED_STRUCTURALLY,(),Polygon ICRS 53.183849 -27.776703 53.184036 -27.777569 53.183849 -27.778436 53.183622 -27.778831 53.183197 -27.77928...
49,uid://A001/X3845/X776,uid://A002/X130a07a/X2b5ab,ATLAS_C2025_N1,T,UNION,ICRS,11,"(POLYGON, POLYGON, POLYGON, POLYGON, POLYGON, POLYGON, POLYGON, POLYGON, POLYGON, POLYGON, POLYGON)",352,PARSED_STRUCTURALLY,(),Union ICRS ( Polygon 202.260021 -6.903152 202.259639 -6.905193 202.258527 -6.906951 202.256520 -6.908320 202.254468 ...
50,uid://A001/X3845/X776,uid://A002/X130a07a/X2b5ab,ATLAS_C2025_N1,T,UNION,ICRS,11,"(POLYGON, POLYGON, POLYGON, POLYGON, POLYGON, POLYGON, POLYGON, POLYGON, POLYGON, POLYGON, POLYGON)",352,PARSED_STRUCTURALLY,(),Union ICRS ( Polygon 202.260021 -6.903152 202.259639 -6.905193 202.258527 -6.906951 202.256520 -6.908320 202.254468 ...
51,uid://A001/X3845/X776,uid://A002/X130a07a/X2b5ab,ATLAS_C2025_N1,T,UNION,ICRS,11,"(POLYGON, POLYGON, POLYGON, POLYGON, POLYGON, POLYGON, POLYGON, POLYGON, POLYGON, POLYGON, POLYGON)",352,PARSED_STRUCTURALLY,(),Union ICRS ( Polygon 202.260021 -6.903152 202.259639 -6.905193 202.258527 -6.906951 202.256520 -6.908320 202.254468 ...


,member_ous_uid,asdm_uid,source_name,archive_rows,distinct_footprints,geometry_types,mosaic_states,parse_statuses


,cardinality_row_index,proposal_id,obs_publisher_did,group_ous_uid,member_ous_uid,obs_id,asdm_uid,target_name,s_ra,s_dec,s_fov,s_region,s_resolution,t_min,t_max,t_exptime,t_resolution,em_min,em_max,em_res_power,frequency,bandwidth,frequency_support,spectral_resolution,velocity_resolution,em_resolution,spatial_resolution,spatial_scale_max,sensitivity_10kms,cont_sensitivity_bandwidth,antenna_arrays,is_mosaic,pol_states,band_list,scan_intent,science_observation,data_rights,qa2_passed,obs_release_date,dataproduct_type,calib_level,type,pwv,lastModified,obs_member_ous_uid,source_name,spw_id,obs_id_parse_status,obs_id_parse_issue,support_grammar_family,support_component_count,support_count_status,source_execution_context_key,geometry_type,coordinate_frame,geometry_component_count,child_geometry_types,numeric_token_count,parentheses_balanced,geometry_parse_status,geometry_issue_codes,normalized_geometry_hash,numeric_range_wrap_candidate_heuristic


## Step 10 — Reconstruction shuffle invariance

This experiment tests, within the complete selected sample, whether
reconstructed identifiers, Member summaries, context cardinalities,
geometry signatures, and brace nearest-centre mappings depend on TAP row
order. It uses deterministic canonical serialization and SHA-256
fingerprints across multiple shuffles. Passing is sample evidence, not an
Archive-wide proof.


In [42]:
SHUFFLE_SEEDS = [0, 1, 7, 42, 2026]


def canonical_dataframe_hash(dataframe: pd.DataFrame) -> str:
    canonical = dataframe.copy()

    def canonical_cell(value: object) -> str:
        if isinstance(value, dict):
            normalized_items = tuple(
                sorted(
                    (str(key), canonical_cell(item))
                    for key, item in value.items()
                )
            )
            return repr(normalized_items)
        if isinstance(value, set):
            return repr(tuple(sorted(canonical_cell(item) for item in value)))
        if isinstance(value, (tuple, list)):
            return repr(tuple(canonical_cell(item) for item in value))
        try:
            if bool(pd.isna(value)):
                return "<MISSING>"
        except (TypeError, ValueError):
            pass
        return repr(value)

    if len(canonical.columns) == 0:
        serialization = "<NO_COLUMNS>"
    elif len(canonical) == 0:
        serialization = "<EMPTY>\n" + ",".join(canonical.columns)
    else:
        for column_name in canonical.columns:
            canonical[column_name] = canonical[column_name].map(canonical_cell)
        canonical = canonical.sort_values(
            list(canonical.columns),
            kind="stable",
        ).reset_index(drop=True)
        serialization = canonical.to_csv(index=False)

    return hashlib.sha256(serialization.encode("utf-8")).hexdigest()


def reconstruct_order_independent_tables(
    dataframe: pd.DataFrame,
) -> dict[str, pd.DataFrame]:
    work = dataframe.copy().reset_index(drop=True)
    parsed = pd.DataFrame(
        work["obs_id"].map(parse_obs_id_structure).tolist()
    )
    for parsed_column in parsed.columns:
        work[parsed_column] = parsed[parsed_column].to_numpy()
    work["support_grammar_family"] = (
        work["frequency_support"].map(classify_support_grammar)
    )

    support_metadata = pd.DataFrame(
        work["frequency_support"]
        .map(support_component_count)
        .tolist()
    )

    for metadata_column in support_metadata.columns:
        work[metadata_column] = (
            support_metadata[metadata_column].to_numpy()
        )

    assert work.columns.is_unique, (
        "Reconstruction produced duplicate columns: "
        f"{work.columns[work.columns.duplicated()].tolist()}"
    )

    row_identity_table = work[
        [
            "member_ous_uid", "asdm_uid", "obs_id", "source_name",
            "spw_id", "frequency", "bandwidth",
            "support_grammar_family", "support_component_count",
        ]
    ].copy()

    member_table = (
        work.groupby("member_ous_uid", dropna=False)
        .agg(
            archive_rows=("obs_id", "size"),
            source_names=(
                "source_name",
                lambda values: tuple(sorted(set(values.dropna().astype(str)))),
            ),
            spw_ids=(
                "spw_id",
                lambda values: tuple(sorted(set(values.dropna().astype(str)))),
            ),
            asdm_uids=(
                "asdm_uid",
                lambda values: tuple(sorted(set(values.dropna().astype(str)))),
            ),
            grammar_families=(
                "support_grammar_family",
                lambda values: tuple(sorted(set(values))),
            ),
        )
        .reset_index()
    )

    context_table = (
        work.groupby(
            [
                "member_ous_uid", "asdm_uid", "source_name",
                "frequency_support",
            ],
            dropna=False,
        )
        .agg(
            archive_rows=("obs_id", "size"),
            spw_ids=(
                "spw_id",
                lambda values: tuple(sorted(set(values.dropna().astype(str)))),
            ),
            frequencies=(
                "frequency",
                lambda values: tuple(sorted(set(values.dropna()))),
            ),
            component_count=("support_component_count", "first"),
            grammar_family=("support_grammar_family", "first"),
        )
        .reset_index()
    )

    geometry_work = pd.DataFrame(
        work["s_region"].map(parse_stcs_structure).tolist()
    )
    geometry_work = pd.concat(
        [
            work[
                ["member_ous_uid", "asdm_uid", "source_name"]
            ].reset_index(drop=True),
            geometry_work,
        ],
        axis=1,
    )
    geometry_table = (
        geometry_work.groupby(
            ["member_ous_uid", "asdm_uid", "source_name"],
            dropna=False,
        )
        .agg(
            geometry_hashes=(
                "normalized_geometry_hash",
                lambda values: tuple(sorted(set(values.dropna().astype(str)))),
            ),
            geometry_types=(
                "geometry_type",
                lambda values: tuple(sorted(set(values))),
            ),
            parse_statuses=(
                "geometry_parse_status",
                lambda values: tuple(sorted(set(values))),
            ),
        )
        .reset_index()
    )

    return {
        "row_identity": row_identity_table,
        "member": member_table,
        "context": context_table,
        "geometry": geometry_table,
    }


def brace_mapping_signature(dataframe: pd.DataFrame) -> pd.DataFrame:
    work = dataframe.copy().reset_index(drop=True)
    parsed = pd.DataFrame(
        work["obs_id"].map(parse_obs_id_structure).tolist()
    )
    for parsed_column in parsed.columns:
        work[parsed_column] = parsed[parsed_column].to_numpy()

    assert work.columns.is_unique, (
        "Brace reconstruction produced duplicate columns: "
        f"{work.columns[work.columns.duplicated()].tolist()}"
    )

    work["support_grammar_family"] = (
        work["frequency_support"].map(classify_support_grammar)
    )
    brace_rows = work.loc[
        work["support_grammar_family"].eq(
            "BRACE_CENTRE_RESOLUTION"
        )
    ].copy()

    mapping_records = []
    for _, row in brace_rows.iterrows():
        bodies = BRACE_COMPONENT_PATTERN.findall(str(row["frequency_support"]))
        component_centres = []
        component_tolerances = []

        for body in bodies:
            first_token = body.split(",", maxsplit=1)[0].strip()
            centre_ghz = parse_numeric_token(first_token, GHZ_TOKEN_PATTERN)
            decimals = decimal_places_from_token(
                first_token,
                GHZ_TOKEN_PATTERN,
            )
            component_centres.append(centre_ghz)
            component_tolerances.append(
                np.nan
                if decimals is None
                else 10 ** (-decimals) * 1e3 / 2
            )

        if not component_centres:
            mapping_records.append(
                {
                    "obs_id": row["obs_id"],
                    "component_index": None,
                    "mapping_status": "NO_COMPONENT",
                    "absolute_centre_difference_mhz": None,
                }
            )
            continue

        differences_mhz = np.abs(
            np.asarray(component_centres, dtype=float)
            - float(row["frequency"])
        ) * 1e3
        minimum_difference = float(np.nanmin(differences_mhz))
        candidate_indices = np.flatnonzero(
            np.isclose(
                differences_mhz,
                minimum_difference,
                rtol=0.0,
                atol=1e-9,
            )
        )
        chosen_index = int(candidate_indices[0])
        within_tolerance = (
            minimum_difference <= component_tolerances[chosen_index]
        )

        if len(candidate_indices) > 1:
            status = "AMBIGUOUS_EQUAL_DISTANCE"
        elif within_tolerance:
            status = "ASSIGNED_NEAREST_CENTRE"
        else:
            status = "OUTSIDE_REPRESENTATION_TOLERANCE"

        mapping_records.append(
            {
                "obs_id": row["obs_id"],
                "member_ous_uid": row["member_ous_uid"],
                "asdm_uid": row["asdm_uid"],
                "source_name": row["source_name"],
                "spw_id": row["spw_id"],
                "component_index": chosen_index,
                "mapping_status": status,
                "absolute_centre_difference_mhz": round(
                    minimum_difference,
                    12,
                ),
            }
        )

    return pd.DataFrame(mapping_records)


baseline_tables = reconstruct_order_independent_tables(
    cardinality_analysis_df
)
baseline_hashes = {
    name: canonical_dataframe_hash(table)
    for name, table in baseline_tables.items()
}
baseline_brace_mapping_hash = canonical_dataframe_hash(
    brace_mapping_signature(cardinality_analysis_df)
)

shuffle_invariance_records = []

for seed in SHUFFLE_SEEDS:
    shuffled = cardinality_analysis_df.sample(
        frac=1,
        random_state=seed,
    ).reset_index(drop=True)
    reconstructed_tables = reconstruct_order_independent_tables(shuffled)
    reconstructed_hashes = {
        name: canonical_dataframe_hash(table)
        for name, table in reconstructed_tables.items()
    }
    brace_hash = canonical_dataframe_hash(
        brace_mapping_signature(shuffled)
    )

    record = {
        "shuffle_seed": seed,
        "row_count": len(shuffled),
        "brace_mapping_matches": brace_hash == baseline_brace_mapping_hash,
    }
    for table_name, baseline_hash in baseline_hashes.items():
        record[f"{table_name}_matches"] = (
            reconstructed_hashes[table_name] == baseline_hash
        )
    record["all_reconstructions_match"] = all(
        value
        for key, value in record.items()
        if key.endswith("_matches")
    )
    shuffle_invariance_records.append(record)

shuffle_invariance_df = pd.DataFrame(shuffle_invariance_records)
display(shuffle_invariance_df)

if not shuffle_invariance_df["all_reconstructions_match"].all():
    raise AssertionError(
        "Reconstruction depends on input row order. Inspect the failed "
        "fingerprint columns before accepting the model."
    )

print("Shuffle-invariance experiment passed for all configured seeds.")


,shuffle_seed,row_count,brace_mapping_matches,row_identity_matches,member_matches,context_matches,geometry_matches,all_reconstructions_match
0,0,227,True,True,True,True,True,True
1,1,227,True,True,True,True,True,True
2,7,227,True,True,True,True,True,True
3,42,227,True,True,True,True,True,True
4,2026,227,True,True,True,True,True,True


Shuffle-invariance experiment passed for all configured seeds.


## Corrected evidence synthesis — Steps 1–10

This table integrates the original robustness evidence with Steps 5–10.
It distinguishes the original sample from the expanded cardinality sample,
separates bracket and brace mapping results, and reports limited searches
as unresolved rather than as evidence of absence.

This is the corrected pre-closure synthesis. Archive-wide brace parsing,
Archive-wide `obs_id`/candidate-key checks, improved repeated-source
discovery, full 73-field classification, and data-model v0.4 remain later
closure tasks.


In [43]:
original_complete_execution_count = int(
    execution_structure_df["complete_source_spw_grid"].sum()
)
original_execution_count = len(execution_structure_df)

expanded_structure_counts = (
    execution_cardinality_df["source_spw_structure"]
    .value_counts(dropna=False)
    .to_dict()
)
expanded_complete_execution_count = int(
    expanded_structure_counts.get("COMPLETE_CARTESIAN_GRID", 0)
)
expanded_execution_count = len(execution_cardinality_df)

grammar_counts = (
    support_grammar_census_df
    .set_index("grammar_family")["archive_row_count"]
    .astype(int)
    .to_dict()
)

initial_grammar_by_row = (
    analysis_df
    .set_index("archive_row_index")["frequency_support"]
    .map(classify_support_grammar)
)
bracket_row_indices = set(
    initial_grammar_by_row.loc[
        initial_grammar_by_row.eq("BRACKET_INTERVAL")
    ].index
)
bracket_assignment_status_counts = (
    assignment_df.loc[
        assignment_df["archive_row_index"].isin(bracket_row_indices),
        "assignment_status",
    ]
    .value_counts(dropna=False)
    .to_dict()
)

brace_mapping_status_counts = (
    brace_row_relationship_df["mapping_status"]
    .value_counts(dropna=False)
    .to_dict()
)

tolerant_interval_passes = int(
    frequency_interval_diagnostic_df[
        "row_frequency_inside_obscore_interval_tolerant"
    ].sum()
)

resolution_equal_count = int(
    np.isclose(
        analysis_df["s_resolution"],
        analysis_df["spatial_resolution"],
        rtol=0.0,
        atol=1e-12,
        equal_nan=False,
    ).sum()
)

geometry_parse_successes = int(
    geometry_analysis_df["geometry_parse_status"]
    .eq("PARSED_STRUCTURALLY")
    .sum()
)
geometry_context_variations = int(
    spatial_context_geometry_df["distinct_footprints"].gt(1).sum()
)

product_multiple_rows = int(
    source_spw_product_df["multiple_product_rows"].sum()
)

evidence_summary_df = pd.DataFrame(
    [
        {
            "model_question": "Live schema captured and fingerprinted",
            "evidence": (
                f"{len(schema_df)} columns; "
                f"SHA-256 {schema_fingerprint[:12]}..."
            ),
            "status": "COMPLETED",
        },
        {
            "model_question": "Complete selected-Member retrieval",
            "evidence": f"{len(analysis_df)} / {expected_row_count} rows",
            "status": (
                "SUPPORTED" if complete_retrieval else "CONTRADICTED"
            ),
        },
        {
            "model_question": "Observed obs_id grammar",
            "evidence": (
                f"{int(analysis_df['obs_id_parse_status'].eq('PARSED').sum())} "
                f"/ {len(analysis_df)} parsed in original sample"
            ),
            "status": "SAMPLE_SUPPORTED_ARCHIVE_WIDE_CHECK_PENDING",
        },
        {
            "model_question": "Archive-wide top-level frequency grammar",
            "evidence": {
                **grammar_counts,
                "partition_total": grammar_partition_rows,
                "science_target_total": grammar_total_science_rows,
            },
            "status": (
                "TOP_LEVEL_PARTITION_COMPLETE"
                if grammar_partition_complete
                else "INCOMPLETE_PARTITION"
            ),
        },
        {
            "model_question": "Original Source-Execution × SPW grid",
            "evidence": (
                f"{original_complete_execution_count} / "
                f"{original_execution_count} executions complete"
            ),
            "status": (
                "SAMPLE_SUPPORTED"
                if original_complete_execution_count
                == original_execution_count
                else "CONTRADICTED_AS_UNIVERSAL_CONSTRAINT"
            ),
        },
        {
            "model_question": "Expanded Source-Execution × SPW grid",
            "evidence": {
                "complete_executions": expanded_complete_execution_count,
                "total_executions": expanded_execution_count,
                "structure_counts": expanded_structure_counts,
            },
            "status": (
                "SAMPLE_SUPPORTED"
                if expanded_complete_execution_count
                == expanded_execution_count
                else "CONTRADICTED_AS_UNIVERSAL_CONSTRAINT"
            ),
        },
        {
            "model_question": "Sparse Source–SPW associations",
            "evidence": int(
                source_spw_association_df[
                    "is_sparse_source_association"
                ].sum()
            ),
            "status": (
                "OBSERVED"
                if source_spw_association_df[
                    "is_sparse_source_association"
                ].any()
                else "NOT_OBSERVED"
            ),
        },
        {
            "model_question": "Support-component cardinality",
            "evidence": (
                context_cardinality_df["cardinality_class"]
                .value_counts(dropna=False)
                .to_dict()
            ),
            "status": "MANY_TO_ONE_OBSERVED",
        },
        {
            "model_question": "Bracket support mapping",
            "evidence": bracket_assignment_status_counts,
            "status": (
                "SAMPLE_SUPPORTED"
                if set(bracket_assignment_status_counts).issubset(
                    {"ASSIGNED"}
                )
                else "REQUIRES_REVIEW"
            ),
        },
        {
            "model_question": "Brace nearest-centre mapping",
            "evidence": {
                "mapping_statuses": brace_mapping_status_counts,
                "many_to_one_components": many_to_one_component_count,
            },
            "status": (
                "SAMPLE_SUPPORTED_SEMANTICS_PARTLY_UNRESOLVED"
                if set(brace_mapping_status_counts).issubset(
                    {"ASSIGNED_NEAREST_CENTRE"}
                )
                else "REQUIRES_REVIEW"
            ),
        },
        {
            "model_question": "Footprint ownership above Source-Execution",
            "evidence": {
                "verified_repeated_source_contexts": len(
                    verified_repeated_source_df
                ),
                "search_status": repeated_source_ownership_status,
            },
            "status": "UNRESOLVED_LIMITED_DISCOVERY",
        },
        {
            "model_question": "STC-S structural parsing",
            "evidence": {
                "parsed_rows": geometry_parse_successes,
                "total_rows": len(geometry_analysis_df),
                "contexts_with_multiple_footprints": (
                    geometry_context_variations
                ),
            },
            "status": "SAMPLE_SUPPORTED",
        },
        {
            "model_question": "ObsCore product multiplicity",
            "evidence": {
                "execution_source_spw_groups_with_multiple_rows": (
                    product_multiple_rows
                ),
                "sample_rows": len(product_analysis_df),
            },
            "status": "SAMPLE_SUPPORTED_NOT_FILE_GRANULARITY_PROOF",
        },
        {
            "model_question": "em_min/em_max numerical consistency",
            "evidence": (
                f"{tolerant_interval_passes} / "
                f"{len(frequency_interval_diagnostic_df)} within interval "
                "using 1-Hz numerical tolerance"
            ),
            "status": (
                "SUPPORTED_WITH_NUMERICAL_TOLERANCE"
                if tolerant_interval_passes
                == len(frequency_interval_diagnostic_df)
                else "CONTRADICTED"
            ),
        },
        {
            "model_question": "s_resolution equals spatial_resolution",
            "evidence": (
                f"{resolution_equal_count} / {len(analysis_df)} "
                "exactly equal"
            ),
            "status": (
                "SAMPLE_SUPPORTED"
                if resolution_equal_count == len(analysis_df)
                else "CONTRADICTED"
            ),
        },
        {
            "model_question": "Shuffle invariance",
            "evidence": (
                shuffle_invariance_df
                .set_index("shuffle_seed")["all_reconstructions_match"]
                .to_dict()
            ),
            "status": (
                "SAMPLE_SUPPORTED"
                if shuffle_invariance_df[
                    "all_reconstructions_match"
                ].all()
                else "CONTRADICTED"
            ),
        },
        {
            "model_question": "MAXREC truncation detectable",
            "evidence": truncation_experiment_df.iloc[0].to_dict(),
            "status": "COMPLETED",
        },
        {
            "model_question": "Controlled anomalies reported",
            "evidence": (
                "malformed obs_id, duplicate id, missing support, "
                "blank optional group"
            ),
            "status": "COMPLETED",
        },
    ]
)

display(evidence_summary_df)


,model_question,evidence,status
0,Live schema captured and fingerprinted,73 columns; SHA-256 2cb2009067ab...,COMPLETED
1,Complete selected-Member retrieval,187 / 187 rows,SUPPORTED
2,Observed obs_id grammar,187 / 187 parsed in original sample,SAMPLE_SUPPORTED_ARCHIVE_WIDE_CHECK_PENDING
3,Archive-wide top-level frequency grammar,"{'BRACKET_INTERVAL': 442452, 'BRACE_CENTRE_RESOLUTION': 55, 'MISSING': 0, 'BLANK_EXACT': 0, 'UNKNOWN_NONBLANK': 0, '...",TOP_LEVEL_PARTITION_COMPLETE
4,Original Source-Execution × SPW grid,33 / 34 executions complete,CONTRADICTED_AS_UNIVERSAL_CONSTRAINT
5,Expanded Source-Execution × SPW grid,"{'complete_executions': 39, 'total_executions': 40, 'structure_counts': {'COMPLETE_CARTESIAN_GRID': 39, 'SPARSE_SOUR...",CONTRADICTED_AS_UNIVERSAL_CONSTRAINT
6,Sparse Source–SPW associations,1,OBSERVED
7,Support-component cardinality,"{'COUNT_EQUAL_ONE_TO_ONE_CANDIDATE': 48, 'MORE_ROWS_THAN_COMPONENTS_MANY_TO_ONE_CANDIDATE': 1}",MANY_TO_ONE_OBSERVED
8,Bracket support mapping,{'ASSIGNED': 176},SAMPLE_SUPPORTED
9,Brace nearest-centre mapping,"{'mapping_statuses': {'ASSIGNED_NEAREST_CENTRE': 11}, 'many_to_one_components': 3}",SAMPLE_SUPPORTED_SEMANTICS_PARTLY_UNRESOLVED


# Supplementary closure experiments

These experiments run after the corrected Steps 1–10. They address only
model decisions that remain unresolved:

1. complete census of all brace-style `frequency_support` rows;
2. Archive-wide `obs_id` grammar and candidate-key census;
3. unbiased repeated-source-across-ASDM discovery;
4. Archive-wide STC-S family coverage;
5. Archive-wide product metadata and resolution-mismatch diagnostics;
6. classification of all 73 live schema fields;
7. final query provenance and a model-closure gate.

Every large retrieval is preceded by `COUNT(*)`, bounded by an explicit
safety limit, and rejected if the returned row count is incomplete or TAP
reports `OVERFLOW`.


## Step 11 — Complete 55-row brace census

**Question.** Does every current brace row follow the same component
grammar, and can every row be mapped to a component without assuming
one-to-one cardinality?

**Process.**

1. Count every science-target row beginning with `{`.
2. Retrieve exactly that many rows with product, spectral, sensitivity,
   antenna, and geometry metadata.
3. Parse every complete support signature and preserve raw tokens.
4. Reconstruct row-to-component mappings using displayed centre precision.
5. Compare token 2 separately with spectral resolution, bandwidth, and
   bandwidth per spectral element.
6. Report grammar, mapping, and semantic exceptions without discarding raw
   rows.


In [44]:
BRACE_CENSUS_SAFETY_LIMIT = 500

brace_census_count_query = """
SELECT COUNT(*) AS total_rows
FROM ivoa.obscore
WHERE science_observation = 'T'
  AND frequency_support LIKE '{%'
"""

brace_census_count_run = run_tap_query(
    brace_census_count_query,
    maxrec=10,
    label="Count complete brace-style frequency-support population",
)
query_runs.append(brace_census_count_run)
brace_census_expected_rows = int(
    brace_census_count_run.table[0]["total_rows"]
)

grammar_census_brace_rows = int(
    support_grammar_census_df.loc[
        support_grammar_census_df["grammar_family"].eq(
            "BRACE_CENTRE_RESOLUTION"
        ),
        "archive_row_count",
    ].iloc[0]
)

if brace_census_expected_rows != grammar_census_brace_rows:
    raise RuntimeError(
        "Brace COUNT changed within the same run: "
        f"{brace_census_expected_rows} vs {grammar_census_brace_rows}."
    )
if brace_census_expected_rows > BRACE_CENSUS_SAFETY_LIMIT:
    raise RuntimeError(
        f"Brace census expects {brace_census_expected_rows} rows, above "
        f"safety limit {BRACE_CENSUS_SAFETY_LIMIT}."
    )

brace_census_columns = [
    name
    for name in [
        "proposal_id", "group_ous_uid", "member_ous_uid", "asdm_uid",
        "obs_id", "target_name", "frequency_support", "frequency",
        "bandwidth", "spectral_resolution", "em_xel", "pol_xel",
        "pol_states", "sensitivity_10kms",
        "cont_sensitivity_bandwidth", "antenna_arrays", "band_list",
        "dataproduct_type", "calib_level", "type", "is_mosaic",
        "s_ra", "s_dec", "s_region", "s_resolution",
        "spatial_resolution", "t_min", "t_max",
    ]
    if name in available_columns
]
brace_census_column_sql = ",\n    ".join(brace_census_columns)

brace_census_query = f"""
SELECT
    {brace_census_column_sql}
FROM ivoa.obscore
WHERE science_observation = 'T'
  AND frequency_support LIKE '{{%'
"""

brace_census_run = run_tap_query(
    brace_census_query,
    maxrec=max(brace_census_expected_rows, 1),
    label="Retrieve complete brace-style frequency-support population",
)
query_runs.append(brace_census_run)
brace_census_df = brace_census_run.table.to_pandas()
brace_census_df.insert(
    0,
    "brace_row_index",
    np.arange(len(brace_census_df), dtype=int),
)

brace_census_overflow = "OVERFLOW" in " ".join(
    brace_census_run.query_status
)
if brace_census_overflow or len(brace_census_df) != brace_census_expected_rows:
    raise RuntimeError(
        "Brace census retrieval is incomplete: "
        f"{len(brace_census_df)} / {brace_census_expected_rows}; "
        f"status={brace_census_run.query_status}."
    )

brace_obs_parsed_df = pd.DataFrame(
    brace_census_df["obs_id"].map(parse_obs_id_structure).tolist()
)
brace_analysis_df = brace_census_df.copy().reset_index(drop=True)
for parsed_column in brace_obs_parsed_df.columns:
    brace_analysis_df[parsed_column] = (
        brace_obs_parsed_df[parsed_column].to_numpy()
    )
brace_analysis_df["support_grammar_family"] = (
    brace_analysis_df["frequency_support"].map(classify_support_grammar)
)

assert brace_analysis_df.columns.is_unique

display(
    pd.DataFrame(
        [
            {
                "expected_rows": brace_census_expected_rows,
                "retrieved_rows": len(brace_analysis_df),
                "member_count": brace_analysis_df[
                    "member_ous_uid"
                ].nunique(),
                "execution_count": brace_analysis_df[
                    ["member_ous_uid", "asdm_uid"]
                ].drop_duplicates().shape[0],
                "source_execution_count": brace_analysis_df[
                    ["member_ous_uid", "asdm_uid", "source_name"]
                ].drop_duplicates().shape[0],
                "obs_id_parse_failures": int(
                    (~brace_analysis_df["obs_id_parse_status"].eq("PARSED"))
                    .sum()
                ),
                "tap_overflow": brace_census_overflow,
            }
        ]
    )
)



--- Count complete brace-style frequency-support population ---
Endpoint: https://almascience.eso.org/tap
MAXREC: 10
ADQL:

SELECT COUNT(*) AS total_rows
FROM ivoa.obscore
WHERE science_observation = 'T'
  AND frequency_support LIKE '{%'

Retrieved rows: 1
QUERY_STATUS: ('OK',)
Warnings: <none>

--- Retrieve complete brace-style frequency-support population ---
Endpoint: https://almascience.eso.org/tap
MAXREC: 55
ADQL:

SELECT
    proposal_id,
    group_ous_uid,
    member_ous_uid,
    asdm_uid,
    obs_id,
    target_name,
    frequency_support,
    frequency,
    bandwidth,
    spectral_resolution,
    em_xel,
    pol_xel,
    pol_states,
    sensitivity_10kms,
    cont_sensitivity_bandwidth,
    antenna_arrays,
    band_list,
    dataproduct_type,
    calib_level,
    type,
    is_mosaic,
    s_ra,
    s_dec,
    s_region,
    s_resolution,
    spatial_resolution,
    t_min,
    t_max
FROM ivoa.obscore
WHERE science_observation = 'T'
  AND frequency_support LIKE '{%'

Retrieved row

,expected_rows,retrieved_rows,member_count,execution_count,source_execution_count,obs_id_parse_failures,tap_overflow
0,55,55,7,7,13,0,False


In [45]:
def parse_complete_brace_signature(
    raw_support: object,
) -> tuple[dict[str, object], list[dict[str, object]]]:
    raw_text = "" if pd.isna(raw_support) else str(raw_support).strip()
    matches = list(BRACE_COMPONENT_PATTERN.finditer(raw_text))

    prefix = raw_text[: matches[0].start()].strip() if matches else raw_text
    suffix = raw_text[matches[-1].end() :].strip() if matches else raw_text
    connectors = []
    for left, right in zip(matches, matches[1:]):
        connectors.append(" ".join(raw_text[left.end() : right.start()].split()))

    component_records = []
    for component_index, match in enumerate(matches):
        raw_component = match.group(0)
        tokens = [token.strip() for token in match.group("body").split(",")]
        token_1_raw = tokens[0] if len(tokens) > 0 else None
        token_2_raw = tokens[1] if len(tokens) > 1 else None
        token_1_ghz = parse_numeric_token(
            token_1_raw or "", GHZ_TOKEN_PATTERN
        )
        token_2_khz = parse_numeric_token(
            token_2_raw or "", KHZ_TOKEN_PATTERN
        )
        decimal_places = decimal_places_from_token(
            token_1_raw or "", GHZ_TOKEN_PATTERN
        )
        tolerance_mhz = (
            np.nan
            if decimal_places is None
            else 10 ** (-decimal_places) * 1e3 / 2
        )
        issues = []
        if len(tokens) != 5:
            issues.append("unexpected_token_count")
        if pd.isna(token_1_ghz):
            issues.append("invalid_centre_frequency_token")
        if pd.isna(token_2_khz):
            issues.append("invalid_spectral_width_token")

        component_records.append(
            {
                "component_index": component_index,
                "raw_component_text": raw_component,
                "token_count": len(tokens),
                "token_1_raw": token_1_raw,
                "token_1_ghz": token_1_ghz,
                "token_1_decimal_places": decimal_places,
                "token_1_representation_tolerance_mhz": tolerance_mhz,
                "token_2_raw": token_2_raw,
                "token_2_khz": token_2_khz,
                "token_3_raw": tokens[2] if len(tokens) > 2 else None,
                "token_4_raw": tokens[3] if len(tokens) > 3 else None,
                "token_5_raw": tokens[4] if len(tokens) > 4 else None,
                "component_issue_codes": tuple(issues),
                "component_parse_status": (
                    "PARSED_STRUCTURALLY" if not issues else "PARTIAL_OR_INVALID"
                ),
            }
        )

    signature_issues = []
    if not matches:
        signature_issues.append("no_brace_components")
    if prefix:
        signature_issues.append("unexpected_prefix")
    if suffix:
        signature_issues.append("unexpected_suffix")
    if any(connector.upper() != "U" for connector in connectors):
        signature_issues.append("unexpected_component_connector")

    signature_record = {
        "component_count": len(matches),
        "connector_count": len(connectors),
        "connector_tokens": tuple(connectors),
        "prefix_text": prefix,
        "suffix_text": suffix,
        "signature_issue_codes": tuple(signature_issues),
        "signature_parse_status": (
            "PARSED_STRUCTURALLY"
            if not signature_issues
            else "PARTIAL_OR_INVALID"
        ),
    }
    return signature_record, component_records


brace_context_keys = [
    "member_ous_uid", "asdm_uid", "source_name", "frequency_support"
]
brace_signature_inventory_df = (
    brace_analysis_df[brace_context_keys].drop_duplicates().copy()
)

brace_signature_records = []
brace_full_component_records = []
for signature_row in brace_signature_inventory_df.itertuples(index=False):
    signature_record, component_records = parse_complete_brace_signature(
        signature_row.frequency_support
    )
    context = {
        key: getattr(signature_row, key) for key in brace_context_keys
    }
    brace_signature_records.append({**context, **signature_record})
    for component_record in component_records:
        brace_full_component_records.append(
            {**context, **component_record}
        )

brace_signature_census_df = pd.DataFrame(brace_signature_records)
brace_full_component_df = pd.DataFrame(brace_full_component_records)

brace_context_cardinality_df = (
    brace_analysis_df
    .groupby(brace_context_keys, dropna=False)
    .agg(
        archive_row_count=("brace_row_index", "size"),
        parsed_spw_count=("spw_id", "nunique"),
        unique_frequency_count=("frequency", "nunique"),
    )
    .reset_index()
    .merge(
        brace_signature_census_df[
            brace_context_keys
            + ["component_count", "signature_parse_status"]
        ],
        on=brace_context_keys,
        how="left",
        validate="one_to_one",
    )
)
brace_context_cardinality_df["cardinality_class"] = np.select(
    [
        brace_context_cardinality_df["archive_row_count"].eq(
            brace_context_cardinality_df["component_count"]
        ),
        brace_context_cardinality_df["archive_row_count"].gt(
            brace_context_cardinality_df["component_count"]
        ),
        brace_context_cardinality_df["archive_row_count"].lt(
            brace_context_cardinality_df["component_count"]
        ),
    ],
    [
        "COUNT_EQUAL_ONE_TO_ONE_CANDIDATE",
        "MORE_ROWS_THAN_COMPONENTS",
        "MORE_COMPONENTS_THAN_ROWS",
    ],
    default="UNRESOLVED",
)

display(
    brace_signature_census_df
    .groupby(
        [
            "signature_parse_status", "component_count",
            "connector_tokens",
        ],
        dropna=False,
    )
    .size()
    .rename("signature_count")
    .reset_index()
)
display(
    brace_full_component_df
    .groupby(
        ["component_parse_status", "token_count"],
        dropna=False,
    )
    .size()
    .rename("component_count")
    .reset_index()
)
display(
    brace_context_cardinality_df["cardinality_class"]
    .value_counts(dropna=False)
    .rename_axis("cardinality_class")
    .reset_index(name="context_count")
)

brace_grammar_exception_df = brace_signature_census_df.loc[
    ~brace_signature_census_df["signature_parse_status"].eq(
        "PARSED_STRUCTURALLY"
    )
].copy()
brace_component_exception_df = brace_full_component_df.loc[
    ~brace_full_component_df["component_parse_status"].eq(
        "PARSED_STRUCTURALLY"
    )
].copy()
display(brace_grammar_exception_df)
display(brace_component_exception_df)


,signature_parse_status,component_count,connector_tokens,signature_count
0,PARSED_STRUCTURALLY,4,"(U, U, U)",13


,component_parse_status,token_count,component_count
0,PARSED_STRUCTURALLY,5,52


,cardinality_class,context_count
0,COUNT_EQUAL_ONE_TO_ONE_CANDIDATE,12
1,MORE_ROWS_THAN_COMPONENTS,1


,member_ous_uid,asdm_uid,source_name,frequency_support,component_count,connector_count,connector_tokens,prefix_text,suffix_text,signature_issue_codes,signature_parse_status


,member_ous_uid,asdm_uid,source_name,frequency_support,component_index,raw_component_text,token_count,token_1_raw,token_1_ghz,token_1_decimal_places,token_1_representation_tolerance_mhz,token_2_raw,token_2_khz,token_3_raw,token_4_raw,token_5_raw,component_issue_codes,component_parse_status


In [46]:
def rows_for_exact_context(
    dataframe: pd.DataFrame,
    keys: list[str],
    values: tuple[object, ...],
) -> pd.DataFrame:
    mask = pd.Series(True, index=dataframe.index)
    for key, value in zip(keys, values, strict=True):
        mask &= dataframe[key].isna() if pd.isna(value) else dataframe[key].eq(value)
    return dataframe.loc[mask].copy()


brace_full_mapping_records = []
for context_values, row_group in brace_analysis_df.groupby(
    brace_context_keys,
    dropna=False,
    sort=False,
):
    if not isinstance(context_values, tuple):
        context_values = (context_values,)
    component_group = rows_for_exact_context(
        brace_full_component_df,
        brace_context_keys,
        context_values,
    )

    for _, archive_row in row_group.iterrows():
        base_record = archive_row.to_dict()
        usable = component_group.dropna(subset=["token_1_ghz"]).copy()
        if usable.empty or pd.isna(archive_row["frequency"]):
            brace_full_mapping_records.append(
                {
                    **base_record,
                    "component_index": np.nan,
                    "mapping_status": "NO_USABLE_COMPONENT",
                    "nearest_candidate_count": 0,
                }
            )
            continue

        usable["absolute_centre_difference_mhz"] = (
            usable["token_1_ghz"]
            .sub(float(archive_row["frequency"]))
            .abs()
            .mul(1e3)
        )
        minimum_difference = float(
            usable["absolute_centre_difference_mhz"].min()
        )
        nearest = usable.loc[
            np.isclose(
                usable["absolute_centre_difference_mhz"],
                minimum_difference,
                rtol=0.0,
                atol=1e-9,
            )
        ].sort_values("component_index", kind="stable")
        chosen = nearest.iloc[0]
        tolerance = float(
            chosen["token_1_representation_tolerance_mhz"]
        )
        within_tolerance = minimum_difference <= tolerance
        if len(nearest) > 1:
            status = "AMBIGUOUS_EQUAL_DISTANCE"
        elif within_tolerance:
            status = "ASSIGNED_NEAREST_CENTRE"
        else:
            status = "OUTSIDE_REPRESENTATION_TOLERANCE"

        brace_full_mapping_records.append(
            {
                **base_record,
                "component_index": int(chosen["component_index"]),
                "mapping_status": status,
                "nearest_candidate_count": len(nearest),
                "token_1_ghz": float(chosen["token_1_ghz"]),
                "token_1_representation_tolerance_mhz": tolerance,
                "token_2_khz": float(chosen["token_2_khz"]),
                "token_3_raw": chosen["token_3_raw"],
                "token_4_raw": chosen["token_4_raw"],
                "token_5_raw": chosen["token_5_raw"],
                "absolute_centre_difference_mhz": minimum_difference,
                "within_representation_tolerance": within_tolerance,
            }
        )

brace_full_mapping_df = pd.DataFrame(brace_full_mapping_records)

brace_full_component_usage_df = (
    brace_full_mapping_df.loc[
        brace_full_mapping_df["component_index"].notna()
    ]
    .groupby(
        brace_context_keys + ["component_index", "token_1_ghz"],
        dropna=False,
    )
    .agg(
        mapped_archive_rows=("brace_row_index", "size"),
        mapped_spw_ids=(
            "spw_id",
            lambda values: tuple(sorted(set(values.dropna().astype(str)))),
        ),
        maximum_centre_difference_mhz=(
            "absolute_centre_difference_mhz", "max"
        ),
        mapping_statuses=(
            "mapping_status",
            lambda values: tuple(sorted(set(values))),
        ),
    )
    .reset_index()
)

display(
    brace_full_mapping_df["mapping_status"]
    .value_counts(dropna=False)
    .rename_axis("mapping_status")
    .reset_index(name="row_count")
)
display(
    brace_full_component_usage_df.loc[
        brace_full_component_usage_df["mapped_archive_rows"].gt(1)
    ]
)
display(
    brace_full_mapping_df.loc[
        ~brace_full_mapping_df["mapping_status"].eq(
            "ASSIGNED_NEAREST_CENTRE"
        )
    ]
)


,mapping_status,row_count
0,ASSIGNED_NEAREST_CENTRE,55


,member_ous_uid,asdm_uid,source_name,frequency_support,component_index,token_1_ghz,mapped_archive_rows,mapped_spw_ids,maximum_centre_difference_mhz,mapping_statuses
44,uid://A001/X3955/X44,uid://A002/X13e7f29/Xc580,Moon_full,"{229.98GHz,2000000.00kHz,20.8mJy/beam@10km/s,1.3mJy/beam@native, XX YY} U {231.98GHz,2000000.00kHz,21.1mJy/beam@10km...",0,229.98,2,"(0, 4)",1.374189,"(ASSIGNED_NEAREST_CENTRE,)"
45,uid://A001/X3955/X44,uid://A002/X13e7f29/Xc580,Moon_full,"{229.98GHz,2000000.00kHz,20.8mJy/beam@10km/s,1.3mJy/beam@native, XX YY} U {231.98GHz,2000000.00kHz,21.1mJy/beam@10km...",1,231.98,2,"(1, 5)",1.212225,"(ASSIGNED_NEAREST_CENTRE,)"
47,uid://A001/X3955/X44,uid://A002/X13e7f29/Xc580,Moon_full,"{229.98GHz,2000000.00kHz,20.8mJy/beam@10km/s,1.3mJy/beam@native, XX YY} U {231.98GHz,2000000.00kHz,21.1mJy/beam@10km...",3,247.98,2,"(3, 6)",0.083484,"(ASSIGNED_NEAREST_CENTRE,)"


,brace_row_index,proposal_id,group_ous_uid,member_ous_uid,asdm_uid,obs_id,target_name,frequency_support,frequency,bandwidth,spectral_resolution,em_xel,pol_xel,pol_states,sensitivity_10kms,cont_sensitivity_bandwidth,antenna_arrays,band_list,dataproduct_type,calib_level,type,is_mosaic,s_ra,s_dec,s_region,s_resolution,spatial_resolution,t_min,t_max,obs_member_ous_uid,source_name,spw_id,obs_id_parse_status,obs_id_parse_issue,support_grammar_family,component_index,mapping_status,nearest_candidate_count,token_1_ghz,token_1_representation_tolerance_mhz,token_2_khz,token_3_raw,token_4_raw,token_5_raw,absolute_centre_difference_mhz,within_representation_tolerance


In [47]:
brace_full_mapping_df["token_2_hz"] = (
    brace_full_mapping_df["token_2_khz"] * 1e3
)
brace_full_mapping_df["bandwidth_per_em_xel_hz"] = (
    brace_full_mapping_df["bandwidth"]
    / pd.to_numeric(brace_full_mapping_df["em_xel"], errors="coerce")
    .replace(0, np.nan)
)
brace_full_mapping_df["token2_equals_spectral_resolution"] = np.isclose(
    brace_full_mapping_df["token_2_khz"],
    brace_full_mapping_df["spectral_resolution"],
    rtol=0.0,
    atol=1e-6,
    equal_nan=False,
)
brace_full_mapping_df["token2_hz_equals_bandwidth"] = np.isclose(
    brace_full_mapping_df["token_2_hz"],
    brace_full_mapping_df["bandwidth"],
    rtol=0.0,
    atol=1.0,
    equal_nan=False,
)
brace_full_mapping_df[
    "token2_hz_equals_bandwidth_per_em_xel"
] = np.isclose(
    brace_full_mapping_df["token_2_hz"],
    brace_full_mapping_df["bandwidth_per_em_xel_hz"],
    rtol=0.0,
    atol=1.0,
    equal_nan=False,
)

brace_token2_summary_df = (
    brace_full_mapping_df
    .groupby(
        ["dataproduct_type", "em_xel"],
        dropna=False,
    )
    .agg(
        archive_rows=("brace_row_index", "size"),
        token2_matches_spectral_resolution=(
            "token2_equals_spectral_resolution", "sum"
        ),
        token2_matches_bandwidth=(
            "token2_hz_equals_bandwidth", "sum"
        ),
        token2_matches_bandwidth_per_em_xel=(
            "token2_hz_equals_bandwidth_per_em_xel", "sum"
        ),
        token2_values_khz=(
            "token_2_khz",
            lambda values: tuple(sorted(set(values.dropna()))),
        ),
    )
    .reset_index()
)

token2_discriminating_mask = ~np.isclose(
    brace_full_mapping_df["spectral_resolution"] * 1e3,
    brace_full_mapping_df["bandwidth"],
    rtol=0.0,
    atol=1.0,
    equal_nan=False,
)
token2_discriminating_rows = int(token2_discriminating_mask.sum())
brace_token2_semantic_status = (
    "DISCRIMINATING_ROWS_AVAILABLE"
    if token2_discriminating_rows
    else "AMBIGUOUS_BANDWIDTH_VS_RESOLUTION_NUMERICAL_DEGENERACY"
)

brace_census_decision_df = pd.DataFrame(
    [
        {
            "check": "Complete brace retrieval",
            "passed": len(brace_analysis_df) == brace_census_expected_rows,
            "details": f"{len(brace_analysis_df)} / {brace_census_expected_rows}",
        },
        {
            "check": "All signatures structurally parsed",
            "passed": brace_grammar_exception_df.empty,
            "details": f"exceptions={len(brace_grammar_exception_df)}",
        },
        {
            "check": "All components structurally parsed",
            "passed": brace_component_exception_df.empty,
            "details": f"exceptions={len(brace_component_exception_df)}",
        },
        {
            "check": "All rows mapped unambiguously within tolerance",
            "passed": brace_full_mapping_df["mapping_status"].eq(
                "ASSIGNED_NEAREST_CENTRE"
            ).all(),
            "details": brace_full_mapping_df["mapping_status"]
            .value_counts(dropna=False)
            .to_dict(),
        },
        {
            "check": "Token-2 semantics independently discriminated",
            "passed": token2_discriminating_rows > 0,
            "details": brace_token2_semantic_status,
        },
    ]
)

display(brace_token2_summary_df)
display(
    brace_full_mapping_df.loc[
        token2_discriminating_mask,
        [
            "member_ous_uid", "asdm_uid", "source_name", "spw_id",
            "em_xel", "bandwidth", "spectral_resolution",
            "token_2_khz", "token2_equals_spectral_resolution",
            "token2_hz_equals_bandwidth",
            "token2_hz_equals_bandwidth_per_em_xel",
        ],
    ]
)
display(brace_census_decision_df)
print("Brace token-2 semantic status:", brace_token2_semantic_status)


,dataproduct_type,em_xel,archive_rows,token2_matches_spectral_resolution,token2_matches_bandwidth,token2_matches_bandwidth_per_em_xel,token2_values_khz
0,image,1,55,55,55,55,"(2000000.0,)"


,member_ous_uid,asdm_uid,source_name,spw_id,em_xel,bandwidth,spectral_resolution,token_2_khz,token2_equals_spectral_resolution,token2_hz_equals_bandwidth,token2_hz_equals_bandwidth_per_em_xel


,check,passed,details
0,Complete brace retrieval,True,55 / 55
1,All signatures structurally parsed,True,exceptions=0
2,All components structurally parsed,True,exceptions=0
3,All rows mapped unambiguously within tolerance,True,{'ASSIGNED_NEAREST_CENTRE': 55}
4,Token-2 semantics independently discriminated,False,AMBIGUOUS_BANDWIDTH_VS_RESOLUTION_NUMERICAL_DEGENERACY


Brace token-2 semantic status: AMBIGUOUS_BANDWIDTH_VS_RESOLUTION_NUMERICAL_DEGENERACY


## Step 12 — Archive-wide identifier, truncation, and key census

**Questions.**

1. How much of the current science-target Archive follows the observed
   `.source.<label>.spw.<token>` `obs_id` grammar?
2. Are failures a new grammar or 64-character truncation?
3. Are `obs_id`, the Source-Execution-SPW composite, or
   `obs_publisher_did` Archive-wide candidate keys?

The Archive is partitioned into proposal-year strata plus a residual
`OTHER` stratum. Every stratum is counted before retrieval. The single
identifier retrieval includes `obs_publisher_did`, so the publisher-ID
census does not repeat the 442k-row download.


In [48]:
OBS_ID_TOTAL_SAFETY_LIMIT = 500_000
OBS_ID_STRATUM_SAFETY_LIMIT = 100_000
OBS_ID_EXCEPTION_EXAMPLE_LIMIT = 100
OBS_ID_DECLARED_WIDTH = 64


def provenance_only_tap_run(run: TapQueryRun) -> TapQueryRun:
    """Keep query evidence without retaining another large TAP table."""
    return TapQueryRun(
        label=run.label,
        endpoint=run.endpoint,
        adql=run.adql,
        maxrec=run.maxrec,
        started_at_utc=run.started_at_utc,
        finished_at_utc=run.finished_at_utc,
        retrieved_rows=run.retrieved_rows,
        query_status=run.query_status,
        warning_messages=run.warning_messages,
        result=None,
        table=None,
    )


def astropy_table_to_pandas(table: object) -> pd.DataFrame:
    """Use Astropy's mask-aware conversion at one named boundary."""
    return table.to_pandas()


obs_id_total_count_query = """
SELECT COUNT(*) AS total_rows
FROM ivoa.obscore
WHERE science_observation = 'T'
"""
obs_id_total_count_run = run_tap_query(
    obs_id_total_count_query,
    maxrec=10,
    label="Count Archive-wide identifier census population",
)
query_runs.append(obs_id_total_count_run)
obs_id_expected_total = int(
    obs_id_total_count_run.table[0]["total_rows"]
)
if obs_id_expected_total > OBS_ID_TOTAL_SAFETY_LIMIT:
    raise RuntimeError(
        f"Identifier population {obs_id_expected_total} exceeds "
        f"safety limit {OBS_ID_TOTAL_SAFETY_LIMIT}."
    )

OBS_ID_YEAR_PREFIXES = [f"{year}." for year in range(2011, 2027)]
obs_id_year_conditions = {
    prefix[:-1]: f"proposal_id LIKE {quote_adql_string(prefix + '%')}"
    for prefix in OBS_ID_YEAR_PREFIXES
}
other_condition = "proposal_id IS NULL OR (" + " AND ".join(
    f"proposal_id NOT LIKE {quote_adql_string(prefix + '%')}"
    for prefix in OBS_ID_YEAR_PREFIXES
) + ")"
OBS_ID_STRATA = {
    **obs_id_year_conditions,
    "OTHER": other_condition,
}

obs_id_stratum_count_records = []
obs_id_stratum_frames = []

for stratum, condition in OBS_ID_STRATA.items():
    count_query = f"""
    SELECT COUNT(*) AS total_rows
    FROM ivoa.obscore
    WHERE science_observation = 'T'
      AND ({condition})
    """
    count_run = run_tap_query(
        count_query,
        maxrec=10,
        label=f"Count identifier census stratum {stratum}",
    )
    query_runs.append(count_run)
    expected_rows = int(count_run.table[0]["total_rows"])
    obs_id_stratum_count_records.append(
        {"stratum": stratum, "expected_rows": expected_rows}
    )

    if expected_rows == 0:
        continue
    if expected_rows > OBS_ID_STRATUM_SAFETY_LIMIT:
        raise RuntimeError(
            f"Identifier stratum {stratum} has {expected_rows} rows, "
            f"above safety limit {OBS_ID_STRATUM_SAFETY_LIMIT}."
        )

    retrieval_query = f"""
    SELECT
        proposal_id,
        obs_publisher_did,
        member_ous_uid,
        asdm_uid,
        obs_id
    FROM ivoa.obscore
    WHERE science_observation = 'T'
      AND ({condition})
    """
    retrieval_run = run_tap_query(
        retrieval_query,
        maxrec=expected_rows,
        label=f"Retrieve complete identifier stratum {stratum}",
    )
    stratum_frame = astropy_table_to_pandas(retrieval_run.table)
    query_runs.append(provenance_only_tap_run(retrieval_run))

    if (
        "OVERFLOW" in " ".join(retrieval_run.query_status)
        or len(stratum_frame) != expected_rows
    ):
        raise RuntimeError(
            f"Identifier stratum {stratum} incomplete: "
            f"expected {expected_rows}, retrieved {len(stratum_frame)}."
        )

    stratum_frame["obs_id_census_stratum"] = stratum
    obs_id_stratum_frames.append(stratum_frame)
    del retrieval_run

obs_id_stratum_count_df = pd.DataFrame(
    obs_id_stratum_count_records
)
obs_id_partition_rows = int(
    obs_id_stratum_count_df["expected_rows"].sum()
)
if obs_id_partition_rows != obs_id_expected_total:
    raise AssertionError(
        "Identifier strata do not partition the Archive population: "
        f"{obs_id_partition_rows} != {obs_id_expected_total}."
    )

obs_id_archive_df = pd.concat(
    obs_id_stratum_frames,
    ignore_index=True,
)
obs_id_census_coverage_complete = (
    len(obs_id_archive_df) == obs_id_expected_total
)
if not obs_id_census_coverage_complete:
    raise AssertionError(
        "Archive-wide identifier retrieval is incomplete: "
        f"{len(obs_id_archive_df)} != {obs_id_expected_total}."
    )

display(obs_id_stratum_count_df)
print("Complete identifier rows retrieved:", len(obs_id_archive_df))



--- Count Archive-wide identifier census population ---
Endpoint: https://almascience.eso.org/tap
MAXREC: 10
ADQL:

SELECT COUNT(*) AS total_rows
FROM ivoa.obscore
WHERE science_observation = 'T'

Retrieved rows: 1
QUERY_STATUS: ('OK',)
Warnings: <none>

--- Count identifier census stratum 2011 ---
Endpoint: https://almascience.eso.org/tap
MAXREC: 10
ADQL:

    SELECT COUNT(*) AS total_rows
    FROM ivoa.obscore
    WHERE science_observation = 'T'
      AND (proposal_id LIKE '2011.%')
    
Retrieved rows: 1
QUERY_STATUS: ('OK',)
Warnings: <none>

--- Retrieve complete identifier stratum 2011 ---
Endpoint: https://almascience.eso.org/tap
MAXREC: 3435
ADQL:

    SELECT
        proposal_id,
        obs_publisher_did,
        member_ous_uid,
        asdm_uid,
        obs_id
    FROM ivoa.obscore
    WHERE science_observation = 'T'
      AND (proposal_id LIKE '2011.%')
    
Retrieved rows: 3435
QUERY_STATUS: ('OK',)
Warnings: <none>

--- Count identifier census stratum 2012 ---
Endpoint: h

,stratum,expected_rows
0,2011,3435
1,2012,5347
2,2013,16802
3,2014,0
4,2015,32123
5,2016,26243
6,2017,26059
7,2018,44258
8,2019,53214
9,2020,0


Complete identifier rows retrieved: 442507


In [49]:
parsed_obs_id_archive_df = pd.DataFrame(
    obs_id_archive_df["obs_id"]
    .map(parse_obs_id_structure)
    .tolist()
)
obs_id_archive_analysis_df = pd.concat(
    [
        obs_id_archive_df.reset_index(drop=True),
        parsed_obs_id_archive_df,
    ],
    axis=1,
)
obs_id_archive_analysis_df.insert(
    0,
    "obs_id_archive_row_index",
    np.arange(len(obs_id_archive_analysis_df)),
)

parsed_obs_id_mask = obs_id_archive_analysis_df[
    "obs_id_parse_status"
].eq("PARSED")
obs_id_archive_analysis_df["obs_member_matches_column"] = (
    parsed_obs_id_mask
    & obs_id_archive_analysis_df["obs_member_ous_uid"].eq(
        obs_id_archive_analysis_df["member_ous_uid"].astype(str)
    )
)
obs_id_archive_analysis_df["source_contains_whitespace"] = (
    parsed_obs_id_mask
    & obs_id_archive_analysis_df["source_name"]
    .astype("string")
    .str.contains(r"\s", regex=True, na=False)
)
obs_id_archive_analysis_df["source_contains_dot"] = (
    parsed_obs_id_mask
    & obs_id_archive_analysis_df["source_name"]
    .astype("string")
    .str.contains(".", regex=False, na=False)
)
obs_id_archive_analysis_df["spw_is_integer_token"] = (
    parsed_obs_id_mask
    & obs_id_archive_analysis_df["spw_id"]
    .astype("string")
    .str.fullmatch(r"\d+", na=False)
)

member_mismatch_mask = (
    parsed_obs_id_mask
    & ~obs_id_archive_analysis_df[
        "obs_member_matches_column"
    ]
)
non_integer_spw_mask = (
    parsed_obs_id_mask
    & ~obs_id_archive_analysis_df["spw_is_integer_token"]
)

obs_id_nonblank_mask = (
    obs_id_archive_analysis_df["obs_id"].notna()
    & obs_id_archive_analysis_df["obs_id"]
    .astype("string")
    .str.strip()
    .ne("")
    .fillna(False)
)
duplicate_obs_id_df = obs_id_archive_analysis_df.loc[
    obs_id_nonblank_mask
    & obs_id_archive_analysis_df.duplicated(
        "obs_id",
        keep=False,
    )
].sort_values("obs_id", kind="stable")

archive_candidate_key_columns = [
    "member_ous_uid", "asdm_uid", "source_name", "spw_id"
]
archive_candidate_key_eligible_df = (
    obs_id_archive_analysis_df.loc[parsed_obs_id_mask].copy()
)
duplicate_source_execution_spw_df = (
    archive_candidate_key_eligible_df.loc[
        archive_candidate_key_eligible_df.duplicated(
            archive_candidate_key_columns,
            keep=False,
        )
    ]
    .sort_values(archive_candidate_key_columns, kind="stable")
)

obs_id_parse_exception_mask = (
    ~parsed_obs_id_mask
    | member_mismatch_mask
    | non_integer_spw_mask
)
obs_id_archive_exception_df = (
    obs_id_archive_analysis_df.loc[
        obs_id_parse_exception_mask
    ].copy()
)
obs_id_boundary_case_df = obs_id_archive_analysis_df.loc[
    obs_id_archive_analysis_df["source_contains_whitespace"]
    | obs_id_archive_analysis_df["source_contains_dot"]
].copy()

obs_id_grammar_summary_df = pd.DataFrame(
    [
        {
            "total_rows": len(obs_id_archive_analysis_df),
            "null_obs_id": int(
                obs_id_archive_analysis_df["obs_id"].isna().sum()
            ),
            "blank_obs_id": int(
                obs_id_archive_analysis_df["obs_id"]
                .astype("string")
                .str.strip()
                .eq("")
                .fillna(False)
                .sum()
            ),
            "parsed_rows": int(parsed_obs_id_mask.sum()),
            "parse_failures": int((~parsed_obs_id_mask).sum()),
            "member_mismatches_among_parsed": int(
                member_mismatch_mask.sum()
            ),
            "source_with_whitespace": int(
                obs_id_archive_analysis_df[
                    "source_contains_whitespace"
                ].sum()
            ),
            "source_with_dot": int(
                obs_id_archive_analysis_df[
                    "source_contains_dot"
                ].sum()
            ),
            "non_integer_spw_among_parsed": int(
                non_integer_spw_mask.sum()
            ),
            "duplicate_obs_id_rows": len(duplicate_obs_id_df),
            "duplicate_source_execution_spw_rows": len(
                duplicate_source_execution_spw_df
            ),
        }
    ]
)

display(obs_id_grammar_summary_df)
display(
    obs_id_archive_analysis_df["obs_id_parse_issue"]
    .fillna("<NONE>")
    .value_counts(dropna=False)
    .rename_axis("parse_issue")
    .reset_index(name="row_count")
)
display(
    obs_id_archive_exception_df.head(
        OBS_ID_EXCEPTION_EXAMPLE_LIMIT
    )
)
display(
    obs_id_boundary_case_df.head(
        OBS_ID_EXCEPTION_EXAMPLE_LIMIT
    )
)


,total_rows,null_obs_id,blank_obs_id,parsed_rows,parse_failures,member_mismatches_among_parsed,source_with_whitespace,source_with_dot,non_integer_spw_among_parsed,duplicate_obs_id_rows,duplicate_source_execution_spw_rows
0,442507,0,0,442141,366,0,790,90435,0,496,114


,parse_issue,row_count
0,<NONE>,442141
1,unexpected_obs_id_format,366


,obs_id_archive_row_index,proposal_id,obs_publisher_did,member_ous_uid,asdm_uid,obs_id,obs_id_census_stratum,obs_member_ous_uid,source_name,spw_id,obs_id_parse_status,obs_id_parse_issue,obs_member_matches_column,source_contains_whitespace,source_contains_dot,spw_is_integer_token
21924,21924,2013.1.00163.S,ADS/JAO.ALMA#2013.1.00163.S,uid://A001/X12e/X1c8,uid://A002/Xa14be9/X1315,uid://A001/X12e/X1c8.source.GY_284 16:27:30.85 -24:24:56.0..spw.,2013,NaN,NaN,NaN,FAILED,unexpected_obs_id_format,False,False,False,False
21925,21925,2013.1.00163.S,ADS/JAO.ALMA#2013.1.00163.S,uid://A001/X12e/X1c8,uid://A002/Xa14be9/X1315,uid://A001/X12e/X1c8.source.GY_284 16:27:30.85 -24:24:56.0..spw.,2013,NaN,NaN,NaN,FAILED,unexpected_obs_id_format,False,False,False,False
21926,21926,2013.1.00163.S,ADS/JAO.ALMA#2013.1.00163.S,uid://A001/X12e/X1c8,uid://A002/Xa14be9/X1315,uid://A001/X12e/X1c8.source.GY_284 16:27:30.85 -24:24:56.0..spw.,2013,NaN,NaN,NaN,FAILED,unexpected_obs_id_format,False,False,False,False
21927,21927,2013.1.00163.S,ADS/JAO.ALMA#2013.1.00163.S,uid://A001/X12e/X1c8,uid://A002/Xa14be9/X1315,uid://A001/X12e/X1c8.source.GY_284 16:27:30.85 -24:24:56.0..spw.,2013,NaN,NaN,NaN,FAILED,unexpected_obs_id_format,False,False,False,False
21928,21928,2013.1.00163.S,ADS/JAO.ALMA#2013.1.00163.S,uid://A001/X12e/X1c8,uid://A002/Xa14be9/X1315,uid://A001/X12e/X1c8.source.GY_284 16:27:30.85 -24:24:56.0..spw.,2013,NaN,NaN,NaN,FAILED,unexpected_obs_id_format,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
133306,133306,2018.A.00061.S,ADS/JAO.ALMA#2018.A.00061.S,uid://A001/X142c/X98,uid://A002/Xe191fa/X2850,uid://A001/X142c/X98.source.Calibrate_North_Molecular_Ridge.spw.,2018,NaN,NaN,NaN,FAILED,unexpected_obs_id_format,False,False,False,False
133307,133307,2018.A.00061.S,ADS/JAO.ALMA#2018.A.00061.S,uid://A001/X142c/X98,uid://A002/Xe191fa/X2850,uid://A001/X142c/X98.source.Calibrate_North_Molecular_Ridge.spw.,2018,NaN,NaN,NaN,FAILED,unexpected_obs_id_format,False,False,False,False
133308,133308,2018.A.00061.S,ADS/JAO.ALMA#2018.A.00061.S,uid://A001/X142c/X98,uid://A002/Xe191fa/X2850,uid://A001/X142c/X98.source.Calibrate_North_Molecular_Ridge.spw.,2018,NaN,NaN,NaN,FAILED,unexpected_obs_id_format,False,False,False,False
133345,133345,2018.A.00061.S,ADS/JAO.ALMA#2018.A.00061.S,uid://A001/X142c/X94,uid://A002/Xe16acd/X82ba,uid://A001/X142c/X94.source.Calibrate_North_Molecular_Ridge.spw.,2018,NaN,NaN,NaN,FAILED,unexpected_obs_id_format,False,False,False,False


,obs_id_archive_row_index,proposal_id,obs_publisher_did,member_ous_uid,asdm_uid,obs_id,obs_id_census_stratum,obs_member_ous_uid,source_name,spw_id,obs_id_parse_status,obs_id_parse_issue,obs_member_matches_column,source_contains_whitespace,source_contains_dot,spw_is_integer_token
59,59,2011.0.00020.S,ADS/JAO.ALMA#2011.0.00020.S,uid://A002/X303d22/X8e,uid://A002/X32d5f1/X570,uid://A002/X303d22/X8e.source.IRAS 00183-7111.spw.23,2011,uid://A002/X303d22/X8e,IRAS 00183-7111,23,PARSED,NaN,True,True,False,True
60,60,2011.0.00020.S,ADS/JAO.ALMA#2011.0.00020.S,uid://A002/X303d22/X8e,uid://A002/X32d5f1/X570,uid://A002/X303d22/X8e.source.IRAS 00183-7111.spw.17,2011,uid://A002/X303d22/X8e,IRAS 00183-7111,17,PARSED,NaN,True,True,False,True
61,61,2011.0.00020.S,ADS/JAO.ALMA#2011.0.00020.S,uid://A002/X303d22/X8e,uid://A002/X32d5f1/X570,uid://A002/X303d22/X8e.source.IRAS 00183-7111.spw.19,2011,uid://A002/X303d22/X8e,IRAS 00183-7111,19,PARSED,NaN,True,True,False,True
62,62,2011.0.00020.S,ADS/JAO.ALMA#2011.0.00020.S,uid://A002/X303d22/X8e,uid://A002/X32d5f1/X570,uid://A002/X303d22/X8e.source.IRAS 00183-7111.spw.21,2011,uid://A002/X303d22/X8e,IRAS 00183-7111,21,PARSED,NaN,True,True,False,True
63,63,2011.0.00020.S,ADS/JAO.ALMA#2011.0.00020.S,uid://A002/X303d22/X9e,uid://A002/X32d5f1/X6dd,uid://A002/X303d22/X9e.source.IRAS 22491-1808.spw.17,2011,uid://A002/X303d22/X9e,IRAS 22491-1808,17,PARSED,NaN,True,True,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
360,360,2011.0.00009.SV,ADS/JAO.ALMA#2011.0.00009.SV,uid://A002/X37afb5/X6,uid://A002/X37b127/X17d,uid://A002/X37afb5/X6.source.Orion KL.spw.71,2011,uid://A002/X37afb5/X6,Orion KL,71,PARSED,NaN,True,True,False,True
361,361,2011.0.00009.SV,ADS/JAO.ALMA#2011.0.00009.SV,uid://A002/X37afb5/X6,uid://A002/X37b127/X17d,uid://A002/X37afb5/X6.source.Orion KL.spw.73,2011,uid://A002/X37afb5/X6,Orion KL,73,PARSED,NaN,True,True,False,True
362,362,2011.0.00009.SV,ADS/JAO.ALMA#2011.0.00009.SV,uid://A002/X37afb5/X6,uid://A002/X37b127/X17d,uid://A002/X37afb5/X6.source.Orion KL.spw.75,2011,uid://A002/X37afb5/X6,Orion KL,75,PARSED,NaN,True,True,False,True
363,363,2011.0.00009.SV,ADS/JAO.ALMA#2011.0.00009.SV,uid://A002/X37afb5/X6,uid://A002/X37b127/X17d,uid://A002/X37afb5/X6.source.Orion KL.spw.77,2011,uid://A002/X37afb5/X6,Orion KL,77,PARSED,NaN,True,True,False,True


### Step 12a — Targeted `obs_id` truncation census

Parse failures are classified separately from valid source labels that
contain spaces or dots. Every value exactly at the declared 64-character
boundary is retained as a truncation/collision candidate—even when the
visible prefix happens to satisfy the regex. A returned 64-character
value must never be reused as an exact server-side predicate.


In [50]:
def add_obs_id_width_evidence(dataframe: pd.DataFrame) -> None:
    """Attach the same width-boundary evidence to every related frame."""
    dataframe["obs_id_length"] = (
        dataframe["obs_id"].astype("string").str.len()
    )
    dataframe["obs_id_at_declared_width"] = (
        dataframe["obs_id_length"].eq(OBS_ID_DECLARED_WIDTH)
    )


# The duplicate frames were copied in Step 12 before Step 12a runs.
# Annotate all three frames explicitly; Pandas copies do not inherit
# columns added later to the parent frame.
for obs_id_evidence_frame in (
    obs_id_archive_analysis_df,
    duplicate_obs_id_df,
    duplicate_source_execution_spw_df,
):
    add_obs_id_width_evidence(obs_id_evidence_frame)

required_width_columns = {
    "obs_id_length",
    "obs_id_at_declared_width",
}
for frame_name, obs_id_evidence_frame in {
    "obs_id_archive_analysis_df": obs_id_archive_analysis_df,
    "duplicate_obs_id_df": duplicate_obs_id_df,
    "duplicate_source_execution_spw_df": (
        duplicate_source_execution_spw_df
    ),
}.items():
    missing_width_columns = (
        required_width_columns - set(obs_id_evidence_frame.columns)
    )
    assert not missing_width_columns, (
        f"{frame_name} is missing width evidence columns: "
        f"{sorted(missing_width_columns)}"
    )
    expected_width_boundary = (
        obs_id_evidence_frame["obs_id_length"]
        .eq(OBS_ID_DECLARED_WIDTH)
        .fillna(False)
    )
    actual_width_boundary = (
        obs_id_evidence_frame["obs_id_at_declared_width"]
        .fillna(False)
    )
    assert actual_width_boundary.equals(expected_width_boundary), (
        f"{frame_name} has inconsistent obs_id width evidence"
    )


def classify_obs_id_failure(row: pd.Series) -> str:
    if row["obs_id_parse_status"] == "PARSED":
        return "PARSED"
    raw_value = row["obs_id"]
    if pd.isna(raw_value):
        return "MISSING"
    text = str(raw_value).strip()
    if not text:
        return "BLANK"
    if len(text) == OBS_ID_DECLARED_WIDTH:
        if text.endswith(".spw."):
            return "TRUNCATED_AFTER_SPW_MARKER_AT_WIDTH"
        if ".source." in text and ".spw." not in text:
            return "TRUNCATED_IN_SOURCE_SEGMENT_AT_WIDTH"
        if ".source." in text:
            return "TRUNCATED_OTHER_SUFFIX_AT_WIDTH"
        return "WIDTH_LIMIT_WITHOUT_SOURCE_MARKER"
    return "NON_WIDTH_GRAMMAR_EXCEPTION"


obs_id_archive_analysis_df["obs_id_failure_class"] = (
    obs_id_archive_analysis_df.apply(
        classify_obs_id_failure,
        axis=1,
    )
)
obs_id_archive_analysis_df["obs_id_width_boundary_class"] = np.select(
    [
        parsed_obs_id_mask
        & obs_id_archive_analysis_df["obs_id_at_declared_width"],
        parsed_obs_id_mask,
        (~parsed_obs_id_mask)
        & obs_id_archive_analysis_df["obs_id_at_declared_width"],
    ],
    [
        "PARSED_AT_DECLARED_WIDTH_TRUNCATION_POSSIBLE",
        "PARSED_BELOW_DECLARED_WIDTH",
        "FAILED_AT_DECLARED_WIDTH_TRUNCATION_LIKELY",
    ],
    default="FAILED_BELOW_DECLARED_WIDTH_REVIEW_REQUIRED",
)
obs_id_failed_df = obs_id_archive_analysis_df.loc[
    ~parsed_obs_id_mask
].copy()
obs_id_truncation_census_df = (
    obs_id_failed_df
    .groupby(
        [
            "obs_id_census_stratum",
            "obs_id_length",
            "obs_id_failure_class",
        ],
        dropna=False,
    )
    .size()
    .rename("failed_rows")
    .reset_index()
)
obs_id_failure_class_summary_df = (
    obs_id_failed_df["obs_id_failure_class"]
    .value_counts(dropna=False)
    .rename_axis("obs_id_failure_class")
    .reset_index(name="failed_rows")
)
obs_id_width_boundary_census_df = (
    obs_id_archive_analysis_df
    .groupby(
        [
            "obs_id_parse_status",
            "obs_id_width_boundary_class",
        ],
        dropna=False,
    )
    .size()
    .rename("archive_rows")
    .reset_index()
)
obs_id_unclassified_failure_df = obs_id_failed_df.loc[
    obs_id_failed_df["obs_id_failure_class"].isin(
        [
            "NON_WIDTH_GRAMMAR_EXCEPTION",
            "WIDTH_LIMIT_WITHOUT_SOURCE_MARKER",
        ]
    )
].copy()
obs_id_failures_classified = (
    len(obs_id_failed_df) == 0
    or obs_id_unclassified_failure_df.empty
)
obs_id_truncation_status = (
    "ALL_FAILURES_CLASSIFIED_AS_DECLARED_WIDTH_TRUNCATION"
    if len(obs_id_failed_df)
    and obs_id_unclassified_failure_df.empty
    else (
        "NO_PARSE_FAILURES"
        if len(obs_id_failed_df) == 0
        else "MIXED_TRUNCATION_AND_GRAMMAR_EXCEPTIONS"
    )
)

display(obs_id_failure_class_summary_df)
display(obs_id_width_boundary_census_df)
display(obs_id_truncation_census_df)
display(
    obs_id_failed_df[
        [
            "proposal_id", "member_ous_uid", "asdm_uid",
            "obs_id", "obs_id_length", "obs_id_failure_class",
        ]
    ].head(OBS_ID_EXCEPTION_EXAMPLE_LIMIT)
)
display(
    obs_id_unclassified_failure_df.head(
        OBS_ID_EXCEPTION_EXAMPLE_LIMIT
    )
)
print("obs_id truncation status:", obs_id_truncation_status)


,obs_id_failure_class,failed_rows
0,TRUNCATED_IN_SOURCE_SEGMENT_AT_WIDTH,223
1,TRUNCATED_AFTER_SPW_MARKER_AT_WIDTH,143


,obs_id_parse_status,obs_id_width_boundary_class,archive_rows
0,FAILED,FAILED_AT_DECLARED_WIDTH_TRUNCATION_LIKELY,366
1,PARSED,PARSED_AT_DECLARED_WIDTH_TRUNCATION_POSSIBLE,275
2,PARSED,PARSED_BELOW_DECLARED_WIDTH,441866


,obs_id_census_stratum,obs_id_length,obs_id_failure_class,failed_rows
0,2013,64,TRUNCATED_AFTER_SPW_MARKER_AT_WIDTH,5
1,2015,64,TRUNCATED_IN_SOURCE_SEGMENT_AT_WIDTH,20
2,2016,64,TRUNCATED_IN_SOURCE_SEGMENT_AT_WIDTH,60
3,2018,64,TRUNCATED_AFTER_SPW_MARKER_AT_WIDTH,80
4,2018,64,TRUNCATED_IN_SOURCE_SEGMENT_AT_WIDTH,16
5,2019,64,TRUNCATED_AFTER_SPW_MARKER_AT_WIDTH,12
6,2019,64,TRUNCATED_IN_SOURCE_SEGMENT_AT_WIDTH,9
7,2021,64,TRUNCATED_AFTER_SPW_MARKER_AT_WIDTH,30
8,2021,64,TRUNCATED_IN_SOURCE_SEGMENT_AT_WIDTH,4
9,2023,64,TRUNCATED_AFTER_SPW_MARKER_AT_WIDTH,16


,proposal_id,member_ous_uid,asdm_uid,obs_id,obs_id_length,obs_id_failure_class
21924,2013.1.00163.S,uid://A001/X12e/X1c8,uid://A002/Xa14be9/X1315,uid://A001/X12e/X1c8.source.GY_284 16:27:30.85 -24:24:56.0..spw.,64,TRUNCATED_AFTER_SPW_MARKER_AT_WIDTH
21925,2013.1.00163.S,uid://A001/X12e/X1c8,uid://A002/Xa14be9/X1315,uid://A001/X12e/X1c8.source.GY_284 16:27:30.85 -24:24:56.0..spw.,64,TRUNCATED_AFTER_SPW_MARKER_AT_WIDTH
21926,2013.1.00163.S,uid://A001/X12e/X1c8,uid://A002/Xa14be9/X1315,uid://A001/X12e/X1c8.source.GY_284 16:27:30.85 -24:24:56.0..spw.,64,TRUNCATED_AFTER_SPW_MARKER_AT_WIDTH
21927,2013.1.00163.S,uid://A001/X12e/X1c8,uid://A002/Xa14be9/X1315,uid://A001/X12e/X1c8.source.GY_284 16:27:30.85 -24:24:56.0..spw.,64,TRUNCATED_AFTER_SPW_MARKER_AT_WIDTH
21928,2013.1.00163.S,uid://A001/X12e/X1c8,uid://A002/Xa14be9/X1315,uid://A001/X12e/X1c8.source.GY_284 16:27:30.85 -24:24:56.0..spw.,64,TRUNCATED_AFTER_SPW_MARKER_AT_WIDTH
...,...,...,...,...,...,...
133306,2018.A.00061.S,uid://A001/X142c/X98,uid://A002/Xe191fa/X2850,uid://A001/X142c/X98.source.Calibrate_North_Molecular_Ridge.spw.,64,TRUNCATED_AFTER_SPW_MARKER_AT_WIDTH
133307,2018.A.00061.S,uid://A001/X142c/X98,uid://A002/Xe191fa/X2850,uid://A001/X142c/X98.source.Calibrate_North_Molecular_Ridge.spw.,64,TRUNCATED_AFTER_SPW_MARKER_AT_WIDTH
133308,2018.A.00061.S,uid://A001/X142c/X98,uid://A002/Xe191fa/X2850,uid://A001/X142c/X98.source.Calibrate_North_Molecular_Ridge.spw.,64,TRUNCATED_AFTER_SPW_MARKER_AT_WIDTH
133345,2018.A.00061.S,uid://A001/X142c/X94,uid://A002/Xe16acd/X82ba,uid://A001/X142c/X94.source.Calibrate_North_Molecular_Ridge.spw.,64,TRUNCATED_AFTER_SPW_MARKER_AT_WIDTH


,obs_id_archive_row_index,proposal_id,obs_publisher_did,member_ous_uid,asdm_uid,obs_id,obs_id_census_stratum,obs_member_ous_uid,source_name,spw_id,obs_id_parse_status,obs_id_parse_issue,obs_member_matches_column,source_contains_whitespace,source_contains_dot,spw_is_integer_token,obs_id_length,obs_id_at_declared_width,obs_id_failure_class,obs_id_width_boundary_class


obs_id truncation status: ALL_FAILURES_CLASSIFIED_AS_DECLARED_WIDTH_TRUNCATION


### Step 12b — Candidate-key and `obs_publisher_did` census

Duplicate counts distinguish duplicate **rows involved** from duplicate
**groups**. A real duplicate is evidence about product granularity, not
an experiment failure.


In [51]:
duplicate_obs_id_group_df = (
    duplicate_obs_id_df
    .groupby("obs_id", dropna=False)
    .agg(
        archive_rows=("obs_id_archive_row_index", "size"),
        member_count=("member_ous_uid", "nunique"),
        asdm_count=("asdm_uid", "nunique"),
        publisher_id_count=("obs_publisher_did", "nunique"),
        width_boundary_rows=("obs_id_at_declared_width", "sum"),
    )
    .reset_index()
)
duplicate_source_execution_spw_group_df = (
    duplicate_source_execution_spw_df
    .groupby(archive_candidate_key_columns, dropna=False)
    .agg(
        archive_rows=("obs_id_archive_row_index", "size"),
        obs_id_count=("obs_id", "nunique"),
        publisher_id_count=("obs_publisher_did", "nunique"),
        width_boundary_rows=("obs_id_at_declared_width", "sum"),
    )
    .reset_index()
)
duplicate_obs_id_group_df["identifier_evidence_class"] = np.where(
    duplicate_obs_id_group_df["width_boundary_rows"].gt(0),
    "WIDTH_BOUNDARY_COLLISION_CANDIDATE",
    "RELIABLE_DUPLICATE_IDENTIFIER_BELOW_WIDTH",
)
duplicate_source_execution_spw_group_df[
    "identifier_evidence_class"
] = np.where(
    duplicate_source_execution_spw_group_df[
        "width_boundary_rows"
    ].gt(0),
    "WIDTH_BOUNDARY_COLLISION_CANDIDATE",
    "RELIABLE_COMPOSITE_DUPLICATE_BELOW_WIDTH",
)
reliable_duplicate_source_execution_spw_group_df = (
    duplicate_source_execution_spw_group_df.loc[
        duplicate_source_execution_spw_group_df[
            "width_boundary_rows"
        ].eq(0)
    ].copy()
)
width_boundary_duplicate_source_execution_spw_group_df = (
    duplicate_source_execution_spw_group_df.loc[
        duplicate_source_execution_spw_group_df[
            "width_boundary_rows"
        ].gt(0)
    ].copy()
)

obs_id_candidate_key_decision_df = pd.DataFrame(
    [
        {
            "candidate_key": "obs_id",
            "eligible_rows": int(obs_id_nonblank_mask.sum()),
            "duplicate_rows_involved": len(duplicate_obs_id_df),
            "duplicate_groups": len(duplicate_obs_id_group_df),
            "archive_wide_unique": (
                duplicate_obs_id_df.empty
                and obs_id_nonblank_mask.all()
            ),
            "status": (
                "ARCHIVE_WIDE_CANDIDATE"
                if duplicate_obs_id_df.empty
                and obs_id_nonblank_mask.all()
                else "NOT_ARCHIVE_WIDE_KEY_USE_SURROGATE"
            ),
        },
        {
            "candidate_key": (
                "member_ous_uid + asdm_uid + source_name + spw_id"
            ),
            "eligible_rows": len(
                archive_candidate_key_eligible_df
            ),
            "duplicate_rows_involved": len(
                duplicate_source_execution_spw_df
            ),
            "duplicate_groups": len(
                duplicate_source_execution_spw_group_df
            ),
            "archive_wide_unique": (
                duplicate_source_execution_spw_df.empty
                and len(archive_candidate_key_eligible_df)
                == len(obs_id_archive_analysis_df)
            ),
            "status": (
                "ARCHIVE_WIDE_CANDIDATE"
                if duplicate_source_execution_spw_df.empty
                and len(archive_candidate_key_eligible_df)
                == len(obs_id_archive_analysis_df)
                else (
                    "NOT_ARCHIVE_WIDE_KEY_"
                    "TRUNCATION_OR_PRODUCT_MULTIPLICITY"
                )
            ),
        },
    ]
)

publisher_text = (
    obs_id_archive_analysis_df["obs_publisher_did"]
    .astype("string")
    .str.strip()
)
publisher_nonblank_mask = (
    obs_id_archive_analysis_df["obs_publisher_did"].notna()
    & publisher_text.ne("").fillna(False)
)
duplicate_obs_publisher_did_df = (
    obs_id_archive_analysis_df.loc[
        publisher_nonblank_mask
        & obs_id_archive_analysis_df.duplicated(
            "obs_publisher_did",
            keep=False,
        )
    ]
    .sort_values("obs_publisher_did", kind="stable")
)
duplicate_obs_publisher_did_group_df = (
    duplicate_obs_publisher_did_df
    .groupby("obs_publisher_did", dropna=False)
    .agg(
        archive_rows=("obs_id_archive_row_index", "size"),
        member_count=("member_ous_uid", "nunique"),
        asdm_count=("asdm_uid", "nunique"),
        obs_id_count=("obs_id", "nunique"),
    )
    .reset_index()
)
obs_publisher_did_census_complete = (
    len(obs_id_archive_analysis_df) == obs_id_expected_total
)
obs_publisher_did_archive_unique = (
    publisher_nonblank_mask.all()
    and duplicate_obs_publisher_did_df.empty
)
obs_publisher_did_census_df = pd.DataFrame(
    [
        {
            "archive_rows": len(obs_id_archive_analysis_df),
            "null_rows": int(
                obs_id_archive_analysis_df[
                    "obs_publisher_did"
                ].isna().sum()
            ),
            "blank_rows": int(
                publisher_text.eq("").fillna(False).sum()
            ),
            "nonblank_rows": int(publisher_nonblank_mask.sum()),
            "duplicate_rows_involved": len(
                duplicate_obs_publisher_did_df
            ),
            "duplicate_groups": len(
                duplicate_obs_publisher_did_group_df
            ),
            "archive_wide_unique_nonblank": (
                obs_publisher_did_archive_unique
            ),
            "status": (
                "ARCHIVE_WIDE_PRODUCT_IDENTIFIER_CANDIDATE"
                if obs_publisher_did_archive_unique
                else "NOT_ARCHIVE_WIDE_KEY_USE_SURROGATE"
            ),
        }
    ]
)

obs_id_census_parse_complete = parsed_obs_id_mask.all()

display(obs_id_candidate_key_decision_df)
display(duplicate_obs_id_group_df.head(OBS_ID_EXCEPTION_EXAMPLE_LIMIT))
display(
    duplicate_source_execution_spw_group_df.head(
        OBS_ID_EXCEPTION_EXAMPLE_LIMIT
    )
)
display(reliable_duplicate_source_execution_spw_group_df)
display(
    width_boundary_duplicate_source_execution_spw_group_df.head(
        OBS_ID_EXCEPTION_EXAMPLE_LIMIT
    )
)
display(obs_publisher_did_census_df)
display(
    duplicate_obs_publisher_did_group_df.head(
        OBS_ID_EXCEPTION_EXAMPLE_LIMIT
    )
)
print("identifier census coverage complete:", obs_id_census_coverage_complete)
print("obs_id parser coverage complete:", obs_id_census_parse_complete)


,candidate_key,eligible_rows,duplicate_rows_involved,duplicate_groups,archive_wide_unique,status
0,obs_id,442507,496,134,False,NOT_ARCHIVE_WIDE_KEY_USE_SURROGATE
1,member_ous_uid + asdm_uid + source_name + spw_id,442141,114,42,False,NOT_ARCHIVE_WIDE_KEY_TRUNCATION_OR_PRODUCT_MULTIPLICITY


,obs_id,archive_rows,member_count,asdm_count,publisher_id_count,width_boundary_rows,identifier_evidence_class
0,uid://A001/X128a/X1bf.source.Transient_Black_Hole_X-ray_Binary.s,4,1,1,1,4,WIDTH_BOUNDARY_COLLISION_CANDIDATE
1,uid://A001/X128a/X1c2.source.Transient_Black_Hole_X-ray_Binary.s,4,1,1,1,4,WIDTH_BOUNDARY_COLLISION_CANDIDATE
2,uid://A001/X128a/X1c9.source.Transient_Black_Hole_X-ray_Binary.s,4,1,1,1,4,WIDTH_BOUNDARY_COLLISION_CANDIDATE
3,uid://A001/X128a/X1cc.source.Transient_Black_Hole_X-ray_Binary.s,4,1,1,1,4,WIDTH_BOUNDARY_COLLISION_CANDIDATE
4,uid://A001/X128a/X1d3.source.Transient_Black_Hole_X-ray_Binary.s,4,1,1,1,4,WIDTH_BOUNDARY_COLLISION_CANDIDATE
...,...,...,...,...,...,...,...
95,uid://A001/X365b/X21e.source.VIKINGJ222718_-332335_z_6.160.spw.2,3,1,1,1,3,WIDTH_BOUNDARY_COLLISION_CANDIDATE
96,uid://A001/X3664/Xf9.source.Bernardinelli-Bernstein_C2014_UN271.,4,1,1,1,4,WIDTH_BOUNDARY_COLLISION_CANDIDATE
97,uid://A001/X3667/X3a3.source.GRO_J1655-40_Jet-interaction_site.s,6,1,1,1,6,WIDTH_BOUNDARY_COLLISION_CANDIDATE
98,uid://A001/X3667/X3a5.source.GRO_J1655-40_Jet-interaction_site.s,6,1,1,1,6,WIDTH_BOUNDARY_COLLISION_CANDIDATE


,member_ous_uid,asdm_uid,source_name,spw_id,archive_rows,obs_id_count,publisher_id_count,width_boundary_rows,identifier_evidence_class
0,uid://A001/X133d/X27a9,uid://A002/Xdd9a29/X174,Northeast_Section_of_NGC6334,1,2,1,1,2,WIDTH_BOUNDARY_COLLISION_CANDIDATE
1,uid://A001/X133d/X27a9,uid://A002/Xdd9a29/X174,Northeast_Section_of_NGC6334,2,5,1,1,5,WIDTH_BOUNDARY_COLLISION_CANDIDATE
2,uid://A001/X133d/Xe97,uid://A002/Xd7aa27/X128,587731511545233578_8082-12701,2,3,1,1,3,WIDTH_BOUNDARY_COLLISION_CANDIDATE
3,uid://A001/X133d/Xe9b,uid://A002/Xd81670/X6f13,587731174919438555_8615-12702,2,3,1,1,3,WIDTH_BOUNDARY_COLLISION_CANDIDATE
4,uid://A001/X1465/X5f0,uid://A002/Xeba1ac/X40ca,eso137-001_-_south_outer_tail,2,3,1,1,3,WIDTH_BOUNDARY_COLLISION_CANDIDATE
5,uid://A001/X1465/X5f2,uid://A002/Xe48598/X19681,eso137-001_-_south_outer_tail,2,3,1,1,3,WIDTH_BOUNDARY_COLLISION_CANDIDATE
6,uid://A001/X1465/X5f6,uid://A002/Xeaff9c/X10b7,eso137-001_-_central_filament,2,3,1,1,3,WIDTH_BOUNDARY_COLLISION_CANDIDATE
7,uid://A001/X1465/X5f8,uid://A002/Xe48598/X18d22,eso137-001_-_central_filament,2,3,1,1,3,WIDTH_BOUNDARY_COLLISION_CANDIDATE
8,uid://A001/X14d8/X2c4,uid://A002/Xe5ed76/X1bb1,2MASS_J04215402+1530299_OFF_0,1,2,1,1,2,WIDTH_BOUNDARY_COLLISION_CANDIDATE
9,uid://A001/X14d8/X2c4,uid://A002/Xe5ed76/X1bb1,2MASS_J04215402+1530299_OFF_0,2,3,1,1,3,WIDTH_BOUNDARY_COLLISION_CANDIDATE


,member_ous_uid,asdm_uid,source_name,spw_id,archive_rows,obs_id_count,publisher_id_count,width_boundary_rows,identifier_evidence_class


,member_ous_uid,asdm_uid,source_name,spw_id,archive_rows,obs_id_count,publisher_id_count,width_boundary_rows,identifier_evidence_class
0,uid://A001/X133d/X27a9,uid://A002/Xdd9a29/X174,Northeast_Section_of_NGC6334,1,2,1,1,2,WIDTH_BOUNDARY_COLLISION_CANDIDATE
1,uid://A001/X133d/X27a9,uid://A002/Xdd9a29/X174,Northeast_Section_of_NGC6334,2,5,1,1,5,WIDTH_BOUNDARY_COLLISION_CANDIDATE
2,uid://A001/X133d/Xe97,uid://A002/Xd7aa27/X128,587731511545233578_8082-12701,2,3,1,1,3,WIDTH_BOUNDARY_COLLISION_CANDIDATE
3,uid://A001/X133d/Xe9b,uid://A002/Xd81670/X6f13,587731174919438555_8615-12702,2,3,1,1,3,WIDTH_BOUNDARY_COLLISION_CANDIDATE
4,uid://A001/X1465/X5f0,uid://A002/Xeba1ac/X40ca,eso137-001_-_south_outer_tail,2,3,1,1,3,WIDTH_BOUNDARY_COLLISION_CANDIDATE
5,uid://A001/X1465/X5f2,uid://A002/Xe48598/X19681,eso137-001_-_south_outer_tail,2,3,1,1,3,WIDTH_BOUNDARY_COLLISION_CANDIDATE
6,uid://A001/X1465/X5f6,uid://A002/Xeaff9c/X10b7,eso137-001_-_central_filament,2,3,1,1,3,WIDTH_BOUNDARY_COLLISION_CANDIDATE
7,uid://A001/X1465/X5f8,uid://A002/Xe48598/X18d22,eso137-001_-_central_filament,2,3,1,1,3,WIDTH_BOUNDARY_COLLISION_CANDIDATE
8,uid://A001/X14d8/X2c4,uid://A002/Xe5ed76/X1bb1,2MASS_J04215402+1530299_OFF_0,1,2,1,1,2,WIDTH_BOUNDARY_COLLISION_CANDIDATE
9,uid://A001/X14d8/X2c4,uid://A002/Xe5ed76/X1bb1,2MASS_J04215402+1530299_OFF_0,2,3,1,1,3,WIDTH_BOUNDARY_COLLISION_CANDIDATE


,archive_rows,null_rows,blank_rows,nonblank_rows,duplicate_rows_involved,duplicate_groups,archive_wide_unique_nonblank,status
0,442507,0,0,442507,442501,5605,False,NOT_ARCHIVE_WIDE_KEY_USE_SURROGATE


,obs_publisher_did,archive_rows,member_count,asdm_count,obs_id_count
0,ADS/JAO.ALMA#2011.0.00001.E,4,1,1,4
1,ADS/JAO.ALMA#2011.0.00001.SV,4,1,1,4
2,ADS/JAO.ALMA#2011.0.00002.E,24,2,2,24
3,ADS/JAO.ALMA#2011.0.00002.SV,4,1,1,4
4,ADS/JAO.ALMA#2011.0.00003.SV,4,2,2,4
...,...,...,...,...,...
95,ADS/JAO.ALMA#2011.0.00497.S,16,4,4,16
96,ADS/JAO.ALMA#2011.0.00510.S,8,2,2,8
97,ADS/JAO.ALMA#2011.0.00511.S,11,5,5,11
98,ADS/JAO.ALMA#2011.0.00524.S,4,1,1,4


identifier census coverage complete: True
obs_id parser coverage complete: False


### Step 12c — Archive-wide `obs_publisher_did ↔ proposal_id` scope census

This census tests whether `obs_publisher_did` is an Archive-row/product
identifier or a proposal/publication-scoped external identifier.

The experiment checks:

1. missing and blank identifiers;
2. exact `ADS/JAO.ALMA#<proposal_id>` correspondence;
3. one publisher DID per proposal;
4. one proposal per publisher DID;
5. repeated publisher DID values across Archive rows.

In [52]:
PUBLISHER_DID_PREFIX = "ADS/JAO.ALMA#"
PUBLISHER_SCOPE_EXAMPLE_LIMIT = 50

required_publisher_scope_columns = {
    "proposal_id",
    "obs_publisher_did",
}

missing_publisher_scope_columns = (
    required_publisher_scope_columns
    - set(obs_id_archive_analysis_df.columns)
)

assert not missing_publisher_scope_columns, (
    "Step 12c requires columns missing from "
    "obs_id_archive_analysis_df: "
    f"{sorted(missing_publisher_scope_columns)}"
)


# Preserve the Archive-wide Step 12 population.
publisher_proposal_scope_df = (
    obs_id_archive_analysis_df[
        [
            "proposal_id",
            "obs_publisher_did",
            "member_ous_uid",
            "asdm_uid",
            "obs_id",
            "obs_id_census_stratum",
        ]
    ]
    .copy()
)


def normalized_identifier_series(
    series: pd.Series,
) -> pd.Series:
    """Normalize only for comparison; raw values remain preserved elsewhere."""
    return series.astype("string").str.strip()


publisher_proposal_scope_df[
    "proposal_id_normalized"
] = normalized_identifier_series(
    publisher_proposal_scope_df["proposal_id"]
)

publisher_proposal_scope_df[
    "obs_publisher_did_normalized"
] = normalized_identifier_series(
    publisher_proposal_scope_df["obs_publisher_did"]
)


proposal_missing_mask = (
    publisher_proposal_scope_df[
        "proposal_id_normalized"
    ].isna()
    | publisher_proposal_scope_df[
        "proposal_id_normalized"
    ].eq("")
)

publisher_missing_mask = (
    publisher_proposal_scope_df[
        "obs_publisher_did_normalized"
    ].isna()
    | publisher_proposal_scope_df[
        "obs_publisher_did_normalized"
    ].eq("")
)

publisher_proposal_scope_df[
    "expected_obs_publisher_did"
] = (
    PUBLISHER_DID_PREFIX
    + publisher_proposal_scope_df[
        "proposal_id_normalized"
    ]
)

publisher_proposal_scope_df[
    "publisher_did_matches_expected"
] = (
    publisher_proposal_scope_df[
        "obs_publisher_did_normalized"
    ]
    .eq(
        publisher_proposal_scope_df[
            "expected_obs_publisher_did"
        ]
    )
    .fillna(False)
)

eligible_publisher_mapping_mask = (
    ~proposal_missing_mask
    & ~publisher_missing_mask
)

publisher_mapping_mismatch_mask = (
    eligible_publisher_mapping_mask
    & ~publisher_proposal_scope_df[
        "publisher_did_matches_expected"
    ]
)


def unique_nonblank_values(
    series: pd.Series,
) -> tuple[str, ...]:
    values = (
        series.astype("string")
        .dropna()
        .str.strip()
    )
    values = values.loc[values.ne("")]
    return tuple(sorted(values.unique().tolist()))


# Direction 1:
# one proposal_id should map to exactly one publisher DID.
proposal_to_publisher_df = (
    publisher_proposal_scope_df.loc[
        eligible_publisher_mapping_mask
    ]
    .groupby(
        "proposal_id_normalized",
        dropna=False,
    )
    .agg(
        archive_rows=(
            "proposal_id_normalized",
            "size",
        ),
        publisher_id_count=(
            "obs_publisher_did_normalized",
            "nunique",
        ),
        publisher_ids=(
            "obs_publisher_did_normalized",
            unique_nonblank_values,
        ),
        member_count=(
            "member_ous_uid",
            "nunique",
        ),
        asdm_count=(
            "asdm_uid",
            "nunique",
        ),
        obs_id_count=(
            "obs_id",
            "nunique",
        ),
    )
    .reset_index()
)

proposal_to_multiple_publishers_df = (
    proposal_to_publisher_df.loc[
        proposal_to_publisher_df[
            "publisher_id_count"
        ].ne(1)
    ]
    .copy()
)


# Direction 2:
# one publisher DID should map to exactly one proposal_id.
publisher_to_proposal_df = (
    publisher_proposal_scope_df.loc[
        eligible_publisher_mapping_mask
    ]
    .groupby(
        "obs_publisher_did_normalized",
        dropna=False,
    )
    .agg(
        archive_rows=(
            "obs_publisher_did_normalized",
            "size",
        ),
        proposal_count=(
            "proposal_id_normalized",
            "nunique",
        ),
        proposal_ids=(
            "proposal_id_normalized",
            unique_nonblank_values,
        ),
        member_count=(
            "member_ous_uid",
            "nunique",
        ),
        asdm_count=(
            "asdm_uid",
            "nunique",
        ),
        obs_id_count=(
            "obs_id",
            "nunique",
        ),
    )
    .reset_index()
)

publisher_to_multiple_proposals_df = (
    publisher_to_proposal_df.loc[
        publisher_to_proposal_df[
            "proposal_count"
        ].ne(1)
    ]
    .copy()
)


# A proposal-scoped identifier is expected to repeat across rows.
publisher_duplicate_row_mask = (
    publisher_proposal_scope_df[
        "obs_publisher_did_normalized"
    ]
    .duplicated(keep=False)
    & ~publisher_missing_mask
)

publisher_proposal_scope_coverage_complete = (
    len(publisher_proposal_scope_df)
    == obs_id_expected_total
)

publisher_proposal_scope_complete = bool(
    publisher_proposal_scope_coverage_complete
    and proposal_missing_mask.sum() == 0
    and publisher_missing_mask.sum() == 0
    and publisher_mapping_mismatch_mask.sum() == 0
    and proposal_to_multiple_publishers_df.empty
    and publisher_to_multiple_proposals_df.empty
)


if not publisher_proposal_scope_coverage_complete:
    publisher_proposal_scope_status = (
        "INCOMPLETE_ARCHIVE_WIDE_CENSUS"
    )
elif proposal_missing_mask.any():
    publisher_proposal_scope_status = (
        "PROPOSAL_ID_MISSING_IN_CENSUS"
    )
elif publisher_missing_mask.any():
    publisher_proposal_scope_status = (
        "OBS_PUBLISHER_DID_MISSING_IN_CENSUS"
    )
elif publisher_mapping_mismatch_mask.any():
    publisher_proposal_scope_status = (
        "NOT_CANONICAL_PROPOSAL_PUBLISHER_MAPPING"
    )
elif not proposal_to_multiple_publishers_df.empty:
    publisher_proposal_scope_status = (
        "PROPOSAL_MAPS_TO_MULTIPLE_PUBLISHER_DIDS"
    )
elif not publisher_to_multiple_proposals_df.empty:
    publisher_proposal_scope_status = (
        "PUBLISHER_DID_MAPS_TO_MULTIPLE_PROPOSALS"
    )
else:
    publisher_proposal_scope_status = (
        "PROPOSAL_SCOPED_PUBLICATION_IDENTIFIER_NOT_ROW_KEY"
    )


publisher_proposal_scope_summary_df = pd.DataFrame(
    [
        {
            "archive_rows": len(
                publisher_proposal_scope_df
            ),
            "expected_archive_rows": (
                obs_id_expected_total
            ),
            "proposal_missing_or_blank_rows": int(
                proposal_missing_mask.sum()
            ),
            "publisher_did_missing_or_blank_rows": int(
                publisher_missing_mask.sum()
            ),
            "eligible_mapping_rows": int(
                eligible_publisher_mapping_mask.sum()
            ),
            "exact_prefix_mapping_rows": int(
                publisher_proposal_scope_df.loc[
                    eligible_publisher_mapping_mask,
                    "publisher_did_matches_expected",
                ].sum()
            ),
            "mapping_mismatch_rows": int(
                publisher_mapping_mismatch_mask.sum()
            ),
            "proposal_count": len(
                proposal_to_publisher_df
            ),
            "publisher_did_count": len(
                publisher_to_proposal_df
            ),
            "proposals_with_multiple_publisher_dids": len(
                proposal_to_multiple_publishers_df
            ),
            "publisher_dids_with_multiple_proposals": len(
                publisher_to_multiple_proposals_df
            ),
            "rows_with_repeated_publisher_did": int(
                publisher_duplicate_row_mask.sum()
            ),
            "archive_wide_row_key": bool(
                not publisher_duplicate_row_mask.any()
                and publisher_missing_mask.sum() == 0
            ),
            "scope_status": (
                publisher_proposal_scope_status
            ),
        }
    ]
)


publisher_mapping_mismatch_examples_df = (
    publisher_proposal_scope_df.loc[
        publisher_mapping_mismatch_mask,
        [
            "proposal_id",
            "obs_publisher_did",
            "expected_obs_publisher_did",
            "member_ous_uid",
            "asdm_uid",
            "obs_id",
        ],
    ]
    .head(PUBLISHER_SCOPE_EXAMPLE_LIMIT)
    .copy()
)


display(publisher_proposal_scope_summary_df)

display(
    proposal_to_multiple_publishers_df.head(
        PUBLISHER_SCOPE_EXAMPLE_LIMIT
    )
)

display(
    publisher_to_multiple_proposals_df.head(
        PUBLISHER_SCOPE_EXAMPLE_LIMIT
    )
)

display(publisher_mapping_mismatch_examples_df)

print(
    "Publisher/proposal scope status:",
    publisher_proposal_scope_status,
)

,archive_rows,expected_archive_rows,proposal_missing_or_blank_rows,publisher_did_missing_or_blank_rows,eligible_mapping_rows,exact_prefix_mapping_rows,mapping_mismatch_rows,proposal_count,publisher_did_count,proposals_with_multiple_publisher_dids,publisher_dids_with_multiple_proposals,rows_with_repeated_publisher_did,archive_wide_row_key,scope_status
0,442507,442507,0,0,442507,442507,0,5611,5611,0,0,442501,False,PROPOSAL_SCOPED_PUBLICATION_IDENTIFIER_NOT_ROW_KEY


,proposal_id_normalized,archive_rows,publisher_id_count,publisher_ids,member_count,asdm_count,obs_id_count


,obs_publisher_did_normalized,archive_rows,proposal_count,proposal_ids,member_count,asdm_count,obs_id_count


,proposal_id,obs_publisher_did,expected_obs_publisher_did,member_ous_uid,asdm_uid,obs_id


Publisher/proposal scope status: PROPOSAL_SCOPED_PUBLICATION_IDENTIFIER_NOT_ROW_KEY


## Step 13 — Normalized repeated source across ASDM

Discovery reuses the complete identifier retrieval and does not use
UID-ordered `TOP` sampling. Raw source labels are preserved, while a
case-folded label is used only to find candidates. Sky positions then
provide a separate tolerance-aware physical cross-check.


In [53]:
parsed_identifier_archive_df = (
    obs_id_archive_analysis_df.loc[parsed_obs_id_mask].copy()
)
parsed_identifier_archive_df["source_name_normalized"] = (
    parsed_identifier_archive_df["source_name"]
    .astype("string")
    .str.strip()
    .str.casefold()
)

archive_multi_asdm_member_df = (
    parsed_identifier_archive_df
    .groupby("member_ous_uid", dropna=False)
    .agg(
        asdm_count=("asdm_uid", "nunique"),
        archive_rows=("obs_id", "size"),
        proposal_ids=(
            "proposal_id",
            lambda values: tuple(
                sorted(set(values.dropna().astype(str)))
            ),
        ),
    )
    .reset_index()
)
archive_multi_asdm_member_df = archive_multi_asdm_member_df.loc[
    archive_multi_asdm_member_df["asdm_count"].gt(1)
].copy()

archive_repeated_source_across_asdm_df = (
    parsed_identifier_archive_df
    .groupby(
        ["member_ous_uid", "source_name_normalized"],
        dropna=False,
    )
    .agg(
        asdm_count=("asdm_uid", "nunique"),
        archive_rows=("obs_id", "size"),
        raw_source_labels=(
            "source_name",
            lambda values: tuple(
                sorted(set(values.dropna().astype(str)))
            ),
        ),
        asdm_uids=(
            "asdm_uid",
            lambda values: tuple(
                sorted(set(values.dropna().astype(str)))
            ),
        ),
        spw_ids=(
            "spw_id",
            lambda values: tuple(
                sorted(set(values.dropna().astype(str)))
            ),
        ),
    )
    .reset_index()
)
archive_repeated_source_across_asdm_df = (
    archive_repeated_source_across_asdm_df.loc[
        archive_repeated_source_across_asdm_df[
            "asdm_count"
        ].gt(1)
    ]
    .sort_values(
        ["asdm_count", "archive_rows"],
        ascending=[False, False],
        kind="stable",
    )
)

REPEATED_SOURCE_DETAIL_MEMBER_LIMIT = 50
REPEATED_SOURCE_DETAIL_SAFETY_LIMIT = 20_000
repeated_source_member_uids = (
    archive_repeated_source_across_asdm_df["member_ous_uid"]
    .dropna()
    .astype(str)
    .drop_duplicates()
    .head(REPEATED_SOURCE_DETAIL_MEMBER_LIMIT)
    .tolist()
)

display(
    pd.DataFrame(
        [
            {
                "archive_multi_asdm_members": len(
                    archive_multi_asdm_member_df
                ),
                "repeated_normalized_source_groups": len(
                    archive_repeated_source_across_asdm_df
                ),
                "members_selected_for_metadata_detail": len(
                    repeated_source_member_uids
                ),
                "identifier_census_parse_complete": bool(
                    obs_id_census_parse_complete
                ),
            }
        ]
    )
)
display(archive_multi_asdm_member_df.head(100))
display(archive_repeated_source_across_asdm_df.head(100))


,archive_multi_asdm_members,repeated_normalized_source_groups,members_selected_for_metadata_detail,identifier_census_parse_complete
0,115,1,1,False


,member_ous_uid,asdm_count,archive_rows,proposal_ids
1089,uid://A001/X1284/X1528,2,12,"(2017.1.00025.S,)"
1116,uid://A001/X1284/X15eb,2,22,"(2017.1.00015.S,)"
1686,uid://A001/X1284/X928,2,180,"(2017.1.00180.S,)"
4691,uid://A001/X133d/X1ef9,2,600,"(2018.1.00526.S,)"
5487,uid://A001/X133d/X3c1b,2,10,"(2018.1.01321.S,)"
...,...,...,...,...
29697,uid://A001/X885/X2ca,2,184,"(2016.1.01346.S,)"
29699,uid://A001/X885/X2d0,3,184,"(2016.1.01346.S,)"
29702,uid://A001/X885/X2d9,2,24,"(2016.1.01346.S,)"
29720,uid://A001/X885/X30f,2,136,"(2016.1.01346.S,)"


,member_ous_uid,source_name_normalized,asdm_count,archive_rows,raw_source_labels,asdm_uids,spw_ids
96393,uid://A002/X788a57/X41,3c279,2,16,"(3C279, 3c279)","(uid://A002/X7d6d46/X8c, uid://A002/X97c221/X28eb)","(17, 19, 21, 23, 31, 33, 35, 37)"


In [54]:
repeated_source_detail_columns = [
    name
    for name in [
        "proposal_id", "member_ous_uid", "group_ous_uid",
        "asdm_uid", "obs_id", "target_name", "s_ra", "s_dec",
        "s_fov", "s_region", "is_mosaic", "s_resolution",
        "spatial_resolution", "spatial_scale_max",
        "frequency_support", "antenna_arrays", "t_min", "t_max",
        "cont_sensitivity_bandwidth", "dataproduct_type",
        "calib_level", "frequency", "bandwidth",
    ]
    if name in available_columns
]

if repeated_source_member_uids:
    repeated_source_member_sql = ",\n    ".join(
        quote_adql_string(uid)
        for uid in repeated_source_member_uids
    )
    repeated_source_detail_count_query = f"""
    SELECT COUNT(*) AS total_rows
    FROM ivoa.obscore
    WHERE science_observation = 'T'
      AND member_ous_uid IN (
        {repeated_source_member_sql}
      )
    """
    repeated_source_detail_count_run = run_tap_query(
        repeated_source_detail_count_query,
        maxrec=10,
        label="Count repeated-source metadata-detail population",
    )
    query_runs.append(repeated_source_detail_count_run)
    repeated_source_detail_expected_rows = int(
        repeated_source_detail_count_run.table[0]["total_rows"]
    )
    if (
        repeated_source_detail_expected_rows
        > REPEATED_SOURCE_DETAIL_SAFETY_LIMIT
    ):
        raise RuntimeError(
            "Repeated-source detail exceeds safety limit: "
            f"{repeated_source_detail_expected_rows}."
        )

    repeated_source_column_sql = ",\n    ".join(
        repeated_source_detail_columns
    )
    repeated_source_detail_query = f"""
    SELECT
        {repeated_source_column_sql}
    FROM ivoa.obscore
    WHERE science_observation = 'T'
      AND member_ous_uid IN (
        {repeated_source_member_sql}
      )
    """
    repeated_source_detail_run = run_tap_query(
        repeated_source_detail_query,
        maxrec=max(repeated_source_detail_expected_rows, 1),
        label="Retrieve repeated-source metadata detail",
    )
    query_runs.append(repeated_source_detail_run)
    repeated_source_detail_df = astropy_table_to_pandas(
        repeated_source_detail_run.table
    )
    if (
        "OVERFLOW" in " ".join(
            repeated_source_detail_run.query_status
        )
        or len(repeated_source_detail_df)
        != repeated_source_detail_expected_rows
    ):
        raise RuntimeError(
            "Repeated-source metadata detail is incomplete."
        )

    repeated_source_parsed_df = pd.DataFrame(
        repeated_source_detail_df["obs_id"]
        .map(parse_obs_id_structure)
        .tolist()
    )
    for parsed_column in repeated_source_parsed_df.columns:
        repeated_source_detail_df[parsed_column] = (
            repeated_source_parsed_df[parsed_column].to_numpy()
        )
    repeated_source_detail_df["source_name_normalized"] = (
        repeated_source_detail_df["source_name"]
        .astype("string")
        .str.strip()
        .str.casefold()
    )

    candidate_keys = archive_repeated_source_across_asdm_df[
        ["member_ous_uid", "source_name_normalized"]
    ].drop_duplicates()
    repeated_source_verified_df = repeated_source_detail_df.merge(
        candidate_keys,
        on=["member_ous_uid", "source_name_normalized"],
        how="inner",
        validate="many_to_one",
    )
else:
    repeated_source_detail_expected_rows = 0
    repeated_source_detail_df = pd.DataFrame()
    repeated_source_verified_df = pd.DataFrame()

CROSS_ASDM_FIELDS = [
    "target_name", "s_ra", "s_dec", "s_fov", "s_region",
    "is_mosaic", "s_resolution", "spatial_resolution",
    "spatial_scale_max", "frequency_support", "antenna_arrays",
    "t_min", "t_max", "cont_sensitivity_bandwidth",
]
cross_asdm_field_records = []
if len(repeated_source_verified_df):
    for context_values, context_rows in repeated_source_verified_df.groupby(
        ["member_ous_uid", "source_name_normalized"],
        dropna=False,
        sort=True,
    ):
        for field in CROSS_ASDM_FIELDS:
            if field not in context_rows.columns:
                continue
            values_by_asdm = {}
            for asdm_uid, asdm_rows in context_rows.groupby(
                "asdm_uid", dropna=False, sort=True
            ):
                values_by_asdm[str(asdm_uid)] = tuple(
                    sorted(
                        {
                            repr(value)
                            for value in asdm_rows[field].tolist()
                        }
                    )
                )
            distinct_signatures = {
                signature for signature in values_by_asdm.values()
            }
            cross_asdm_field_records.append(
                {
                    "member_ous_uid": context_values[0],
                    "source_name_normalized": context_values[1],
                    "field": field,
                    "asdm_count": len(values_by_asdm),
                    "distinct_asdm_value_signatures": len(
                        distinct_signatures
                    ),
                    "cross_asdm_status": (
                        "STABLE_ACROSS_ASDM"
                        if len(distinct_signatures) == 1
                        else "VARIES_ACROSS_ASDM"
                    ),
                    "values_by_asdm": values_by_asdm,
                }
            )

archive_cross_asdm_field_evidence_df = pd.DataFrame(
    cross_asdm_field_records
)


def maximum_pairwise_separation_arcsec(
    ra_degrees: np.ndarray,
    dec_degrees: np.ndarray,
) -> float:
    if len(ra_degrees) < 2:
        return 0.0
    ra = np.deg2rad(np.asarray(ra_degrees, dtype=float))
    dec = np.deg2rad(np.asarray(dec_degrees, dtype=float))
    maximum = 0.0
    for left in range(len(ra) - 1):
        delta_ra = ra[left + 1 :] - ra[left]
        delta_dec = dec[left + 1 :] - dec[left]
        haversine_a = (
            np.sin(delta_dec / 2.0) ** 2
            + np.cos(dec[left])
            * np.cos(dec[left + 1 :])
            * np.sin(delta_ra / 2.0) ** 2
        )
        angles = 2.0 * np.arctan2(
            np.sqrt(np.clip(haversine_a, 0.0, 1.0)),
            np.sqrt(np.clip(1.0 - haversine_a, 0.0, 1.0)),
        )
        if len(angles):
            maximum = max(maximum, float(np.max(angles)))
    return float(np.rad2deg(maximum) * 3600.0)


SOURCE_POSITION_TOLERANCE_ARCSEC = 1.0
repeated_source_spatial_records = []
if len(repeated_source_verified_df):
    context_columns = [
        "member_ous_uid", "source_name_normalized"
    ]
    for context_values, context_rows in repeated_source_verified_df.groupby(
        context_columns,
        dropna=False,
        sort=True,
    ):
        position_rows = (
            context_rows[["asdm_uid", "s_ra", "s_dec"]]
            .dropna()
            .drop_duplicates()
        )
        separation = maximum_pairwise_separation_arcsec(
            position_rows["s_ra"].to_numpy(),
            position_rows["s_dec"].to_numpy(),
        )
        repeated_source_spatial_records.append(
            {
                "member_ous_uid": context_values[0],
                "source_name_normalized": context_values[1],
                "asdm_count": context_rows["asdm_uid"].nunique(),
                "raw_source_labels": tuple(
                    sorted(
                        set(
                            context_rows["source_name"]
                            .dropna()
                            .astype(str)
                        )
                    )
                ),
                "maximum_position_separation_arcsec": separation,
                "within_1_arcsec_position_tolerance": (
                    separation <= SOURCE_POSITION_TOLERANCE_ARCSEC
                ),
            }
        )
archive_repeated_source_spatial_evidence_df = pd.DataFrame(
    repeated_source_spatial_records
)

verified_repeated_source_spatial_df = (
    archive_repeated_source_spatial_evidence_df.loc[
        archive_repeated_source_spatial_evidence_df[
            "within_1_arcsec_position_tolerance"
        ].fillna(False)
    ].copy()
    if len(archive_repeated_source_spatial_evidence_df)
    else pd.DataFrame()
)

repeated_source_archive_status = (
    "SPATIALLY_VERIFIED_REPEATED_SOURCE_ACROSS_ASDM_FOUND"
    if len(verified_repeated_source_spatial_df)
    else (
        "NORMALIZED_LABEL_CANDIDATES_FOUND_BUT_NOT_SPATIALLY_VERIFIED"
        if len(archive_repeated_source_across_asdm_df)
        else (
            "NO_REPEATED_SOURCE_IN_COMPLETE_PARSEABLE_POPULATION"
            if obs_id_census_parse_complete
            else "NO_REPEATED_SOURCE_BUT_TRUNCATED_IDENTIFIERS_REMAIN"
        )
    )
)
repeated_source_discovery_sufficient = bool(
    len(verified_repeated_source_spatial_df)
    or obs_id_census_parse_complete
)

display(archive_cross_asdm_field_evidence_df)
display(archive_repeated_source_spatial_evidence_df)
display(verified_repeated_source_spatial_df)
print("Repeated-source Archive status:", repeated_source_archive_status)



--- Count repeated-source metadata-detail population ---
Endpoint: https://almascience.eso.org/tap
MAXREC: 10
ADQL:

    SELECT COUNT(*) AS total_rows
    FROM ivoa.obscore
    WHERE science_observation = 'T'
      AND member_ous_uid IN (
        'uid://A002/X788a57/X41'
      )
    
Retrieved rows: 1
QUERY_STATUS: ('OK',)
Warnings: <none>

--- Retrieve repeated-source metadata detail ---
Endpoint: https://almascience.eso.org/tap
MAXREC: 16
ADQL:

    SELECT
        proposal_id,
    member_ous_uid,
    group_ous_uid,
    asdm_uid,
    obs_id,
    target_name,
    s_ra,
    s_dec,
    s_fov,
    s_region,
    is_mosaic,
    s_resolution,
    spatial_resolution,
    spatial_scale_max,
    frequency_support,
    antenna_arrays,
    t_min,
    t_max,
    cont_sensitivity_bandwidth,
    dataproduct_type,
    calib_level,
    frequency,
    bandwidth
    FROM ivoa.obscore
    WHERE science_observation = 'T'
      AND member_ous_uid IN (
        'uid://A002/X788a57/X41'
      )
    
Retrieve

,member_ous_uid,source_name_normalized,field,asdm_count,distinct_asdm_value_signatures,cross_asdm_status,values_by_asdm
0,uid://A002/X788a57/X41,3c279,target_name,2,2,VARIES_ACROSS_ASDM,"{'uid://A002/X7d6d46/X8c': (''3C279'',), 'uid://A002/X97c221/X28eb': (''3c279'',)}"
1,uid://A002/X788a57/X41,3c279,s_ra,2,2,VARIES_ACROSS_ASDM,"{'uid://A002/X7d6d46/X8c': ('194.0465275',), 'uid://A002/X97c221/X28eb': ('194.04652736999998',)}"
2,uid://A002/X788a57/X41,3c279,s_dec,2,2,VARIES_ACROSS_ASDM,"{'uid://A002/X7d6d46/X8c': ('-5.7893125',), 'uid://A002/X97c221/X28eb': ('-5.789312420000001',)}"
3,uid://A002/X788a57/X41,3c279,s_fov,2,2,VARIES_ACROSS_ASDM,"{'uid://A002/X7d6d46/X8c': ('0.015018393004362316',), 'uid://A002/X97c221/X28eb': ('0.015027305166000503',)}"
4,uid://A002/X788a57/X41,3c279,s_region,2,2,VARIES_ACROSS_ASDM,"{'uid://A002/X7d6d46/X8c': (''Circle ICRS 194.046528 -5.789313 0.007509'',), 'uid://A002/X97c221/X28eb': (''Circle I..."
5,uid://A002/X788a57/X41,3c279,is_mosaic,2,1,STABLE_ACROSS_ASDM,"{'uid://A002/X7d6d46/X8c': (''F'',), 'uid://A002/X97c221/X28eb': (''F'',)}"
6,uid://A002/X788a57/X41,3c279,s_resolution,2,2,VARIES_ACROSS_ASDM,"{'uid://A002/X7d6d46/X8c': ('54.066214815704335',), 'uid://A002/X97c221/X28eb': ('1.7846245565974124',)}"
7,uid://A002/X788a57/X41,3c279,spatial_resolution,2,2,VARIES_ACROSS_ASDM,"{'uid://A002/X7d6d46/X8c': ('54.066214815704335',), 'uid://A002/X97c221/X28eb': ('1.7846245565974124',)}"
8,uid://A002/X788a57/X41,3c279,spatial_scale_max,2,2,VARIES_ACROSS_ASDM,"{'uid://A002/X7d6d46/X8c': ('865.0594370512694',), 'uid://A002/X97c221/X28eb': ('10.268247464331386',)}"
9,uid://A002/X788a57/X41,3c279,frequency_support,2,2,VARIES_ACROSS_ASDM,"{'uid://A002/X7d6d46/X8c': (''[99.83..101.82GHz,976.56kHz,24.7mJy/beam@10km/s,1mJy/beam@native, XX YY] U [101.70..10..."


,member_ous_uid,source_name_normalized,asdm_count,raw_source_labels,maximum_position_separation_arcsec,within_1_arcsec_position_tolerance
0,uid://A002/X788a57/X41,3c279,2,"(3C279, 3c279)",0.000547,True


,member_ous_uid,source_name_normalized,asdm_count,raw_source_labels,maximum_position_separation_arcsec,within_1_arcsec_position_tolerance
0,uid://A002/X788a57/X41,3c279,2,"(3C279, 3c279)",0.000547,True


Repeated-source Archive status: SPATIALLY_VERIFIED_REPEATED_SOURCE_ACROSS_ASDM_FOUND


## Step 14 — Archive-wide client-side STC-S family census

`s_region` is an Oracle `MDSYS.SDO_GEOMETRY` value inside the TAP
database. Server-side `LIKE` is therefore invalid even though the TAP
result serializes the value as STC-S text. This corrected experiment
retrieves each already-counted proposal-year stratum, classifies the
serialized geometry in Python, retains only bounded examples, and
releases each large table before continuing.


In [55]:
STCS_EXAMPLE_LIMIT = 40
STCS_STRATUM_SAFETY_LIMIT = OBS_ID_STRATUM_SAFETY_LIMIT


def scalar_is_missing(value: object) -> bool:
    if value is None:
        return True
    try:
        return bool(pd.isna(value))
    except (TypeError, ValueError):
        return False


def stcs_text_or_none(value: object) -> str | None:
    if scalar_is_missing(value):
        return None
    return str(value).strip()


def classify_stcs_family_client(value: object) -> str:
    text = stcs_text_or_none(value)
    if text is None:
        return "MISSING"
    if text == "":
        return "BLANK_EXACT"
    head_match = re.match(r"^([A-Za-z]+)", text)
    if head_match is None:
        return "UNKNOWN_NONBLANK"
    family = head_match.group(1).upper()
    if family in {"CIRCLE", "POLYGON", "UNION"}:
        return family
    return "UNKNOWN_NONBLANK"


stcs_stratum_expected = (
    obs_id_stratum_count_df.set_index("stratum")[
        "expected_rows"
    ].to_dict()
)
stcs_stratum_family_records = []
stcs_example_frames = []
stcs_example_counts = {
    family: 0
    for family in [
        "CIRCLE", "POLYGON", "UNION", "MISSING",
        "BLANK_EXACT", "UNKNOWN_NONBLANK",
    ]
}
stcs_retrieved_total = 0

for stratum, condition in OBS_ID_STRATA.items():
    expected_rows = int(stcs_stratum_expected.get(stratum, 0))
    if expected_rows == 0:
        continue
    if expected_rows > STCS_STRATUM_SAFETY_LIMIT:
        raise RuntimeError(
            f"STC-S stratum {stratum} has {expected_rows} rows, "
            f"above safety limit {STCS_STRATUM_SAFETY_LIMIT}."
        )

    stcs_query = f"""
    SELECT
        proposal_id,
        member_ous_uid,
        asdm_uid,
        obs_id,
        s_region,
        is_mosaic,
        dataproduct_type
    FROM ivoa.obscore
    WHERE science_observation = 'T'
      AND ({condition})
    """
    stcs_run = run_tap_query(
        stcs_query,
        maxrec=expected_rows,
        label=f"Retrieve complete STC-S stratum {stratum}",
    )
    stcs_frame = astropy_table_to_pandas(stcs_run.table)
    query_runs.append(provenance_only_tap_run(stcs_run))
    if (
        "OVERFLOW" in " ".join(stcs_run.query_status)
        or len(stcs_frame) != expected_rows
    ):
        raise RuntimeError(
            f"STC-S stratum {stratum} incomplete: "
            f"expected {expected_rows}, retrieved {len(stcs_frame)}."
        )

    stcs_frame["stcs_family"] = stcs_frame["s_region"].map(
        classify_stcs_family_client
    )
    stratum_counts = (
        stcs_frame["stcs_family"]
        .value_counts(dropna=False)
        .to_dict()
    )
    for family in stcs_example_counts:
        family_rows = int(stratum_counts.get(family, 0))
        stcs_stratum_family_records.append(
            {
                "stratum": stratum,
                "stcs_family": family,
                "archive_rows": family_rows,
            }
        )
        remaining = (
            STCS_EXAMPLE_LIMIT - stcs_example_counts[family]
        )
        if remaining > 0 and family_rows:
            examples = stcs_frame.loc[
                stcs_frame["stcs_family"].eq(family)
            ].head(remaining).copy()
            examples["stcs_census_stratum"] = stratum
            stcs_example_frames.append(examples)
            stcs_example_counts[family] += len(examples)

    stcs_retrieved_total += len(stcs_frame)
    del stcs_frame, stcs_run

stcs_stratum_family_df = pd.DataFrame(
    stcs_stratum_family_records
)
stcs_family_census_df = (
    stcs_stratum_family_df
    .groupby("stcs_family", dropna=False)["archive_rows"]
    .sum()
    .rename("archive_rows")
    .reset_index()
)
stcs_family_census_df["archive_fraction"] = (
    stcs_family_census_df["archive_rows"]
    / obs_id_expected_total
)
stcs_family_examples_df = (
    pd.concat(stcs_example_frames, ignore_index=True)
    if stcs_example_frames
    else pd.DataFrame()
)
stcs_partition_rows = int(
    stcs_family_census_df["archive_rows"].sum()
)
stcs_total_rows = obs_id_expected_total
stcs_partition_complete = (
    stcs_retrieved_total == stcs_total_rows
    and stcs_partition_rows == stcs_total_rows
)

display(stcs_family_census_df)
display(stcs_stratum_family_df)
print("STC-S rows retrieved:", stcs_retrieved_total)
print("STC-S family partition complete:", stcs_partition_complete)



--- Retrieve complete STC-S stratum 2011 ---
Endpoint: https://almascience.eso.org/tap
MAXREC: 3435
ADQL:

    SELECT
        proposal_id,
        member_ous_uid,
        asdm_uid,
        obs_id,
        s_region,
        is_mosaic,
        dataproduct_type
    FROM ivoa.obscore
    WHERE science_observation = 'T'
      AND (proposal_id LIKE '2011.%')
    
Retrieved rows: 3435
QUERY_STATUS: ('OK',)
Warnings: <none>

--- Retrieve complete STC-S stratum 2012 ---
Endpoint: https://almascience.eso.org/tap
MAXREC: 5347
ADQL:

    SELECT
        proposal_id,
        member_ous_uid,
        asdm_uid,
        obs_id,
        s_region,
        is_mosaic,
        dataproduct_type
    FROM ivoa.obscore
    WHERE science_observation = 'T'
      AND (proposal_id LIKE '2012.%')
    
Retrieved rows: 5347
QUERY_STATUS: ('OK',)
Warnings: <none>

--- Retrieve complete STC-S stratum 2013 ---
Endpoint: https://almascience.eso.org/tap
MAXREC: 16802
ADQL:

    SELECT
        proposal_id,
        member_ou

,stcs_family,archive_rows,archive_fraction
0,BLANK_EXACT,0,0.000000
1,CIRCLE,194500,0.439541
2,MISSING,0,0.000000
3,POLYGON,245655,0.555144
4,UNION,2352,0.005315
5,UNKNOWN_NONBLANK,0,0.000000


,stratum,stcs_family,archive_rows
0,2011,CIRCLE,3084
1,2011,POLYGON,267
2,2011,UNION,84
3,2011,MISSING,0
4,2011,BLANK_EXACT,0
...,...,...,...
73,2025,POLYGON,35299
74,2025,UNION,112
75,2025,MISSING,0
76,2025,BLANK_EXACT,0


STC-S rows retrieved: 442507
STC-S family partition complete: True


In [56]:
stcs_structural_records = []
if len(stcs_family_examples_df):
    for _, row in stcs_family_examples_df.iterrows():
        parsed_geometry = parse_stcs_structure(row["s_region"])
        stcs_structural_records.append(
            {
                "proposal_id": row.get("proposal_id"),
                "member_ous_uid": row.get("member_ous_uid"),
                "asdm_uid": row.get("asdm_uid"),
                "obs_id": row.get("obs_id"),
                "is_mosaic": row.get("is_mosaic"),
                "dataproduct_type": row.get("dataproduct_type"),
                "census_family": row.get("stcs_family"),
                **parsed_geometry,
            }
        )
stcs_structural_example_df = pd.DataFrame(
    stcs_structural_records
)

if len(stcs_structural_example_df):
    parsed_family_column = (
        "geometry_grammar"
        if "geometry_grammar" in stcs_structural_example_df.columns
        else "geometry_type"
    )
    stcs_structural_example_df["family_matches_census"] = (
        stcs_structural_example_df[parsed_family_column]
        .astype("string")
        .str.upper()
        .eq(
            stcs_structural_example_df["census_family"]
            .astype("string")
            .str.upper()
        )
    )
    stcs_structure_exception_df = (
        stcs_structural_example_df.loc[
            ~stcs_structural_example_df[
                "family_matches_census"
            ].fillna(False)
            | stcs_structural_example_df[
                "geometry_issue_codes"
            ].map(bool)
        ].copy()
    )
    stcs_frame_summary_df = (
        stcs_structural_example_df
        .groupby(
            ["census_family", "coordinate_frame"],
            dropna=False,
        )
        .size()
        .rename("bounded_example_rows")
        .reset_index()
    )
else:
    stcs_structure_exception_df = pd.DataFrame()
    stcs_frame_summary_df = pd.DataFrame()

display(stcs_frame_summary_df)
display(stcs_structural_example_df)
display(stcs_structure_exception_df)


,census_family,coordinate_frame,bounded_example_rows
0,CIRCLE,ICRS,40
1,POLYGON,ICRS,40
2,UNION,ICRS,40


,proposal_id,member_ous_uid,asdm_uid,obs_id,is_mosaic,dataproduct_type,census_family,geometry_type,coordinate_frame,geometry_component_count,child_geometry_types,numeric_token_count,parentheses_balanced,geometry_parse_status,geometry_issue_codes,normalized_geometry_hash,numeric_range_wrap_candidate_heuristic,family_matches_census
0,2011.0.00957.S,uid://A001/X74/X287,uid://A002/X321719/Xb2f,uid://A001/X74/X287.source.SPT_0319-47.spw.17,F,cube,CIRCLE,CIRCLE,ICRS,1,"(CIRCLE,)",3,True,PARSED_STRUCTURALLY,(),2f8107d44af9765e34e7f39db4f3684adb8f1818bbe4a32e5bab81b8c3de1940,False,True
1,2011.0.00957.S,uid://A001/X74/X287,uid://A002/X321719/Xb2f,uid://A001/X74/X287.source.SPT_0319-47.spw.19,F,cube,CIRCLE,CIRCLE,ICRS,1,"(CIRCLE,)",3,True,PARSED_STRUCTURALLY,(),2f8107d44af9765e34e7f39db4f3684adb8f1818bbe4a32e5bab81b8c3de1940,False,True
2,2011.0.00957.S,uid://A001/X74/X287,uid://A002/X321719/Xb2f,uid://A001/X74/X287.source.SPT_0319-47.spw.21,F,cube,CIRCLE,CIRCLE,ICRS,1,"(CIRCLE,)",3,True,PARSED_STRUCTURALLY,(),2f8107d44af9765e34e7f39db4f3684adb8f1818bbe4a32e5bab81b8c3de1940,False,True
3,2011.0.00957.S,uid://A001/X74/X287,uid://A002/X321719/Xb2f,uid://A001/X74/X287.source.SPT_0319-47.spw.23,F,cube,CIRCLE,CIRCLE,ICRS,1,"(CIRCLE,)",3,True,PARSED_STRUCTURALLY,(),2f8107d44af9765e34e7f39db4f3684adb8f1818bbe4a32e5bab81b8c3de1940,False,True
4,2011.0.00957.S,uid://A001/X74/X287,uid://A002/X321719/Xb2f,uid://A001/X74/X287.source.SPT_0346-52.spw.17,F,cube,CIRCLE,CIRCLE,ICRS,1,"(CIRCLE,)",3,True,PARSED_STRUCTURALLY,(),03baf1f520efad870058d1b12cf34cbd99144a91cea7955ed7aaeaae7bf0c4f0,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,2011.0.00539.S,uid://A002/X391d0b/X16f,uid://A002/X47bd4d/X98b,uid://A002/X391d0b/X16f.source.ADFS01.spw.15,T,image,UNION,UNION,ICRS,8,"(POLYGON, POLYGON, POLYGON, POLYGON, POLYGON, POLYGON, POLYGON, POLYGON)",320,True,PARSED_STRUCTURALLY,(),51484aeb6c9217fbec52bf40ad34284a088280883ecdd8db2059ca84b2f84ab4,False,True
116,2011.0.00779.S,uid://A002/X391d0b/X187,uid://A002/X513d18/X261,uid://A002/X391d0b/X187.source.Io.spw.17,T,cube,UNION,UNION,ICRS,2,"(POLYGON, POLYGON)",80,True,PARSED_STRUCTURALLY,(),d75bc59aba24547d30022d047669771f1a885cd3b53c80f11c22296c5bb2de50,False,True
117,2011.0.00779.S,uid://A002/X391d0b/X187,uid://A002/X513d18/X261,uid://A002/X391d0b/X187.source.Io.spw.19,T,cube,UNION,UNION,ICRS,2,"(POLYGON, POLYGON)",80,True,PARSED_STRUCTURALLY,(),d75bc59aba24547d30022d047669771f1a885cd3b53c80f11c22296c5bb2de50,False,True
118,2011.0.00779.S,uid://A002/X391d0b/X187,uid://A002/X513d18/X261,uid://A002/X391d0b/X187.source.Io.spw.21,T,cube,UNION,UNION,ICRS,2,"(POLYGON, POLYGON)",80,True,PARSED_STRUCTURALLY,(),d75bc59aba24547d30022d047669771f1a885cd3b53c80f11c22296c5bb2de50,False,True


,proposal_id,member_ous_uid,asdm_uid,obs_id,is_mosaic,dataproduct_type,census_family,geometry_type,coordinate_frame,geometry_component_count,child_geometry_types,numeric_token_count,parentheses_balanced,geometry_parse_status,geometry_issue_codes,normalized_geometry_hash,numeric_range_wrap_candidate_heuristic,family_matches_census


## Step 15 — Archive-wide product and resolution diagnostics

This step completes product granularity after the identifier census
disproved both `obs_id` and Source-Execution-SPW as universal row keys.
Returned `obs_id` values at the 64-character boundary are not reused in
ADQL equality predicates. Duplicate detail is retrieved by complete
affected Members, then filtered and classified client-side. This
distinguishes reliable below-width product multiplicity from possible
identifier-width collisions.


In [57]:
product_combination_query = """
SELECT
    dataproduct_type,
    calib_level,
    access_format,
    COUNT(*) AS archive_row_count
FROM ivoa.obscore
WHERE science_observation = 'T'
GROUP BY dataproduct_type, calib_level, access_format
"""
product_combination_run = run_tap_query(
    product_combination_query,
    maxrec=1_000,
    label="Archive-wide product/calibration/access-format census",
)
query_runs.append(product_combination_run)
if "OVERFLOW" in " ".join(product_combination_run.query_status):
    raise RuntimeError(
        "Product-combination census overflowed its 1,000-row limit."
    )
archive_product_combination_df = astropy_table_to_pandas(
    product_combination_run.table
)

PRODUCT_OPTIONAL_FIELDS = [
    "access_estsize", "em_xel", "pol_xel",
    "s_xel1", "s_xel2", "t_xel",
]
archive_product_missingness_records = []
for field in PRODUCT_OPTIONAL_FIELDS:
    if field not in available_columns:
        archive_product_missingness_records.append(
            {
                "field": field,
                "archive_row_count": obs_id_expected_total,
                "null_count": np.nan,
                "available_count": np.nan,
                "status": "COLUMN_NOT_AVAILABLE",
            }
        )
        continue
    null_count_query = f"""
    SELECT COUNT(*) AS total_rows
    FROM ivoa.obscore
    WHERE science_observation = 'T'
      AND {field} IS NULL
    """
    null_count_run = run_tap_query(
        null_count_query,
        maxrec=10,
        label=f"Count Archive-wide NULL values for {field}",
    )
    query_runs.append(null_count_run)
    null_count = int(null_count_run.table[0]["total_rows"])
    archive_product_missingness_records.append(
        {
            "field": field,
            "archive_row_count": obs_id_expected_total,
            "null_count": null_count,
            "available_count": obs_id_expected_total - null_count,
            "status": "COUNTED",
        }
    )

archive_product_missingness_df = pd.DataFrame(
    archive_product_missingness_records
)
product_census_complete = (
    not archive_product_combination_df.empty
    and archive_product_missingness_df["status"].isin(
        ["COUNTED", "COLUMN_NOT_AVAILABLE"]
    ).all()
)

display(archive_product_combination_df)
display(archive_product_missingness_df)



--- Archive-wide product/calibration/access-format census ---
Endpoint: https://almascience.eso.org/tap
MAXREC: 1000
ADQL:

SELECT
    dataproduct_type,
    calib_level,
    access_format,
    COUNT(*) AS archive_row_count
FROM ivoa.obscore
WHERE science_observation = 'T'
GROUP BY dataproduct_type, calib_level, access_format

Retrieved rows: 2
QUERY_STATUS: ('OK',)
Warnings: <none>

--- Count Archive-wide NULL values for access_estsize ---
Endpoint: https://almascience.eso.org/tap
MAXREC: 10
ADQL:

    SELECT COUNT(*) AS total_rows
    FROM ivoa.obscore
    WHERE science_observation = 'T'
      AND access_estsize IS NULL
    
Retrieved rows: 1
QUERY_STATUS: ('OK',)
Warnings: <none>

--- Count Archive-wide NULL values for em_xel ---
Endpoint: https://almascience.eso.org/tap
MAXREC: 10
ADQL:

    SELECT COUNT(*) AS total_rows
    FROM ivoa.obscore
    WHERE science_observation = 'T'
      AND em_xel IS NULL
    
Retrieved rows: 1
QUERY_STATUS: ('OK',)
Warnings: <none>

--- Count Archive

,dataproduct_type,calib_level,access_format,archive_row_count
0,cube,2,applicati,305618
1,image,2,applicati,136889


,field,archive_row_count,null_count,available_count,status
0,access_estsize,442507,442507,0,COUNTED
1,em_xel,442507,0,442507,COUNTED
2,pol_xel,442507,0,442507,COUNTED
3,s_xel1,442507,442507,0,COUNTED
4,s_xel2,442507,442507,0,COUNTED
5,t_xel,442507,0,442507,COUNTED


In [58]:
PRODUCT_DUPLICATE_MEMBER_SAFETY_LIMIT = 50_000
product_key_columns = [
    "member_ous_uid", "asdm_uid", "source_name", "spw_id"
]
product_duplicate_member_uids = (
    duplicate_source_execution_spw_group_df["member_ous_uid"]
    .dropna()
    .astype(str)
    .drop_duplicates()
    .tolist()
)

if product_duplicate_member_uids:
    product_duplicate_member_sql = ",\n    ".join(
        quote_adql_string(uid)
        for uid in product_duplicate_member_uids
    )
    product_member_count_query = f"""
    SELECT COUNT(*) AS total_rows
    FROM ivoa.obscore
    WHERE science_observation = 'T'
      AND member_ous_uid IN (
        {product_duplicate_member_sql}
      )
    """
    product_member_count_run = run_tap_query(
        product_member_count_query,
        maxrec=10,
        label="Count Members containing composite-key candidates",
    )
    query_runs.append(product_member_count_run)
    product_member_expected_rows = int(
        product_member_count_run.table[0]["total_rows"]
    )
    if (
        product_member_expected_rows
        > PRODUCT_DUPLICATE_MEMBER_SAFETY_LIMIT
    ):
        raise RuntimeError(
            "Composite-candidate Member population exceeds safety "
            f"limit: {product_member_expected_rows}."
        )

    product_duplicate_columns = [
        name
        for name in [
            "proposal_id", "obs_publisher_did", "member_ous_uid",
            "asdm_uid", "obs_id", "dataproduct_type", "calib_level",
            "type", "access_format", "access_estsize", "em_xel",
            "pol_xel", "s_xel1", "s_xel2", "t_xel", "frequency",
            "bandwidth", "frequency_support",
        ]
        if name in available_columns
    ]
    product_duplicate_column_sql = ",\n    ".join(
        product_duplicate_columns
    )
    product_member_query = f"""
    SELECT
        {product_duplicate_column_sql}
    FROM ivoa.obscore
    WHERE science_observation = 'T'
      AND member_ous_uid IN (
        {product_duplicate_member_sql}
      )
    """
    product_member_run = run_tap_query(
        product_member_query,
        maxrec=max(product_member_expected_rows, 1),
        label="Retrieve Members containing composite-key candidates",
    )
    product_duplicate_member_population_df = astropy_table_to_pandas(
        product_member_run.table
    )
    query_runs.append(provenance_only_tap_run(product_member_run))
    product_member_retrieval_complete = (
        "OVERFLOW" not in " ".join(product_member_run.query_status)
        and len(product_duplicate_member_population_df)
        == product_member_expected_rows
    )
    if not product_member_retrieval_complete:
        raise RuntimeError(
            "Composite-candidate Member retrieval is incomplete."
        )

    product_duplicate_parsed_df = pd.DataFrame(
        product_duplicate_member_population_df["obs_id"]
        .map(parse_obs_id_structure)
        .tolist()
    )
    for parsed_column in product_duplicate_parsed_df.columns:
        product_duplicate_member_population_df[parsed_column] = (
            product_duplicate_parsed_df[parsed_column].to_numpy()
        )
    product_duplicate_member_population_df["obs_id_length"] = (
        product_duplicate_member_population_df["obs_id"]
        .astype("string")
        .str.len()
    )
    product_duplicate_member_population_df[
        "obs_id_at_declared_width"
    ] = product_duplicate_member_population_df["obs_id_length"].eq(
        OBS_ID_DECLARED_WIDTH
    )

    candidate_key_evidence_df = (
        duplicate_source_execution_spw_group_df[
            product_key_columns
            + ["identifier_evidence_class", "width_boundary_rows"]
        ]
        .drop_duplicates()
    )
    product_duplicate_detail_df = (
        product_duplicate_member_population_df.merge(
            candidate_key_evidence_df,
            on=product_key_columns,
            how="inner",
            validate="many_to_one",
        )
    )

    retrieved_candidate_key_df = (
        product_duplicate_detail_df[product_key_columns]
        .drop_duplicates()
    )
    missing_product_candidate_key_df = (
        candidate_key_evidence_df.merge(
            retrieved_candidate_key_df,
            on=product_key_columns,
            how="left",
            indicator=True,
        )
        .loc[lambda frame: frame["_merge"].eq("left_only")]
        .drop(columns="_merge")
    )

    product_discriminator_columns = [
        column
        for column in [
            "obs_publisher_did", "dataproduct_type", "calib_level",
            "type", "access_format", "em_xel", "pol_xel", "t_xel",
            "frequency", "bandwidth",
        ]
        if column in product_duplicate_detail_df.columns
    ]
    product_group_records = []
    for key_values, group in product_duplicate_detail_df.groupby(
        product_key_columns,
        dropna=False,
        sort=True,
    ):
        discriminator_variants = len(
            group[product_discriminator_columns]
            .astype("string")
            .drop_duplicates()
        )
        width_boundary = bool(
            group["obs_id_at_declared_width"].any()
            or group["width_boundary_rows"].gt(0).any()
        )
        publisher_id_count = group[
            "obs_publisher_did"
        ].nunique(dropna=True)
        if width_boundary:
            evidence_class = (
                "UNRESOLVED_OBS_ID_WIDTH_BOUNDARY_COLLISION"
            )
        elif len(group) > 1 and (
            publisher_id_count > 1 or discriminator_variants > 1
        ):
            evidence_class = "VERIFIED_MULTIPLE_ARCHIVE_PRODUCTS"
        elif len(group) > 1:
            evidence_class = "REPEATED_INDISTINGUISHABLE_ROWS"
        else:
            evidence_class = "NOT_REPRODUCED_IN_MEMBER_RETRIEVAL"

        product_group_records.append(
            {
                **dict(zip(product_key_columns, key_values, strict=True)),
                "archive_rows": len(group),
                "obs_id_at_declared_width": width_boundary,
                "publisher_id_count": publisher_id_count,
                "product_discriminator_variants": discriminator_variants,
                "product_types": tuple(
                    sorted(
                        set(
                            group["dataproduct_type"]
                            .dropna()
                            .astype(str)
                        )
                    )
                ),
                "calibration_levels": tuple(
                    sorted(set(group["calib_level"].dropna()))
                ),
                "access_formats": tuple(
                    sorted(
                        set(
                            group["access_format"]
                            .dropna()
                            .astype(str)
                        )
                    )
                ),
                "product_evidence_class": evidence_class,
            }
        )
    product_duplicate_group_df = pd.DataFrame(product_group_records)
else:
    product_member_expected_rows = 0
    product_member_retrieval_complete = True
    product_duplicate_member_population_df = pd.DataFrame()
    product_duplicate_detail_df = pd.DataFrame()
    product_duplicate_group_df = pd.DataFrame()
    missing_product_candidate_key_df = pd.DataFrame()

reliable_product_multiplicity_group_df = (
    product_duplicate_group_df.loc[
        product_duplicate_group_df["product_evidence_class"].eq(
            "VERIFIED_MULTIPLE_ARCHIVE_PRODUCTS"
        )
    ].copy()
    if len(product_duplicate_group_df)
    else pd.DataFrame()
)
width_boundary_product_collision_df = (
    product_duplicate_group_df.loc[
        product_duplicate_group_df["product_evidence_class"].eq(
            "UNRESOLVED_OBS_ID_WIDTH_BOUNDARY_COLLISION"
        )
    ].copy()
    if len(product_duplicate_group_df)
    else pd.DataFrame()
)
product_duplicate_detail_complete = bool(
    product_member_retrieval_complete
    and missing_product_candidate_key_df.empty
)
archive_product_granularity_status = (
    "VERIFIED_MULTIPLE_PRODUCTS_PER_SOURCE_EXECUTION_SPW"
    if len(reliable_product_multiplicity_group_df)
    else (
        "COMPOSITE_DUPLICATES_UNRESOLVED_DUE_OBS_ID_WIDTH_BOUNDARY"
        if len(width_boundary_product_collision_df)
        else "NO_RELIABLE_MULTIPLE_PRODUCT_GROUPS_OBSERVED"
    )
)

print("Archive product granularity status:", archive_product_granularity_status)
display(product_duplicate_group_df)
display(reliable_product_multiplicity_group_df)
display(width_boundary_product_collision_df)
display(missing_product_candidate_key_df)
display(product_duplicate_detail_df.head(200))



--- Count Members containing composite-key candidates ---
Endpoint: https://almascience.eso.org/tap
MAXREC: 10
ADQL:

    SELECT COUNT(*) AS total_rows
    FROM ivoa.obscore
    WHERE science_observation = 'T'
      AND member_ous_uid IN (
        'uid://A001/X133d/X27a9',
    'uid://A001/X133d/Xe97',
    'uid://A001/X133d/Xe9b',
    'uid://A001/X1465/X5f0',
    'uid://A001/X1465/X5f2',
    'uid://A001/X1465/X5f6',
    'uid://A001/X1465/X5f8',
    'uid://A001/X14d8/X2c4',
    'uid://A001/X14d8/X365',
    'uid://A001/X1590/X344b',
    'uid://A001/X1590/X344f',
    'uid://A001/X2d20/X21c1',
    'uid://A001/X2d6/X175',
    'uid://A001/X2d6/X69',
    'uid://A001/X3621/X1dd1',
    'uid://A001/X362e/X4d0',
    'uid://A001/X365b/X136',
    'uid://A001/X365b/X21e',
    'uid://A001/X3788/X563d',
    'uid://A001/X3788/X5641',
    'uid://A001/X3788/X5645',
    'uid://A001/X3788/X9703',
    'uid://A001/X3788/X971e',
    'uid://A001/X3788/Xc197',
    'uid://A001/X3788/Xc19e',
    'uid://A001/X3788

,member_ous_uid,asdm_uid,source_name,spw_id,archive_rows,obs_id_at_declared_width,publisher_id_count,product_discriminator_variants,product_types,calibration_levels,access_formats,product_evidence_class
0,uid://A001/X133d/X27a9,uid://A002/Xdd9a29/X174,Northeast_Section_of_NGC6334,1,2,True,1,2,"(cube,)","(2,)","(applicati,)",UNRESOLVED_OBS_ID_WIDTH_BOUNDARY_COLLISION
1,uid://A001/X133d/X27a9,uid://A002/Xdd9a29/X174,Northeast_Section_of_NGC6334,2,5,True,1,5,"(cube, image)","(2,)","(applicati,)",UNRESOLVED_OBS_ID_WIDTH_BOUNDARY_COLLISION
2,uid://A001/X133d/Xe97,uid://A002/Xd7aa27/X128,587731511545233578_8082-12701,2,3,True,1,3,"(cube, image)","(2,)","(applicati,)",UNRESOLVED_OBS_ID_WIDTH_BOUNDARY_COLLISION
3,uid://A001/X133d/Xe9b,uid://A002/Xd81670/X6f13,587731174919438555_8615-12702,2,3,True,1,3,"(cube, image)","(2,)","(applicati,)",UNRESOLVED_OBS_ID_WIDTH_BOUNDARY_COLLISION
4,uid://A001/X1465/X5f0,uid://A002/Xeba1ac/X40ca,eso137-001_-_south_outer_tail,2,3,True,1,3,"(cube,)","(2,)","(applicati,)",UNRESOLVED_OBS_ID_WIDTH_BOUNDARY_COLLISION
5,uid://A001/X1465/X5f2,uid://A002/Xe48598/X19681,eso137-001_-_south_outer_tail,2,3,True,1,3,"(cube,)","(2,)","(applicati,)",UNRESOLVED_OBS_ID_WIDTH_BOUNDARY_COLLISION
6,uid://A001/X1465/X5f6,uid://A002/Xeaff9c/X10b7,eso137-001_-_central_filament,2,3,True,1,3,"(cube,)","(2,)","(applicati,)",UNRESOLVED_OBS_ID_WIDTH_BOUNDARY_COLLISION
7,uid://A001/X1465/X5f8,uid://A002/Xe48598/X18d22,eso137-001_-_central_filament,2,3,True,1,3,"(cube,)","(2,)","(applicati,)",UNRESOLVED_OBS_ID_WIDTH_BOUNDARY_COLLISION
8,uid://A001/X14d8/X2c4,uid://A002/Xe5ed76/X1bb1,2MASS_J04215402+1530299_OFF_0,1,2,True,1,2,"(cube,)","(2,)","(applicati,)",UNRESOLVED_OBS_ID_WIDTH_BOUNDARY_COLLISION
9,uid://A001/X14d8/X2c4,uid://A002/Xe5ed76/X1bb1,2MASS_J04215402+1530299_OFF_0,2,3,True,1,3,"(cube,)","(2,)","(applicati,)",UNRESOLVED_OBS_ID_WIDTH_BOUNDARY_COLLISION


,member_ous_uid,asdm_uid,source_name,spw_id,archive_rows,obs_id_at_declared_width,publisher_id_count,product_discriminator_variants,product_types,calibration_levels,access_formats,product_evidence_class


,member_ous_uid,asdm_uid,source_name,spw_id,archive_rows,obs_id_at_declared_width,publisher_id_count,product_discriminator_variants,product_types,calibration_levels,access_formats,product_evidence_class
0,uid://A001/X133d/X27a9,uid://A002/Xdd9a29/X174,Northeast_Section_of_NGC6334,1,2,True,1,2,"(cube,)","(2,)","(applicati,)",UNRESOLVED_OBS_ID_WIDTH_BOUNDARY_COLLISION
1,uid://A001/X133d/X27a9,uid://A002/Xdd9a29/X174,Northeast_Section_of_NGC6334,2,5,True,1,5,"(cube, image)","(2,)","(applicati,)",UNRESOLVED_OBS_ID_WIDTH_BOUNDARY_COLLISION
2,uid://A001/X133d/Xe97,uid://A002/Xd7aa27/X128,587731511545233578_8082-12701,2,3,True,1,3,"(cube, image)","(2,)","(applicati,)",UNRESOLVED_OBS_ID_WIDTH_BOUNDARY_COLLISION
3,uid://A001/X133d/Xe9b,uid://A002/Xd81670/X6f13,587731174919438555_8615-12702,2,3,True,1,3,"(cube, image)","(2,)","(applicati,)",UNRESOLVED_OBS_ID_WIDTH_BOUNDARY_COLLISION
4,uid://A001/X1465/X5f0,uid://A002/Xeba1ac/X40ca,eso137-001_-_south_outer_tail,2,3,True,1,3,"(cube,)","(2,)","(applicati,)",UNRESOLVED_OBS_ID_WIDTH_BOUNDARY_COLLISION
5,uid://A001/X1465/X5f2,uid://A002/Xe48598/X19681,eso137-001_-_south_outer_tail,2,3,True,1,3,"(cube,)","(2,)","(applicati,)",UNRESOLVED_OBS_ID_WIDTH_BOUNDARY_COLLISION
6,uid://A001/X1465/X5f6,uid://A002/Xeaff9c/X10b7,eso137-001_-_central_filament,2,3,True,1,3,"(cube,)","(2,)","(applicati,)",UNRESOLVED_OBS_ID_WIDTH_BOUNDARY_COLLISION
7,uid://A001/X1465/X5f8,uid://A002/Xe48598/X18d22,eso137-001_-_central_filament,2,3,True,1,3,"(cube,)","(2,)","(applicati,)",UNRESOLVED_OBS_ID_WIDTH_BOUNDARY_COLLISION
8,uid://A001/X14d8/X2c4,uid://A002/Xe5ed76/X1bb1,2MASS_J04215402+1530299_OFF_0,1,2,True,1,2,"(cube,)","(2,)","(applicati,)",UNRESOLVED_OBS_ID_WIDTH_BOUNDARY_COLLISION
9,uid://A001/X14d8/X2c4,uid://A002/Xe5ed76/X1bb1,2MASS_J04215402+1530299_OFF_0,2,3,True,1,3,"(cube,)","(2,)","(applicati,)",UNRESOLVED_OBS_ID_WIDTH_BOUNDARY_COLLISION


,member_ous_uid,asdm_uid,source_name,spw_id,identifier_evidence_class,width_boundary_rows


,proposal_id,obs_publisher_did,member_ous_uid,asdm_uid,obs_id,dataproduct_type,calib_level,type,access_format,access_estsize,em_xel,pol_xel,s_xel1,s_xel2,t_xel,frequency,bandwidth,frequency_support,obs_member_ous_uid,source_name,spw_id,obs_id_parse_status,obs_id_parse_issue,obs_id_length,obs_id_at_declared_width,identifier_evidence_class,width_boundary_rows
0,2022.1.00716.S,ADS/JAO.ALMA#2022.1.00716.S,uid://A001/X2d20/X21c1,uid://A002/X10239e1/X17012,uid://A001/X2d20/X21c1.source.GALEXASC_J103804.55-284333.9.spw.1,cube,2,S,applicati,<NA>,2048,2,<NA>,<NA>,1,99.548014,2.000000e+09,"[98.55..100.55GHz,1128.91kHz,15.2mJy/beam@10km/s,618.9uJy/beam@native, XX YY] U [100.51..102.51GHz,1128.91kHz,14.9mJ...",uid://A001/X2d20/X21c1,GALEXASC_J103804.55-284333.9,1,PARSED,None,64,True,WIDTH_BOUNDARY_COLLISION_CANDIDATE,3
1,2022.1.00716.S,ADS/JAO.ALMA#2022.1.00716.S,uid://A001/X2d20/X21c1,uid://A002/X10239e1/X17012,uid://A001/X2d20/X21c1.source.GALEXASC_J103804.55-284333.9.spw.1,cube,2,S,applicati,<NA>,2048,2,<NA>,<NA>,1,101.505946,2.000000e+09,"[98.55..100.55GHz,1128.91kHz,15.2mJy/beam@10km/s,618.9uJy/beam@native, XX YY] U [100.51..102.51GHz,1128.91kHz,14.9mJ...",uid://A001/X2d20/X21c1,GALEXASC_J103804.55-284333.9,1,PARSED,None,64,True,WIDTH_BOUNDARY_COLLISION_CANDIDATE,3
2,2022.1.00716.S,ADS/JAO.ALMA#2022.1.00716.S,uid://A001/X2d20/X21c1,uid://A002/X10239e1/X17012,uid://A001/X2d20/X21c1.source.GALEXASC_J103804.55-284333.9.spw.1,cube,2,S,applicati,<NA>,2048,2,<NA>,<NA>,1,111.554421,2.000000e+09,"[98.55..100.55GHz,1128.91kHz,15.2mJy/beam@10km/s,618.9uJy/beam@native, XX YY] U [100.51..102.51GHz,1128.91kHz,14.9mJ...",uid://A001/X2d20/X21c1,GALEXASC_J103804.55-284333.9,1,PARSED,None,64,True,WIDTH_BOUNDARY_COLLISION_CANDIDATE,3
3,2023.1.00127.L,ADS/JAO.ALMA#2023.1.00127.L,uid://A001/X362e/X4d0,uid://A002/X112077c/X37700,uid://A001/X362e/X4d0.source.Q1211+1030_z0p8999_CO43_band6.spw.1,image,2,L,applicati,<NA>,128,2,<NA>,<NA>,1,230.990168,2.000000e+09,"[228.00..229.98GHz,31250.00kHz,765.2uJy/beam@10km/s,47.3uJy/beam@native, XX YY] U [230.00..231.98GHz,31250.00kHz,814...",uid://A001/X362e/X4d0,Q1211+1030_z0p8999_CO43_band6,1,PARSED,None,64,True,WIDTH_BOUNDARY_COLLISION_CANDIDATE,2
4,2023.1.00127.L,ADS/JAO.ALMA#2023.1.00127.L,uid://A001/X362e/X4d0,uid://A002/X112077c/X37700,uid://A001/X362e/X4d0.source.Q1211+1030_z0p8999_CO43_band6.spw.2,image,2,L,applicati,<NA>,128,2,<NA>,<NA>,1,244.598048,2.000000e+09,"[228.00..229.98GHz,31250.00kHz,765.2uJy/beam@10km/s,47.3uJy/beam@native, XX YY] U [230.00..231.98GHz,31250.00kHz,814...",uid://A001/X362e/X4d0,Q1211+1030_z0p8999_CO43_band6,2,PARSED,None,64,True,WIDTH_BOUNDARY_COLLISION_CANDIDATE,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109,2021.1.00094.S,ADS/JAO.ALMA#2021.1.00094.S,uid://A001/X1590/X344b,uid://A002/Xf5c3d2/X19bd,uid://A001/X1590/X344b.source.587731512617664623_8154-9102.spw.2,image,2,S,applicati,<NA>,128,2,<NA>,<NA>,1,110.422937,2.000000e+09,"[97.47..99.46GHz,31250.00kHz,795.6uJy/beam@10km/s,32.2uJy/beam@native, XX YY] U [99.22..101.21GHz,31250.00kHz,782.5u...",uid://A001/X1590/X344b,587731512617664623_8154-9102,2,PARSED,None,64,True,WIDTH_BOUNDARY_COLLISION_CANDIDATE,2
110,2021.1.00094.S,ADS/JAO.ALMA#2021.1.00094.S,uid://A001/X1590/X344b,uid://A002/Xf5c3d2/X19bd,uid://A001/X1590/X344b.source.587731512617664623_8154-9102.spw.2,cube,2,S,applicati,<NA>,3840,2,<NA>,<NA>,1,112.152656,1.875000e+09,"[97.47..99.46GHz,31250.00kHz,795.6uJy/beam@10km/s,32.2uJy/beam@native, XX YY] U [99.22..101.21GHz,31250.00kHz,782.5u...",uid://A001/X1590/X344b,587731512617664623_8154-9102,2,PARSED,None,64,True,WIDTH_BOUNDARY_COLLISION_CANDIDATE,2
111,2021.1.00094.S,ADS/JAO.ALMA#2021.1.00094.S,uid://A001/X1590/X344b,uid://A002/Xf5c3d2/X19bd,uid://A001/X1590/X344b.source.587731512617664623_8154-9102.spw.1,image,2,S,applicati,<NA>,128,2,<NA>,<NA>,1,98.463422,2.000000e+09,"[97.47..99.46GHz,31250.00kHz,795.6uJy/beam@10km/s,32.2uJy/beam@native, XX YY] U [99.

In [59]:
RESOLUTION_MISMATCH_EXAMPLE_LIMIT = 500
RESOLUTION_NUMERICAL_ATOL_ARCSEC = 1e-6
RESOLUTION_NUMERICAL_RTOL = 1e-9

resolution_comparison_population_query = """
SELECT COUNT(*) AS total_rows
FROM ivoa.obscore
WHERE science_observation = 'T'
  AND s_resolution IS NOT NULL
  AND spatial_resolution IS NOT NULL
"""
resolution_comparison_population_run = run_tap_query(
    resolution_comparison_population_query,
    maxrec=10,
    label="Count Archive-wide comparable spatial resolutions",
)
query_runs.append(resolution_comparison_population_run)
resolution_comparable_rows = int(
    resolution_comparison_population_run.table[0]["total_rows"]
)

resolution_mismatch_count_query = """
SELECT COUNT(*) AS total_rows
FROM ivoa.obscore
WHERE science_observation = 'T'
  AND s_resolution IS NOT NULL
  AND spatial_resolution IS NOT NULL
  AND s_resolution <> spatial_resolution
"""
resolution_mismatch_count_run = run_tap_query(
    resolution_mismatch_count_query,
    maxrec=10,
    label="Count Archive-wide exact spatial-resolution mismatches",
)
query_runs.append(resolution_mismatch_count_run)
resolution_mismatch_rows = int(
    resolution_mismatch_count_run.table[0]["total_rows"]
)

resolution_example_limit = min(
    resolution_mismatch_rows,
    RESOLUTION_MISMATCH_EXAMPLE_LIMIT,
)
if resolution_example_limit:
    resolution_mismatch_example_query = f"""
    SELECT TOP {resolution_example_limit}
        proposal_id,
        member_ous_uid,
        asdm_uid,
        obs_id,
        s_resolution,
        spatial_resolution,
        antenna_arrays,
        dataproduct_type,
        frequency_support,
        is_mosaic,
        type
    FROM ivoa.obscore
    WHERE science_observation = 'T'
      AND s_resolution IS NOT NULL
      AND spatial_resolution IS NOT NULL
      AND s_resolution <> spatial_resolution
    ORDER BY proposal_id DESC, member_ous_uid, obs_id
    """
    resolution_mismatch_example_run = run_tap_query(
        resolution_mismatch_example_query,
        maxrec=resolution_example_limit,
        label="Retrieve spatial-resolution mismatch examples",
    )
    query_runs.append(resolution_mismatch_example_run)
    resolution_mismatch_example_df = astropy_table_to_pandas(
        resolution_mismatch_example_run.table
    )
    resolution_mismatch_example_df["resolution_difference_arcsec"] = (
        resolution_mismatch_example_df["s_resolution"]
        - resolution_mismatch_example_df["spatial_resolution"]
    )
    resolution_mismatch_example_df[
        "absolute_resolution_difference_arcsec"
    ] = resolution_mismatch_example_df[
        "resolution_difference_arcsec"
    ].abs()
    resolution_mismatch_example_df[
        "numerically_equal_with_tolerance"
    ] = np.isclose(
        resolution_mismatch_example_df["s_resolution"],
        resolution_mismatch_example_df["spatial_resolution"],
        rtol=RESOLUTION_NUMERICAL_RTOL,
        atol=RESOLUTION_NUMERICAL_ATOL_ARCSEC,
        equal_nan=False,
    )
    resolution_mismatch_example_df["difference_class"] = np.where(
        resolution_mismatch_example_df[
            "numerically_equal_with_tolerance"
        ],
        "NUMERICAL_ONLY",
        "MATERIAL_DIFFERENCE",
    )
    resolution_mismatch_example_df["antenna_prefixes"] = (
        resolution_mismatch_example_df["antenna_arrays"]
        .map(antenna_prefixes)
    )
    resolution_mismatch_example_df["array_evidence_class"] = (
        resolution_mismatch_example_df["antenna_prefixes"]
        .map(classify_array_evidence)
    )
    resolution_mismatch_example_df["support_grammar_family"] = (
        resolution_mismatch_example_df["frequency_support"]
        .map(classify_support_grammar)
    )
else:
    resolution_mismatch_example_df = pd.DataFrame()

resolution_mismatch_summary_df = (
    resolution_mismatch_example_df
    .groupby(
        [
            "difference_class", "array_evidence_class",
            "dataproduct_type", "support_grammar_family",
            "is_mosaic", "type",
        ],
        dropna=False,
    )
    .agg(
        example_rows=("obs_id", "size"),
        minimum_absolute_difference_arcsec=(
            "absolute_resolution_difference_arcsec", "min"
        ),
        median_absolute_difference_arcsec=(
            "absolute_resolution_difference_arcsec", "median"
        ),
        maximum_absolute_difference_arcsec=(
            "absolute_resolution_difference_arcsec", "max"
        ),
    )
    .reset_index()
    if len(resolution_mismatch_example_df)
    else pd.DataFrame()
)

resolution_diagnostic_complete = (
    resolution_comparable_rows >= resolution_mismatch_rows
    and len(resolution_mismatch_example_df) == resolution_example_limit
)
display(
    pd.DataFrame(
        [
            {
                "comparable_rows": resolution_comparable_rows,
                "exact_mismatch_rows": resolution_mismatch_rows,
                "bounded_example_rows": len(
                    resolution_mismatch_example_df
                ),
                "examples_complete_population": (
                    len(resolution_mismatch_example_df)
                    == resolution_mismatch_rows
                ),
                "diagnostic_complete": resolution_diagnostic_complete,
            }
        ]
    )
)
display(resolution_mismatch_summary_df)



--- Count Archive-wide comparable spatial resolutions ---
Endpoint: https://almascience.eso.org/tap
MAXREC: 10
ADQL:

SELECT COUNT(*) AS total_rows
FROM ivoa.obscore
WHERE science_observation = 'T'
  AND s_resolution IS NOT NULL
  AND spatial_resolution IS NOT NULL



Retrieved rows: 1
QUERY_STATUS: ('OK',)
Warnings: <none>

--- Count Archive-wide exact spatial-resolution mismatches ---
Endpoint: https://almascience.eso.org/tap
MAXREC: 10
ADQL:

SELECT COUNT(*) AS total_rows
FROM ivoa.obscore
WHERE science_observation = 'T'
  AND s_resolution IS NOT NULL
  AND spatial_resolution IS NOT NULL
  AND s_resolution <> spatial_resolution

Retrieved rows: 1
QUERY_STATUS: ('OK',)
Warnings: <none>

--- Retrieve spatial-resolution mismatch examples ---
Endpoint: https://almascience.eso.org/tap
MAXREC: 500
ADQL:

    SELECT TOP 500
        proposal_id,
        member_ous_uid,
        asdm_uid,
        obs_id,
        s_resolution,
        spatial_resolution,
        antenna_arrays,
        dataproduct_type,
        frequency_support,
        is_mosaic,
        type
    FROM ivoa.obscore
    WHERE science_observation = 'T'
      AND s_resolution IS NOT NULL
      AND spatial_resolution IS NOT NULL
      AND s_resolution <> spatial_resolution
    ORDER BY proposa

,comparable_rows,exact_mismatch_rows,bounded_example_rows,examples_complete_population,diagnostic_complete
0,442507,41365,500,False,True


,difference_class,array_evidence_class,dataproduct_type,support_grammar_family,is_mosaic,type,example_rows,minimum_absolute_difference_arcsec,median_absolute_difference_arcsec,maximum_absolute_difference_arcsec
0,MATERIAL_DIFFERENCE,12M_LIKE+TP_LIKE,cube,BRACKET_INTERVAL,F,S,4,6.088800,6.088800,6.088800
1,MATERIAL_DIFFERENCE,TP_LIKE,cube,BRACKET_INTERVAL,F,S,485,2.488548,6.119075,6.119726
2,MATERIAL_DIFFERENCE,TP_LIKE,image,BRACE_CENTRE_RESOLUTION,F,S,11,2.371890,2.371890,2.371890


## Step 16 — Classify all 73 live schema fields

Classification is scope assignment, not a claim that every field is a
production comparison input. Identifier evidence is attached explicitly:
`obs_id` is descriptive and potentially truncated, while
`obs_publisher_did` is an external publisher identifier whose scope and
uniqueness must be decided by its census.


In [60]:
previously_unclassified_field_scope = {
    "access_estsize": "PRODUCT_OPTIONAL",
    "collections": "PRODUCT_OPTIONAL",
    "em_xel": "PRODUCT_OPTIONAL",
    "facility_name": "SERVICE_CONTEXT",
    "gal_latitude": "DERIVED_SPATIAL_CROSS_CHECK",
    "gal_longitude": "DERIVED_SPATIAL_CROSS_CHECK",
    "instrument_name": "SERVICE_CONTEXT",
    "o_ucd": "RAW_METADATA_ONLY",
    "obs_collection": "SERVICE_CONTEXT",
    "obs_creator_name": "DEFERRED_PROVENANCE",
    "pi_name": "DEFERRED_PROVENANCE",
    "pi_userid": "DEFERRED_SENSITIVE_IDENTIFIER",
    "pol_xel": "PRODUCT_OPTIONAL",
    "s_xel1": "PRODUCT_OPTIONAL",
    "s_xel2": "PRODUCT_OPTIONAL",
    "t_xel": "PRODUCT_OPTIONAL",
}


def final_schema_scope(column_name: str) -> str:
    if column_name in modeled_fields:
        return "CORE_OR_RECONSTRUCTION"
    if column_name in cross_check_fields:
        return "CROSS_CHECK"
    if column_name in deferred_fields:
        return "DEFERRED"
    return previously_unclassified_field_scope.get(
        column_name,
        "UNCLASSIFIED",
    )


identifier_evidence_note = {
    "obs_id": (
        "RAW_DESCRIPTIVE_IDENTIFIER_NOT_KEY; "
        + obs_id_truncation_status
    ),
  
    "obs_publisher_did": (
        publisher_proposal_scope_status
    ),
    "member_ous_uid": "MEMBER_ENTITY_IDENTIFIER",
    "asdm_uid": "EXECUTION_IDENTIFIER_RETAIN_IN_CONTEXT_KEY",
}

final_schema_scope_df = schema_df.copy()
final_schema_scope_df["final_scope"] = (
    final_schema_scope_df["column_name"].map(final_schema_scope)
)
final_schema_scope_df["evidence_note"] = (
    final_schema_scope_df["column_name"]
    .map(identifier_evidence_note)
    .fillna("")
)
final_unclassified_schema_df = final_schema_scope_df.loc[
    final_schema_scope_df["final_scope"].eq("UNCLASSIFIED")
].copy()
all_schema_fields_classified = final_unclassified_schema_df.empty

final_schema_scope_summary_df = (
    final_schema_scope_df["final_scope"]
    .value_counts(dropna=False)
    .rename_axis("final_scope")
    .reset_index(name="column_count")
)

display(final_schema_scope_summary_df)
display(final_schema_scope_df.sort_values("column_name"))
display(final_unclassified_schema_df)

if not all_schema_fields_classified:
    raise AssertionError(
        "Live schema still contains unclassified fields: "
        f"{final_unclassified_schema_df['column_name'].tolist()}"
    )


,final_scope,column_count
0,CORE_OR_RECONSTRUCTION,31
1,DEFERRED,14
2,CROSS_CHECK,12
3,PRODUCT_OPTIONAL,7
4,SERVICE_CONTEXT,3
5,DERIVED_SPATIAL_CROSS_CHECK,2
6,DEFERRED_PROVENANCE,2
7,RAW_METADATA_ONLY,1
8,DEFERRED_SENSITIVE_IDENTIFIER,1


,column_name,datatype,unit,ucd,description,final_scope,evidence_note
0,access_estsize,int,kbyte,phys.size;meta.file,Estimated size of datasets in kilobytes,PRODUCT_OPTIONAL,
1,access_format,char,,meta.code.mime,Content format of the data,DEFERRED,
2,access_url,char,,meta.ref.url,URL to download the data,DEFERRED,
3,antenna_arrays,char,,meta.code.member;instr.setup,"Blank-separated list of Pad:Antenna pairs, i.e., A109:DV09 J504:DV02 J505:DV05 for antennas DV09, DV02 and DV05 sitt...",CORE_OR_RECONSTRUCTION,
4,asdm_uid,char,,meta.id,UID of the ASDM containing this Field.,CORE_OR_RECONSTRUCTION,EXECUTION_IDENTIFIER_RETAIN_IN_CONTEXT_KEY
...,...,...,...,...,...,...,...
68,t_resolution,double,s,time.resolution,typical temporal resolution,CROSS_CHECK,
69,t_xel,int,,meta.number,Number of elements along the time axis,PRODUCT_OPTIONAL,
70,target_name,char,,meta.id;src,name of intended target,CORE_OR_RECONSTRUCTION,
71,type,char,,,Type flags.,CROSS_CHECK,


,column_name,datatype,unit,ucd,description,final_scope,evidence_note


## Step 17 — Final provenance, evidence summary, and closure gate

The final provenance snapshot is generated only after every TAP query.
Data exceptions do not fail closure when they have been completely
counted, classified, and represented in the proposed model.


In [61]:
final_query_provenance_df = pd.DataFrame(
    [
        {
            "sequence": sequence,
            "label": run.label,
            "endpoint": run.endpoint,
            "started_at_utc": run.started_at_utc,
            "finished_at_utc": run.finished_at_utc,
            "maxrec": run.maxrec,
            "retrieved_rows": run.retrieved_rows,
            "query_status": run.query_status,
            "warnings": run.warning_messages,
            "adql_sha256": hashlib.sha256(
                " ".join(run.adql.split()).encode("utf-8")
            ).hexdigest(),
        }
        for sequence, run in enumerate(query_runs, start=1)
    ]
)
final_provenance_complete = (
    len(final_query_provenance_df) == len(query_runs)
)
final_unexpected_overflow_df = final_query_provenance_df.loc[
    final_query_provenance_df["query_status"]
    .map(lambda statuses: "OVERFLOW" in " ".join(statuses))
    & ~final_query_provenance_df["label"].eq(
        "Intentional MAXREC truncation experiment"
    )
].copy()

display(final_query_provenance_df)
display(final_unexpected_overflow_df)
print("Final provenance rows:", len(final_query_provenance_df))
print("Tracked query runs:", len(query_runs))


,sequence,label,endpoint,started_at_utc,finished_at_utc,maxrec,retrieved_rows,query_status,warnings,adql_sha256
0,1,Retrieve live ivoa.obscore schema,https://almascience.eso.org/tap,2026-08-25T12:59:37.036486+00:00,2026-08-25T12:59:37.903958+00:00,5000,73,"(OK,)",(),73384702fcc0ea62ea503e1d601947fa14a0b8847bf87741835eba37ff159f88
1,2,Discover early_cycles,https://almascience.eso.org/tap,2026-08-25T12:59:38.008739+00:00,2026-08-25T12:59:38.437282+00:00,80,80,"(OK,)",(),843fd4977b3d5a26b61ea24c04e329dde43d99e18d236920b3a218f196f3a649
2,3,Discover middle_cycles,https://almascience.eso.org/tap,2026-08-25T12:59:38.440770+00:00,2026-08-25T12:59:39.409598+00:00,80,80,"(OK,)",(),dbaf54b7d0cb6c5970622666a18d2be473ccbffc87f5baee81ee2384af3241f6
3,4,Discover recent_cycles,https://almascience.eso.org/tap,2026-08-25T12:59:39.415520+00:00,2026-08-25T12:59:40.586473+00:00,80,80,"(OK,)",(),208df42927151846081033b7088fe6ca2f12eccede524129e7208aeb504a5ea7
4,5,Discover band3,https://almascience.eso.org/tap,2026-08-25T12:59:40.593914+00:00,2026-08-25T12:59:46.691600+00:00,80,80,"(OK,)",(),0c6e060eb74cb9221b4a23c0e7f0950a28e057b538836ccd59e8e2ef82151f17
...,...,...,...,...,...,...,...,...,...,...
87,88,Count Members containing composite-key candidates,https://almascience.eso.org/tap,2026-08-25T13:03:00.218983+00:00,2026-08-25T13:03:00.714675+00:00,10,1,"(OK,)",(),6ca8f8488e20070f3bce367d4be9b976257fa4e0900b6c8c3ccf4d1d24230d34
88,89,Retrieve Members containing composite-key candidates,https://almascience.eso.org/tap,2026-08-25T13:03:00.715132+00:00,2026-08-25T13:03:02.375441+00:00,375,375,"(OK,)",(),1c364e3d78334ba84e299015dd4e0d0b4dda9ed365f320cf156d1b4c6202acc8
89,90,Count Archive-wide comparable spatial resolutions,https://almascience.eso.org/tap,2026-08-25T13:03:02.808177+00:00,2026-08-25T13:03:04.137959+00:00,10,1,"(OK,)",(),baadccfbf298feba76d58a16d4b4fb4c745fc9ba8aec97a186224f919e6a9ac7
90,91,Count Archive-wide exact spatial-resolution mismatches,https://almascience.eso.org/tap,2026-08-25T13:03:04.139334+00:00,2026-08-25T13:03:04.689280+00:00,10,1,"(OK,)",(),04722d1eeffea365ad4451610fe69abbe83353fbc893a37928d8195280519669


,sequence,label,endpoint,started_at_utc,finished_at_utc,maxrec,retrieved_rows,query_status,warnings,adql_sha256


Final provenance rows: 92
Tracked query runs: 92


In [62]:
brace_structural_checks_pass = bool(
    brace_census_decision_df.loc[
        ~brace_census_decision_df["check"].eq(
            "Token-2 semantics independently discriminated"
        ),
        "passed",
    ].all()
)
identifier_exceptions_resolved = bool(
    obs_id_census_coverage_complete
    and obs_id_failures_classified
    and member_mismatch_mask.sum() == 0
    and non_integer_spw_mask.sum() == 0
)

publisher_census_resolved = bool(
    obs_publisher_did_census_complete
    and publisher_proposal_scope_complete
)

product_experiments_complete = bool(
    product_census_complete
    and product_duplicate_detail_complete
    and resolution_diagnostic_complete
)

supplementary_closure_gate_df = pd.DataFrame(
    [
        {
            "closure_requirement": "Complete brace population retrieved",
            "passed": len(brace_analysis_df) == brace_census_expected_rows,
            "status_or_details": (
                f"{len(brace_analysis_df)} / {brace_census_expected_rows}"
            ),
        },
        {
            "closure_requirement": "Brace structure and mapping validated",
            "passed": brace_structural_checks_pass,
            "status_or_details": brace_token2_semantic_status,
        },
        {
            "closure_requirement": "Archive-wide identifier retrieval complete",
            "passed": obs_id_census_coverage_complete,
            "status_or_details": (
                f"{len(obs_id_archive_analysis_df)} / "
                f"{obs_id_expected_total}"
            ),
        },
        {
            "closure_requirement": "obs_id failures classified",
            "passed": identifier_exceptions_resolved,
            "status_or_details": obs_id_truncation_status,
        },
        {
            "closure_requirement": "obs_publisher_did census complete",
            "passed": publisher_census_resolved,
            
            "status_or_details": (
                publisher_proposal_scope_status
            ),
        },
        {
            "closure_requirement": "Repeated-source discovery sufficient",
            "passed": repeated_source_discovery_sufficient,
            "status_or_details": repeated_source_archive_status,
        },
        {
            "closure_requirement": "STC-S client-side partition complete",
            "passed": stcs_partition_complete,
            "status_or_details": (
                f"{stcs_partition_rows} / {stcs_total_rows}"
            ),
        },
        {
            "closure_requirement": "Product diagnostics complete",
            "passed": product_experiments_complete,
            "status_or_details": archive_product_granularity_status,
        },
        {
            "closure_requirement": "All live schema fields classified",
            "passed": all_schema_fields_classified,
            "status_or_details": (
                f"unclassified={len(final_unclassified_schema_df)}"
            ),
        },
        {
            "closure_requirement": "Final query provenance complete",
            "passed": (
                final_provenance_complete
                and final_unexpected_overflow_df.empty
            ),
            "status_or_details": (
                f"tracked={len(final_query_provenance_df)}; "
                f"unexpected_overflow={len(final_unexpected_overflow_df)}"
            ),
        },
        {
            "closure_requirement": "Shuffle invariance retained",
            "passed": bool(
                shuffle_invariance_df[
                    "all_reconstructions_match"
                ].all()
            ),
            "status_or_details": (
                shuffle_invariance_df
                .set_index("shuffle_seed")[
                    "all_reconstructions_match"
                ]
                .to_dict()
            ),
        },
    ]
)

supplementary_closure_status = (
    "READY_FOR_FINAL_EVIDENCE_REVIEW_AND_DATA_MODEL_V0_4"
    if supplementary_closure_gate_df["passed"].all()
    else "NOT_READY_REVIEW_FAILED_CLOSURE_REQUIREMENTS"
)

final_identifier_evidence_df = pd.DataFrame(
    [
        {
            "question": "obs_id Archive-wide row key",
            "evidence": (
                f"duplicate_rows={len(duplicate_obs_id_df)}; "
                f"duplicate_groups={len(duplicate_obs_id_group_df)}"
            ),
            "model_decision": "REJECT_USE_SURROGATE_KEY",
        },
        {
            "question": "Source-Execution-SPW row key",
            "evidence": (
                "duplicate_rows="
                f"{len(duplicate_source_execution_spw_df)}; "
                "duplicate_groups="
                f"{len(duplicate_source_execution_spw_group_df)}; "
                "verified_product_groups="
                f"{len(reliable_product_multiplicity_group_df)}; "
                "width_collision_groups="
                f"{len(width_boundary_product_collision_df)}"
            ),
            "model_decision": (
                "REJECT_ROW_KEY_ADD_ARCHIVE_PRODUCT_ENTITY"
                if len(reliable_product_multiplicity_group_df)
                else (
                    "REJECT_ROW_KEY_PRODUCT_MULTIPLICITY_"
                    "UNRESOLVED_AT_WIDTH_BOUNDARY"
                )
            ),
        },

        {
            "question": (
                "obs_publisher_did external identifier"
            ),
            "evidence": (
                publisher_proposal_scope_status
            ),
            "model_decision": (
                "PRESERVE_AS_PROPOSAL_PUBLICATION_IDENTIFIER_"
                "NOT_PRODUCT_OR_ROW_IDENTIFIER"
            ),
        },

        {
            "question": "Repeated source across ASDM",
            "evidence": repeated_source_archive_status,
            "model_decision": (
                "KEEP_GEOMETRY_SPECTRAL_RESOLUTION_TIME_"
                "AT_SOURCE_EXECUTION_LEVEL"
            ),
        },
    ]
)

display(final_identifier_evidence_df)
display(supplementary_closure_gate_df)
print("Supplementary closure status:", supplementary_closure_status)


,question,evidence,model_decision
0,obs_id Archive-wide row key,duplicate_rows=496; duplicate_groups=134,REJECT_USE_SURROGATE_KEY
1,Source-Execution-SPW row key,duplicate_rows=114; duplicate_groups=42; verified_product_groups=0; width_collision_groups=42,REJECT_ROW_KEY_PRODUCT_MULTIPLICITY_UNRESOLVED_AT_WIDTH_BOUNDARY
2,obs_publisher_did external identifier,PROPOSAL_SCOPED_PUBLICATION_IDENTIFIER_NOT_ROW_KEY,PRESERVE_AS_PROPOSAL_PUBLICATION_IDENTIFIER_NOT_PRODUCT_OR_ROW_IDENTIFIER
3,Repeated source across ASDM,SPATIALLY_VERIFIED_REPEATED_SOURCE_ACROSS_ASDM_FOUND,KEEP_GEOMETRY_SPECTRAL_RESOLUTION_TIME_AT_SOURCE_EXECUTION_LEVEL


,closure_requirement,passed,status_or_details
0,Complete brace population retrieved,True,55 / 55
1,Brace structure and mapping validated,True,AMBIGUOUS_BANDWIDTH_VS_RESOLUTION_NUMERICAL_DEGENERACY
2,Archive-wide identifier retrieval complete,True,442507 / 442507
3,obs_id failures classified,True,ALL_FAILURES_CLASSIFIED_AS_DECLARED_WIDTH_TRUNCATION
4,obs_publisher_did census complete,True,PROPOSAL_SCOPED_PUBLICATION_IDENTIFIER_NOT_ROW_KEY
5,Repeated-source discovery sufficient,True,SPATIALLY_VERIFIED_REPEATED_SOURCE_ACROSS_ASDM_FOUND
6,STC-S client-side partition complete,True,442507 / 442507
7,Product diagnostics complete,True,COMPOSITE_DUPLICATES_UNRESOLVED_DUE_OBS_ID_WIDTH_BOUNDARY
8,All live schema fields classified,True,unclassified=0
9,Final query provenance complete,True,tracked=92; unexpected_overflow=0


Supplementary closure status: READY_FOR_FINAL_EVIDENCE_REVIEW_AND_DATA_MODEL_V0_4
